In [1]:
import sys, os
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
SRC  = ROOT / "src"

print("ROOT exists:", ROOT.exists())
print("SRC exists: ", SRC.exists())
print()

# Is tutor importable from here?
sys.path.insert(0, str(SRC))
try:
    from tutor import drill
    print("import tutor.drill        OK")
    print("drill file:               ", drill.__file__)
except Exception as e:
    print("import tutor.drill FAILED:", type(e).__name__, e)

# Do the pieces we need exist?
for name in ("schemas", "learner_store", "analogue_library",
             "template_router", "math_grader", "step_grader",
             "presentation_matcher", "diagnosis", "slash_commands"):
    try:
        mod = __import__(f"tutor.{name}", fromlist=[name])
        print(f"  tutor.{name:<22} OK")
    except Exception as e:
        print(f"  tutor.{name:<22} {type(e).__name__}: {e}")

print()

# Where is the learner store expected?
for rel in ("data/learner_store", "data/processed/tutor", "docs"):
    p = ROOT / rel
    print(f"{rel:<26} {'exists' if p.exists() else 'MISSING'}")

print()

# Any existing learners.db?
for p in ROOT.rglob("*.db"):
    print("db found:", p.relative_to(ROOT), f"({p.stat().st_size:,} bytes)")

ROOT exists: True
SRC exists:  True

import tutor.drill        OK
drill file:                C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\src\tutor\drill.py
  tutor.schemas                OK
  tutor.learner_store          OK
  tutor.analogue_library       OK
  tutor.template_router        OK
  tutor.math_grader            OK
  tutor.step_grader            OK
  tutor.presentation_matcher   OK
  tutor.diagnosis              OK
  tutor.slash_commands         OK

data/learner_store         exists
data/processed/tutor       exists
docs                       exists

db found: data\learner_store\learners.db (135,168 bytes)


In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\src")))

from tutor import learner_store
from tutor.schemas import Problem
import inspect

print("=== learner_store public API ===")
for name in dir(learner_store):
    if name.startswith("_"): continue
    obj = getattr(learner_store, name)
    if inspect.isclass(obj):
        print(f"  class {name}")
    elif inspect.isfunction(obj):
        print(f"  def   {name}{inspect.signature(obj)}")

print()
print("=== classes: their __init__ signatures ===")
for name in dir(learner_store):
    if name.startswith("_"): continue
    obj = getattr(learner_store, name)
    if inspect.isclass(obj):
        try:
            print(f"  {name}{inspect.signature(obj.__init__)}")
        except Exception as e:
            print(f"  {name} (sig fail: {e})")

print()
print("=== Problem fields ===")
try:
    print("  ", inspect.signature(Problem))
except Exception as e:
    print("  no sig:", e)
    print("  fields:", [f for f in dir(Problem) if not f.startswith("_")])

print()
print("=== drill_server helper check: does store expose these? ===")
# We only need to know which of these the store has, to build the server
candidates = [
    "save_attempt", "get_mastery", "recompute_mastery",
    "interventions", "discoveries",
]
# try to find the store class and its methods
for name in dir(learner_store):
    if name.startswith("_"): continue
    cls = getattr(learner_store, name)
    if not inspect.isclass(cls): continue
    print(f"  --- {name} ---")
    for m in candidates:
        print(f"     {m:<20} {'yes' if hasattr(cls, m) else 'no'}")

=== learner_store public API ===
  class DiscoveryStore
  class InterventionStore
  class LearnerStore
  class MasteryState
  class Path
  class datetime
  def   open_store(path: pathlib.Path) -> tutor.learner_store.LearnerStore
  class timedelta
  class timezone

=== classes: their __init__ signatures ===
  DiscoveryStore(self, conn)
  InterventionStore(self, conn)
  LearnerStore(self, db_path: pathlib.Path)
  MasteryState(self, /, *args, **kwargs)
  Path(self, *args)
  datetime(self, /, *args, **kwargs)
  timedelta(self, /, *args, **kwargs)
  timezone(self, /, *args, **kwargs)

=== Problem fields ===
   (problem_id: str, skill_id: str, topic: str, subtopic: str, structure_type: str, prompt: str, expected_answer: str, topic_v2: str | None = None, step_scheme: list | None = None) -> None

=== drill_server helper check: does store expose these? ===
  --- DiscoveryStore ---
     save_attempt         no
     get_mastery          no
     recompute_mastery    no
     interventions        no

In [3]:
# ══════════════════════════════════════════════════════════════════
# Rewrite src/tutor/drill.py — same behaviour as before, plus:
#   · set_line_provider()  (CLI or web drives the same drill)
#   · grade_submission()    (one POST → one grade)
# ══════════════════════════════════════════════════════════════════
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
TARGET = ROOT / "src" / "tutor" / "drill.py"
BACKUP = ROOT / "src" / "tutor" / "drill.py.bak"

NEW_DRILL = r'''"""
Drill engine — practice mode (v2, holistic rewrite).

Anchor → attempt → grade → save attempt → sibling on fail → loop → mastery.

Works identically for every topic. No answer leaks. No note spam.
"""
from __future__ import annotations

import json
import random
import uuid
from typing import Optional

from .schemas import Problem, WorkingSubmission, WorkingStep
from . import analogue_library
from .template_router import route_template
from .presentation_matcher import check_presentation, format_note
from .math_grader import grade, diagnose_wrong, MatchKind
from .step_grader import grade_steps as grade_steps_fn, grade_steps_inline
from .step_schemes import build_scheme
from .diagnosis import diagnose


SITTING_SIZE = 5
PASS_THRESHOLD = 4


# ============================================================
# Line provider — CLI uses input(); web server sets a callback
# ============================================================
_LINE_PROVIDER = None


def set_line_provider(fn):
    """When set, _collect_working calls fn(prompt) instead of input(prompt)."""
    global _LINE_PROVIDER
    _LINE_PROVIDER = fn


def _read_line(prompt: str) -> str:
    if _LINE_PROVIDER is not None:
        return _LINE_PROVIDER(prompt)
    return input(prompt)


# ============================================================
# Pool helpers
# ============================================================
def _template_path_for_anchor(problem_id: str):
    """Map an anchor problem_id to its DBE template JSON path."""
    from pathlib import Path as _P
    _root = _P(__file__).resolve().parents[2]
    tdir = _root / "data" / "processed" / "tutor" / "question_templates"
    parts = str(problem_id).split("_")
    if len(parts) < 3:
        return None
    year, paper = parts[0], parts[1]
    qpart = parts[2].split(".")[0]  # 'Q1.1.1' -> 'Q1'
    p = tdir / f"template_{year}_{paper}_{qpart}.json"
    return p if p.exists() else None


def _anchor_template_metadata(problem_id: str):
    """Return the template dict for an anchor problem_id, or None."""
    tp = _template_path_for_anchor(problem_id)
    if not tp:
        return None
    try:
        import json as _json
        return _json.loads(tp.read_text(encoding="utf-8"))
    except Exception:
        return None


def _try_get_anchor_diagram_image(problem_id: str):
    """Return the parent paper's page-image path for this anchor,
    if the template carries one."""
    t = _anchor_template_metadata(problem_id)
    if not t:
        return None
    img = t.get("diagram_image")
    if not img:
        return None
    from pathlib import Path as _P
    root = _P(__file__).resolve().parents[2]
    p = root / img
    return str(p) if p.exists() else None


def _try_render_anchor_diagram(problem_id: str):
    """Preferred: parent page image. Fallback: rendered spec."""
    t = _anchor_template_metadata(problem_id)
    if not t:
        return None
    d = t.get("diagram")
    s = t.get("diagram_spec")
    page = _try_get_anchor_diagram_image(problem_id)
    if page:
        return page
    if d and s:
        try:
            from pathlib import Path as _P
            from tutor import diagram_dispatcher as _disp
            root = _P(__file__).resolve().parents[2]
            cache = root / "data" / "processed" / "diagrams" / "drill_cache"
            cache.mkdir(parents=True, exist_ok=True)
            safe = str(problem_id).replace(".", "_").replace("/", "_")
            out = cache / f"{safe}.png"
            _disp.render_diagram(d, out_path=out, **s)
            return str(out)
        except Exception:
            pass
    return None


_SKILL_DIAGRAM_INDEX = None


def _load_skill_diagram_index():
    global _SKILL_DIAGRAM_INDEX
    if _SKILL_DIAGRAM_INDEX is None:
        import json as _json
        from pathlib import Path as _P
        root = _P(__file__).resolve().parents[2]
        p = root / "data" / "processed" / "tutor" / "skill_diagram_index.json"
        _SKILL_DIAGRAM_INDEX = (
            _json.loads(p.read_text(encoding="utf-8"))
            if p.exists() else {}
        )
    return _SKILL_DIAGRAM_INDEX


def _reference_diagram_for_skill(skill_id: str):
    """Return a parent page image path for a variant of this skill."""
    index = _load_skill_diagram_index()
    entries = index.get(skill_id, [])
    if not entries:
        return None
    from pathlib import Path as _P
    root = _P(__file__).resolve().parents[2]
    for e in entries:
        img = e.get("image")
        if not img:
            continue
        p = root / img
        if p.exists():
            return str(p)
    return None


def _load_pool(template_id: str):
    return analogue_library.POOL.get(template_id, [])


def _extract_expected(analogue: dict) -> str:
    steps = analogue.get("steps", [])
    if not steps:
        return ""
    last = str(steps[-1])
    if ":" in last:
        last = last.split(":", 1)[1]
    return last.strip()


def _analogue_to_problem(analogue: dict, template_id: str, skill_id: str) -> Problem:
    expected = _extract_expected(analogue)
    scheme = build_scheme(template_id, analogue, expected)
    return Problem(
        problem_id=f"drill_{template_id}_{uuid.uuid4().hex[:6]}",
        skill_id=skill_id,
        topic="",
        subtopic="",
        structure_type="routine_calculation",
        prompt=analogue["problem"],
        expected_answer=expected,
        topic_v2=None,
        step_scheme=scheme,
    )


# ============================================================
# Working input
# ============================================================
def _collect_working(q_num: int, total: int):
    steps = []
    print()
    print(f"  Question {q_num} of {total}")
    print("  Show your working. 'done' to submit, 'help' for a worked example,")
    print("  'next' to skip this question, 'end' to finish this sitting,")
    print("  'menu' to pick a different topic, 'q' to quit the tutor.")
    print()
    while True:
        idx = len(steps) + 1
        try:
            raw = _read_line(f"    Line {idx}: ").rstrip()
        except (EOFError, KeyboardInterrupt):
            return None
        s = raw.strip()
        if not s:
            continue
        low = s.lower()
        if low in ("q", "quit", "exit"):
            return None
        if low in ("next", "skip", "n"):
            return "NEXT"
        if low in ("menu", "topics", "back", "topics:"):
            return "MENU"
        if low in ("end", "stop", "enough"):
            return "END"
        if low in ("help", "answer", "hint"):
            return "HELP"
        if low in ("more", "another"):
            return "MORE"
        if low in ("done", "submit"):
            if not steps:
                print("    (type at least one line, 'help', or 'q' to quit)")
                continue
            return WorkingSubmission(
                steps=[WorkingStep(index=i + 1, raw_text=t) for i, t in enumerate(steps)],
                source="typed",
                final_answer=steps[-1],
            )
        if low.startswith("answer:"):
            ans = s[len("answer:"):].strip()
            if ans:
                return WorkingSubmission(
                    steps=[WorkingStep(index=1, raw_text=ans)],
                    source="typed",
                    final_answer=ans,
                )
        steps.append(s)


# ============================================================
# Sibling example — same template, different variant
# ============================================================
def _show_sibling(pool, shown_keys):
    candidates = [a for a in pool if analogue_library.analogue_key(a) not in shown_keys]
    if not candidates:
        candidates = pool
    if not candidates:
        return None
    ex = random.choice(candidates)
    print()
    print("  Let's look at a similar one first.")
    print()
    print(f"    Problem: {ex['problem']}")
    for s in ex.get("steps", []):
        print(f"      {s}")
    if ex.get("note"):
        print(f"    Note: {ex['note']}")
    shown_keys.add(analogue_library.analogue_key(ex))
    return ex


# ============================================================
# Warm diagnosis message
# ============================================================
def _speak_diagnosis(kind: str):
    if kind == "sign_flip":
        print()
        print("  I see the shape of the working. The sign on one root looks")
        print("  flipped — the number is right, the sign isn't. Worth a look.")
    elif kind == "partial":
        print()
        print("  One of the roots is right. The other is missing or different.")
        print("  Both matter for this question.")
    else:
        print()
        print("  There's something off. Let's look at a similar one together.")


# ============================================================
# Record attempt into the store
# ============================================================
def _save_attempt(store, session_id, learner_id, skill_id,
                  response, rule_id, error_type,
                  misconception_id=None, hint_level=None):
    try:
        store.save_attempt(
            session_id=session_id,
            learner_id=learner_id,
            skill_id=skill_id,
            response=response[:500],
            diagnosis_rule_id=rule_id or "",
            misconception_id=misconception_id,
            hint_level=hint_level,
            error_type=error_type,
        )
    except Exception:
        pass


# ============================================================
# Mastery summary
# ============================================================
def _speak_mastery(skill_id, template_id, questions_correct, total_questions,
                   topic_label, questions_skipped=0):
    print()
    print("=" * 60)
    print(f"  Drill complete — {template_id}")
    line = f"  Correct: {questions_correct}/{total_questions}"
    if questions_skipped:
        line += f"   (skipped: {questions_skipped})"
    print(line)
    print("=" * 60)
    print()
    if questions_correct >= PASS_THRESHOLD:
        print(f"  The method held together across the variants, {topic_label}.")
        print("  That is the point of the drill.")
    elif questions_skipped and questions_correct > 0:
        print(f"  You moved on from {questions_skipped} of them. That is your call.")
        print("  Come back to this pattern when you feel like it.")
    else:
        print(f"  You got {questions_correct} of {total_questions}. That is close.")
        print("  One more sitting and the method will lock in.")
    print()
    print("  Disclaimer — practice is one thing, exam is another. In the exam")
    print("  you'll be marked strictly against the DBE memo. Same method, but")
    print("  line by line, in their format.")


import re as _re


def _corpus_id_to_memo_id(problem_id: str) -> str:
    """2023_P1_Q1_1.1.1 -> 2023_P1_Q1.1.1 (drop underscore + repeated digit)."""
    if not problem_id or problem_id.startswith("drill_"):
        return ""
    return _re.sub(r"Q(\d+)_\1\.", r"Q\1.", problem_id)


def _step_feedback(problem_id: str, learner_lines: list[str]) -> dict | None:
    """Return step-grader result for the anchor, or None if not applicable."""
    memo_id = _corpus_id_to_memo_id(problem_id)
    if not memo_id:
        return None
    try:
        r = grade_steps_fn(memo_id, learner_lines)
    except Exception:
        return None
    if r["total_marks"] == 0:
        return None
    return r


def _print_step_feedback(result: dict) -> None:
    """Format the step result for the CLI."""
    awarded = result["awarded_marks"]
    total = result["total_marks"]
    steps = result["steps"]

    missing = [s for s in steps if s.get("match_status") == "NOT_MATCHED"]
    uncertain = [s for s in steps if s.get("match_status") == "UNCERTAIN"]

    if not missing and not uncertain:
        print(f"  Full method shown — memo awards {total}/{total}.")
        return

    print(f"  Memo would award {awarded}/{total}.")
    for s in missing:
        mark_word = "mark" if s["marks"] == 1 else "marks"
        print(f"    You skipped: {s['desc']}  ({s['marks']} {mark_word})")
        forms = s.get("forms") or []
        if forms:
            print(f"      The memo would show:   {forms[0]}")
    for u in uncertain:
        print(f"    Cannot verify: {u['desc']}")
    print("    (In the exam, markers award marks per step.)")


# ============================================================
# Auto-outcome measurement
# ============================================================
def _close_open_interventions(store, learner_id, skill_id, step_result,
                               current_qid, session_id):
    """Record whether each open intervention is now satisfied."""
    try:
        open_invs = store.interventions.list_open(learner_id, skill_id)
    except Exception:
        return
    if not open_invs:
        return

    step_status = {}
    if step_result is not None:
        for s in step_result.get("steps", []):
            step_status[int(s.get("index", -1))] = s.get("match_status")

    for inv in open_invs:
        try:
            ctx = json.loads(inv.get("context") or "{}")
        except Exception:
            ctx = {}
        step_index = ctx.get("step_index")

        if step_index is not None and int(step_index) in step_status:
            status = step_status[int(step_index)]
            if status == "MATCHED":
                result = "improved"
            elif status == "NOT_MATCHED":
                result = "unchanged"
            else:
                continue
            evidence = {
                "measured_via": "step_grader",
                "step_index": int(step_index),
                "current_qid": current_qid,
                "session_id": session_id,
            }
        else:
            result = "improved"
            evidence = {
                "measured_via": "answer_match",
                "current_qid": current_qid,
                "session_id": session_id,
            }

        try:
            store.interventions.record_outcome(
                intervention_id=inv["intervention_id"],
                learner_id=learner_id,
                result=result,
                measurement_kind="follow_up",
                evidence=evidence,
            )
        except Exception:
            pass


# ============================================================
# Main drill
# ============================================================
def run(engine, store, learner_id, session_id, anchor, logger=None,
        interactive_input=True, sitting_number=1):
    template_id = route_template(anchor.skill_id, anchor.prompt)
    if template_id is None:
        print(f"  (No template for skill {anchor.skill_id} — skipping.)")
        return {"sittings_passed": 0, "correct": 0, "total": 0, "exit_reason": "no_template"}

    pool = _load_pool(template_id)
    if not pool:
        print(f"  (No variants for {template_id} — skipping.)")
        return {"sittings_passed": 0, "correct": 0, "total": 0, "exit_reason": "empty_pool"}

    topic_label = anchor.topic or anchor.skill_id.split(".")[0]

    print()
    print("=" * 60)
    print(f"  Drill: {anchor.skill_id}")
    if sitting_number > 1:
        print(f"  Sitting {sitting_number}")
    print(f"  Template: {template_id}  ({len(pool)} variants)")
    print("=" * 60)

    sequence = []
    anchor_ok = bool(anchor.expected_answer and anchor.expected_answer.strip())
    if anchor_ok:
        sequence.append({"type": "anchor", "problem": anchor})

    cycle = pool.copy()
    random.shuffle(cycle)
    cycle_idx = 0
    shown_variant_keys = set()

    while len(sequence) < SITTING_SIZE:
        if cycle_idx >= len(cycle):
            cycle_idx = 0
            random.shuffle(cycle)
        analogue = cycle[cycle_idx]
        cycle_idx += 1
        key = analogue_library.analogue_key(analogue)
        if key in shown_variant_keys:
            continue
        shown_variant_keys.add(key)
        sequence.append({
            "type": "variation",
            "problem": _analogue_to_problem(analogue, template_id, anchor.skill_id),
        })

    questions_correct = 0
    questions_first_try = 0
    questions_skipped = 0
    questions_attempted = 0
    total_attempts = 0
    shown_note_ids = set()
    shown_sibling_keys = set()

    for i, item in enumerate(sequence, 1):
        drill_problem = item["problem"]

        print()
        print("-" * 60)
        if item["type"] == "anchor":
            _diag = _try_render_anchor_diagram(drill_problem.problem_id)
            if _diag:
                print(f"  Diagram: {_diag}")
                print()
        else:
            _ref = _reference_diagram_for_skill(drill_problem.skill_id)
            if _ref:
                print(f"  Reference diagram: {_ref}")
                print("  (Same shape, different numbers.)")
                print()
        print(f"  Problem: {drill_problem.prompt}")
        if item["type"] == "anchor":
            print("  (This is the original question you picked.)")
        print("-" * 60)

        attempts_on_this_q = 0
        while True:
            if not interactive_input:
                return {"sittings_passed": 0, "correct": questions_correct,
                        "total": total_attempts, "exit_reason": "non_interactive"}

            submission = _collect_working(i, SITTING_SIZE)

            if submission is None:
                _close_early(questions_correct, i - 1)
                _try_recompute_mastery(store, learner_id, anchor.skill_id)
                return {"sittings_passed": 0, "correct": questions_correct,
                        "total": total_attempts, "exit_reason": "quit"}

            if submission == "MENU":
                print()
                print("  Back to the topic menu. Pick anything you like.")
                _try_recompute_mastery(store, learner_id, anchor.skill_id)
                return {"sittings_passed": 0, "correct": questions_correct,
                        "total": questions_attempted,
                        "skipped": questions_skipped,
                        "exit_reason": "menu"}

            if submission == "END":
                questions_attempted += 1
                print()
                print("  Good. Free will beats force. Ending the sitting here.")
                _try_recompute_mastery(store, learner_id, anchor.skill_id)
                print()
                print(f"  This sitting: {questions_correct} correct, "
                      f"{questions_skipped} skipped, "
                      f"{questions_attempted - questions_correct - questions_skipped} left.")
                return {"sittings_passed": 0, "correct": questions_correct,
                        "total": questions_attempted,
                        "skipped": questions_skipped,
                        "exit_reason": "learner_end"}

            if submission == "NEXT":
                questions_skipped += 1
                questions_attempted += 1
                print()
                print("  Moving on. You can come back to this pattern any time.")
                break

            if submission == "HELP" or submission == "MORE":
                _show_sibling(pool, shown_sibling_keys)
                print()
                print("  Try the question again when ready — or type 'next' to move on.")
                continue

            from . import slash_commands as _sc

            _steps = getattr(submission, "steps", None) or []
            _first_raw = ""
            if _steps and hasattr(_steps[0], "raw_text"):
                _first_raw = _steps[0].raw_text.strip()

            if _sc.is_help(_first_raw) and len(_steps) == 1:
                print(_sc.HELP_TEXT)
                continue

            for _s in _steps:
                if hasattr(_s, "raw_text"):
                    _s.raw_text = _sc.expand(_s.raw_text)

            attempts_on_this_q += 1
            total_attempts += 1
            learner_text = submission.joined()

            result = grade(learner_text, drill_problem.expected_answer)

            if result.kind == MatchKind.MATCH:
                questions_correct += 1
                questions_attempted += 1
                if attempts_on_this_q == 1:
                    questions_first_try += 1

                _save_attempt(store, session_id, learner_id, anchor.skill_id,
                              learner_text, f"D_{result.matcher}", "none")

                print()
                print(f"  ✓ Correct.  ({questions_correct}/{i} this sitting)")

                learner_lines_for_steps = [s.raw_text for s in submission.steps]
                step_result = _step_feedback(
                    drill_problem.problem_id,
                    learner_lines_for_steps,
                )
                if step_result is None and getattr(drill_problem, "step_scheme", None):
                    try:
                        step_result = grade_steps_inline(
                            drill_problem.step_scheme,
                            learner_lines_for_steps,
                        )
                    except Exception:
                        step_result = None
                if step_result is not None:
                    _print_step_feedback(step_result)
                    try:
                        missing = [
                            s for s in step_result.get("steps", [])
                            if s.get("match_status") == "NOT_MATCHED"
                        ]
                        for m in missing:
                            store.interventions.log(
                                learner_id=learner_id,
                                skill_id=anchor.skill_id,
                                action_type="memo_line_shown",
                                action_desc="Showed memo version of missed step: " + m["desc"],
                                question_id=drill_problem.problem_id,
                                session_id=session_id,
                                context={"step_index": m["index"],
                                         "step_desc": m["desc"]},
                            )
                    except Exception:
                        pass
                    _close_open_interventions(
                        store, learner_id, anchor.skill_id, step_result,
                        drill_problem.problem_id, session_id,
                    )

                try:
                    notes = check_presentation(
                        [s.raw_text for s in submission.steps],
                        topic=anchor.topic_v2, max_notes=5,
                    )
                    for n in notes:
                        if n.rule_id in shown_note_ids:
                            continue
                        shown_note_ids.add(n.rule_id)
                        print(format_note(n))
                        break
                except Exception:
                    pass
                break

            if result.kind == MatchKind.UNCERTAIN:
                print()
                print("  I want to be sure this is the shape the memo expects.")
                _show_sibling(pool, shown_sibling_keys)
                print()
                print("  Try the same question again — or type 'next' to move on.")
                continue

            kind = diagnose_wrong(learner_text, drill_problem.expected_answer)

            misconception_id = None
            try:
                diag = diagnose(anchor.skill_id, learner_text,
                                drill_problem.expected_answer)
                if diag and diag.misconception_id:
                    misconception_id = diag.misconception_id
            except Exception:
                pass

            _save_attempt(store, session_id, learner_id, anchor.skill_id,
                          learner_text, "D_NO_MATCH",
                          (kind or "unknown"), misconception_id)

            try:
                if (kind or "unknown") == "unknown":
                    store.discoveries.record(
                        learner_id=learner_id,
                        skill_id=anchor.skill_id,
                        question_id=drill_problem.problem_id,
                        learner_lines=[s.raw_text for s in submission.steps],
                        expected_answer=drill_problem.expected_answer,
                        session_id=session_id,
                    )
            except Exception:
                pass

            _speak_diagnosis(kind or "unknown")
            if misconception_id:
                print(f"  Noticed: {misconception_id}")

            try:
                notes = check_presentation(
                    [s.raw_text for s in submission.steps],
                    topic=anchor.topic_v2, max_notes=5,
                )
                for n in notes:
                    if n.rule_id in shown_note_ids:
                        continue
                    shown_note_ids.add(n.rule_id)
                    print(format_note(n))
                    break
            except Exception:
                pass

            _show_sibling(pool, shown_sibling_keys)
            print()
            print("  Try the question again — or type 'next' to move on.")

    passed = questions_correct >= PASS_THRESHOLD
    _speak_mastery(anchor.skill_id, template_id,
                   questions_correct, SITTING_SIZE, topic_label,
                   questions_skipped=questions_skipped)

    _try_recompute_mastery(store, learner_id, anchor.skill_id)

    try:
        m = store.get_mastery(learner_id, anchor.skill_id)
        print(f"  Mastery: {m['mastery_state']}  |  Sittings passed: {m['sittings_passed']}")
    except Exception:
        pass

    if interactive_input:
        print()
        print("  Options:")
        print("    Enter / y  →  another sitting on this skill")
        print("    menu       →  pick a different topic")
        print("    q          →  quit")
        again = input("  > ").strip().lower()
        if again in ("", "y", "yes"):
            return run(engine, store, learner_id, session_id, anchor,
                       logger=logger, interactive_input=interactive_input,
                       sitting_number=sitting_number + 1)
        if again in ("menu", "m"):
            _try_recompute_mastery(store, learner_id, anchor.skill_id)
            return {"sittings_passed": 1 if passed else 0,
                    "correct": questions_correct, "total": SITTING_SIZE,
                    "exit_reason": "menu",
                    "template_id": template_id,
                    "sitting_number": sitting_number}

    return {
        "sittings_passed": 1 if passed else 0,
        "correct": questions_correct,
        "total": SITTING_SIZE,
        "first_try": questions_first_try,
        "attempts": total_attempts,
        "exit_reason": "completed",
        "template_id": template_id,
        "sitting_number": sitting_number,
    }


def _try_recompute_mastery(store, learner_id, skill_id):
    try:
        store.recompute_mastery(learner_id, skill_id)
    except Exception:
        pass


def _close_early(questions_correct, questions_attempted):
    print()
    print("=" * 60)
    print(f"  Session closed.  {questions_correct}/{questions_attempted} correct.")
    print("  Progress saved.")
    print("=" * 60)


# ============================================================
# Web entry point — grade ONE submission against ONE anchor
# ============================================================
def grade_submission(store, learner_id, session_id, anchor,
                     learner_lines, sbc_message=""):
    """Grade a single keyboard submission against the current anchor."""
    from . import slash_commands as _sc

    expanded = [_sc.expand(line) if line else "" for line in learner_lines]

    if sbc_message.strip():
        try:
            store.interventions.log(
                learner_id=learner_id,
                skill_id=anchor.skill_id,
                action_type="sbc_message",
                action_desc=sbc_message.strip()[:400],
                question_id=anchor.problem_id,
                session_id=session_id,
                context={"source": "keyboard"},
            )
        except Exception:
            pass

    learner_text = " ".join(expanded).strip()
    try:
        result = grade(learner_text, anchor.expected_answer)
    except Exception as e:
        return {"match": "error", "message": f"grader failed: {e}",
                "step_feedback": None, "raw_lines": expanded,
                "tanaka_message": "", "mastery": None}

    outcome = {
        "match": result.kind.name if hasattr(result.kind, "name") else str(result.kind),
        "matcher": getattr(result, "matcher", ""),
        "raw_lines": expanded,
        "step_feedback": None,
        "tanaka_message": "",
        "mastery": None,
    }

    if result.kind == MatchKind.MATCH:
        _save_attempt(store, session_id, learner_id, anchor.skill_id,
                      learner_text, f"D_{result.matcher}", "none")

        step_result = _step_feedback(anchor.problem_id, expanded)
        if step_result is None and getattr(anchor, "step_scheme", None):
            try:
                step_result = grade_steps_inline(anchor.step_scheme, expanded)
            except Exception:
                step_result = None

        if step_result is not None:
            outcome["step_feedback"] = {
                "awarded": step_result["awarded_marks"],
                "total":   step_result["total_marks"],
                "steps":   [
                    {"index": s.get("index"), "desc": s.get("desc"),
                     "marks": s.get("marks"),
                     "match_status": s.get("match_status"),
                     "first_form": (s.get("forms") or [None])[0]}
                    for s in step_result.get("steps", [])
                ],
            }
            try:
                missing = [s for s in step_result.get("steps", [])
                           if s.get("match_status") == "NOT_MATCHED"]
                for m in missing:
                    store.interventions.log(
                        learner_id=learner_id,
                        skill_id=anchor.skill_id,
                        action_type="memo_line_shown",
                        action_desc="Showed memo version of missed step: " + m["desc"],
                        question_id=anchor.problem_id,
                        session_id=session_id,
                        context={"step_index": m["index"], "step_desc": m["desc"]},
                    )
            except Exception:
                pass
            _close_open_interventions(store, learner_id, anchor.skill_id,
                                      step_result, anchor.problem_id, session_id)

        try:
            notes = check_presentation(expanded, topic=anchor.topic_v2, max_notes=1)
            if notes:
                outcome["tanaka_message"] = format_note(notes[0]).strip()
        except Exception:
            pass
        outcome["tanaka_message"] = outcome["tanaka_message"] or "Nice — the shape is right."

    elif result.kind == MatchKind.UNCERTAIN:
        outcome["tanaka_message"] = (
            "I want to be sure this is the shape the memo expects. "
            "Check each step against the method."
        )

    else:
        try:
            kind = diagnose_wrong(learner_text, anchor.expected_answer) or "unknown"
        except Exception:
            kind = "unknown"
        misconception_id = None
        try:
            diag = diagnose(anchor.skill_id, learner_text, anchor.expected_answer)
            if diag and diag.misconception_id:
                misconception_id = diag.misconception_id
        except Exception:
            pass
        _save_attempt(store, session_id, learner_id, anchor.skill_id,
                      learner_text, "D_NO_MATCH", kind, misconception_id)
        if kind == "unknown":
            try:
                store.discoveries.record(
                    learner_id=learner_id, skill_id=anchor.skill_id,
                    question_id=anchor.problem_id, learner_lines=expanded,
                    expected_answer=anchor.expected_answer, session_id=session_id,
                )
            except Exception:
                pass
        if kind == "sign_flip":
            outcome["tanaka_message"] = ("The number is right — the sign on one "
                                         "root looks flipped. Worth a look.")
        elif kind == "partial":
            outcome["tanaka_message"] = ("One root is right. The other is missing "
                                         "or different. Both matter here.")
        else:
            outcome["tanaka_message"] = ("There is something off. Show the working "
                                         "step by step.")
        if misconception_id:
            outcome["tanaka_message"] += f"  Noticed: {misconception_id}"

    _try_recompute_mastery(store, learner_id, anchor.skill_id)
    try:
        m = store.get_mastery(learner_id, anchor.skill_id)
        outcome["mastery"] = {"state": m.get("mastery_state"),
                              "passed": m.get("sittings_passed", 0)}
    except Exception:
        outcome["mastery"] = None

    return outcome
'''

# --- backup, write, report ---
if TARGET.exists():
    BACKUP.write_text(TARGET.read_text(encoding="utf-8"), encoding="utf-8")
    print(f"  ✓ backup written: {BACKUP.name}")

TARGET.write_text(NEW_DRILL, encoding="utf-8")
print(f"  ✓ {TARGET.name} rewritten ({len(NEW_DRILL):,} chars)")

# quick sanity import
import sys
ROOT_SRC = ROOT / "src"
if str(ROOT_SRC) not in sys.path:
    sys.path.insert(0, str(ROOT_SRC))

import importlib
import tutor.drill as _drill
importlib.reload(_drill)
print("  ✓ tutor.drill reloads cleanly")
print("  · grade_submission present:", hasattr(_drill, "grade_submission"))
print("  · set_line_provider present:", hasattr(_drill, "set_line_provider"))

  ✓ backup written: drill.py.bak
  ✓ drill.py rewritten (33,360 chars)
  ✓ tutor.drill reloads cleanly
  · grade_submission present: True
  · set_line_provider present: True


In [4]:
# ══════════════════════════════════════════════════════════════════
# Start the server in a background thread + health check
# ══════════════════════════════════════════════════════════════════
import sys, time, json, urllib.request
from pathlib import Path

sys.path.insert(0, str(Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\src")))

from tutor import drill_server

# in case you re-run this cell, don't start twice
if "SERVER_THREAD" not in dir():
    try:
        SERVER_THREAD = drill_server.serve_in_thread()
    except OSError as e:
        print("Port busy — server may already be running:", e)

time.sleep(0.5)

try:
    with urllib.request.urlopen("http://127.0.0.1:8765/health", timeout=2) as r:
        print("health:", r.read().decode())
except Exception as e:
    print("health check failed:", e)

drill_server at http://127.0.0.1:8765drill_server running in background thread on http://127.0.0.1:8765

  keyboard: http://127.0.0.1:8765/
  anchor:   2023_P1_Q1.1.1  |  learner: tanaka
health: {"ok": true, "anchor": "2023_P1_Q1.1.1"}


In [5]:
import sys, time, json, urllib.request, importlib
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
SRC  = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# 1) Confirm the file is really on disk
target = SRC / "tutor" / "drill_server.py"
print("drill_server.py exists:", target.exists(), f"({target.stat().st_size:,} bytes)" if target.exists() else "")

# 2) Invalidate import caches so the new file is seen
importlib.invalidate_caches()

# 3) Force a fresh import (not from cache)
if "tutor.drill_server" in sys.modules:
    del sys.modules["tutor.drill_server"]

from tutor import drill_server
print("import tutor.drill_server OK:", drill_server.__file__)

# 4) Start the server (don't double-start if already running)
try:
    SERVER_THREAD = drill_server.serve_in_thread()
except OSError as e:
    print("Port busy — server already running?", e)

time.sleep(0.5)

# 5) Health check
try:
    with urllib.request.urlopen("http://127.0.0.1:8765/health", timeout=2) as r:
        print("health:", r.read().decode())
except Exception as e:
    print("health check failed:", e)

drill_server.py exists: True (5,846 bytes)
import tutor.drill_server OK: C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\src\tutor\drill_server.py
drill_server at http://127.0.0.1:8765
  keyboard: http://127.0.0.1:8765/
  anchor:   2023_P1_Q1.1.1  |  learner: tanaka
drill_server running in background thread on http://127.0.0.1:8765
health: {"ok": true, "anchor": "2023_P1_Q1.1.1"}


In [6]:
import sys
from pathlib import Path

# What does Python think "tutor" is?
import tutor
print("tutor package loaded from:", tutor.__file__)
pkg_dir = Path(tutor.__file__).parent
print("tutor package dir:        ", pkg_dir)

# Where did we write the file?
expected = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\src\tutor\drill_server.py")
print("expected path:            ", expected)
print("expected exists:          ", expected.exists())

# Actually list what's in the package directory
print()
print("Files actually in the package directory:")
for f in sorted(pkg_dir.glob("*.py")):
    print("  ", f.name)

print()
# Is there a stray drill_server.py somewhere else?
hits = list(Path(r"C:\Users\Administrator\Desktop").rglob("drill_server.py"))
print("drill_server.py found at:")
for h in hits:
    print("  ", h)

tutor package loaded from: C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\src\tutor\__init__.py
tutor package dir:         C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\src\tutor
expected path:             C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\src\tutor\drill_server.py
expected exists:           True

Files actually in the package directory:
   __init__.py
   analogue_generators.py
   analogue_library.py
   answer_matcher.py
   diagnosis.py
   diagram_dispatcher.py
   discovery_store.py
   disengagement.py
   drill.py
   drill_server.py
   event_log.py
   hint_policy.py
   intervention_selector.py
   intervention_store.py
   learner_store.py
   llm_caller.py
   llm_config.py
   llm_conversation_prompt.py
   llm_diagnoser.py
   llm_gate.py
   llm_phraser.py
   mapping_adapter.py
   math_grader.py
   meta_library.py
   meta_prompt.py
   meta_renderer.py
   misconception_hints.py
   n08_loader.py
   presentation_matcher.py
   question

drill_server.py found at:
   C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\src\tutor\drill_server.py


In [7]:
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
TARGET = ROOT / "src" / "tutor" / "drill_server.py"

SERVER = r'''"""
drill_server.py — thin HTTP shell over drill.grade_submission.
"""
from __future__ import annotations

import json
import sys
import threading
import traceback
import uuid
from http.server import BaseHTTPRequestHandler, HTTPServer
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
SRC  = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from tutor import drill
from tutor.learner_store import open_store
from tutor.schemas import Problem

STORE = open_store(ROOT / "data" / "learner_store" / "learners.db")
ENGINE = None
LEARNER_ID = "tanaka"

ANCHORS = [
    Problem(
        problem_id="2023_P1_Q1.1.1",
        skill_id="algebra.quadratic.solve",
        topic="Algebra",
        subtopic="Quadratic equations",
        structure_type="routine_calculation",
        prompt="Solve for x:  x^2 + x - 12 = 0",
        expected_answer="x = -4 or x = 3",
    ),
]

SESSION = {
    "id": str(uuid.uuid4()),
    "anchor": ANCHORS[0],
}


class Handler(BaseHTTPRequestHandler):
    def log_message(self, *a):
        pass

    def _send(self, code, obj):
        body = json.dumps(obj, ensure_ascii=False).encode("utf-8")
        self.send_response(code)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(body)))
        self.send_header("Access-Control-Allow-Origin", "*")
        self.send_header("Access-Control-Allow-Headers", "Content-Type")
        self.send_header("Access-Control-Allow-Methods", "GET,POST,OPTIONS")
        self.end_headers()
        self.wfile.write(body)

    def do_OPTIONS(self):
        self._send(204, {})

    def do_GET(self):
        if self.path.startswith("/next_question"):
            a = SESSION["anchor"]
            return self._send(200, {
                "problem_id":      a.problem_id,
                "skill_id":        a.skill_id,
                "prompt":          a.prompt,
                "expected_answer": a.expected_answer,
                "session_id":      SESSION["id"],
            })
        if self.path.startswith("/health"):
            return self._send(200, {"ok": True,
                                    "anchor": SESSION["anchor"].problem_id})
        return self._send(404, {"error": "not found"})

    def do_POST(self):
        length = int(self.headers.get("Content-Length", "0") or 0)
        try:
            body = json.loads(self.rfile.read(length) or b"{}")
        except Exception:
            return self._send(400, {"error": "bad json"})

        if self.path.startswith("/reset_session"):
            SESSION["id"] = str(uuid.uuid4())
            return self._send(200, {"session_id": SESSION["id"]})

        if self.path.startswith("/grade"):
            lines = body.get("learner_lines_latex") or []
            if not isinstance(lines, list):
                return self._send(400, {"error": "learner_lines_latex must be a list"})
            sbc = body.get("sbc_message") or ""
            try:
                result = drill.grade_submission(
                    store=STORE,
                    learner_id=LEARNER_ID,
                    session_id=SESSION["id"],
                    anchor=SESSION["anchor"],
                    learner_lines=lines,
                    sbc_message=sbc,
                )
                return self._send(200, result)
            except Exception as e:
                traceback.print_exc()
                return self._send(500, {"error": str(e)})

        return self._send(404, {"error": "not found"})


def serve_forever(host="127.0.0.1", port=8765):
    print(f"drill_server at http://{host}:{port}")
    print(f"  anchor: {SESSION['anchor'].problem_id}  |  learner: {LEARNER_ID}")
    HTTPServer((host, port), Handler).serve_forever()


def serve_in_thread(host="127.0.0.1", port=8765):
    t = threading.Thread(target=serve_forever, args=(host, port), daemon=True)
    t.start()
    print(f"drill_server running in background thread on http://{host}:{port}")
    return t


if __name__ == "__main__":
    serve_forever()
'''

print("Will write to:", TARGET)
print("Parent exists:", TARGET.parent.exists())

TARGET.write_text(SERVER, encoding="utf-8")

print("After write:")
print("  exists:", TARGET.exists())
print("  size:  ", TARGET.stat().st_size if TARGET.exists() else "N/A")

Will write to: C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\src\tutor\drill_server.py
Parent exists: True
After write:
  exists: True
  size:   4200


In [8]:
import importlib, sys, time, urllib.request

# clear any cached stub
if "tutor.drill_server" in sys.modules:
    del sys.modules["tutor.drill_server"]
importlib.invalidate_caches()

from tutor import drill_server
print("import OK:", drill_server.__file__)

# start once
try:
    SERVER_THREAD = drill_server.serve_in_thread()
except OSError as e:
    print("Port busy:", e)

time.sleep(0.5)

try:
    with urllib.request.urlopen("http://127.0.0.1:8765/health", timeout=2) as r:
        print("health:", r.read().decode())
except Exception as e:
    print("health failed:", e)

import OK: C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\src\tutor\drill_server.py
drill_server at http://127.0.0.1:8765
  keyboard: http://127.0.0.1:8765/
  anchor:   2023_P1_Q1.1.1  |  learner: tanaka
drill_server running in background thread on http://127.0.0.1:8765


health: {"ok": true, "anchor": "2023_P1_Q1.1.1"}


In [9]:
# ══════════════════════════════════════════════════════════════════
# Patch docs/keyboard_preview.html — question card + fetch-based submit
# ══════════════════════════════════════════════════════════════════
from pathlib import Path
import re

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
HTML_PATH = ROOT / "docs" / "keyboard_preview.html"

html = HTML_PATH.read_text(encoding="utf-8")
original_len = len(html)
report = []

# ── Edit 1: insert question card right before the .err div ──
marker1 = '<div class="err" id="err">'
qcard_html = (
    '<div class="qcard" id="qcard">\n'
    '    <div class="qcard-head">\n'
    '      <span class="qcard-label">Question</span>\n'
    '      <span class="qcard-skill" id="qcardSkill"></span>\n'
    '    </div>\n'
    '    <div class="qcard-body" id="qcardBody">Loading question…</div>\n'
    '  </div>\n  '
)
if marker1 in html and 'id="qcard"' not in html:
    html = html.replace(marker1, qcard_html + marker1, 1)
    report.append("✓ question card HTML inserted")
elif 'id="qcard"' in html:
    report.append("· question card already present, skipped")
else:
    report.append("✗ marker for question card not found")

# ── Edit 2: add qcard CSS right before the closing </style> ──
qcard_css = (
    "/* question card */\n"
    ".qcard{padding:10px 12px;background:#f8fbff;border-bottom:1px solid var(--line)}\n"
    ".qcard-head{display:flex;justify-content:space-between;align-items:baseline;\n"
    "  font-size:10px;color:var(--ink2);font-weight:700;letter-spacing:.04em;\n"
    "  text-transform:uppercase;margin-bottom:4px}\n"
    ".qcard-skill{font-size:10px;font-weight:500;color:var(--ink3);\n"
    "  text-transform:none;letter-spacing:0}\n"
    ".qcard-body{font-size:15px;line-height:1.45;color:var(--ink);\n"
    "  font-family:'Cambria Math','Latin Modern Math',Georgia,serif}\n"
)
if '.qcard{' not in html:
    idx = html.rfind("</style>")
    if idx > 0:
        html = html[:idx] + qcard_css + html[idx:]
        report.append("✓ question card CSS added")
    else:
        report.append("✗ </style> not found")
else:
    report.append("· question card CSS already present, skipped")

# ── Edit 3: add API_BASE + loadQuestion() right after "var MQ = ..." ──
marker3 = "var MQ = MathQuill.getInterface(2);"
api_block = (
    marker3 + "\n"
    "  var API_BASE = \"http://127.0.0.1:8765\";\n\n"
    "  function loadQuestion() {\n"
    "    fetch(API_BASE + \"/next_question\")\n"
    "      .then(function (r) { return r.json(); })\n"
    "      .then(function (q) {\n"
    "        document.getElementById(\"qcardSkill\").textContent = q.skill_id || \"\";\n"
    "        document.getElementById(\"qcardBody\").textContent  = q.prompt   || \"\";\n"
    "      })\n"
    "      .catch(function () {\n"
    "        document.getElementById(\"qcardBody\").textContent =\n"
    "          \"Server not reachable. Start drill_server first.\";\n"
    "      });\n"
    "  }\n"
)
if marker3 in html and "function loadQuestion" not in html:
    html = html.replace(marker3, api_block, 1)
    report.append("✓ loadQuestion() added")
elif "function loadQuestion" in html:
    report.append("· loadQuestion already present, skipped")
else:
    report.append("✗ marker for MQ init not found")

# ── Edit 4: replace submit() ──
# Match from "function submit() {" to the matching closing brace.
# We use a heuristic: find "function submit()" then find the next
# "\n  }" that closes it at the same indent (2 spaces).
def replace_function(src, fn_name, new_body):
    start = src.find(f"function {fn_name}(")
    if start < 0:
        return src, False
    # find the opening brace
    brace = src.find("{", start)
    if brace < 0:
        return src, False
    # walk forward, tracking depth
    depth = 0
    i = brace
    while i < len(src):
        c = src[i]
        if c == "{":
            depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0:
                # include the closing brace
                return src[:start] + new_body + src[i+1:], True
        i += 1
    return src, False

new_submit = '''function submit() {
  var steps = [];
  state.fields.forEach(function (item) {
    var lx = item.field.latex();
    if (lx && lx.trim()) steps.push(lx.trim());
  });
  var payload = {
    learner_lines_latex: steps,
    sbc_message: sbcEl.value.trim(),
    tab_used: state.tab
  };

  resultEl.classList.add("shown");
  resultEl.innerHTML = "<div class='label'>grading…</div>";

  fetch(API_BASE + "/grade", {
    method: "POST",
    headers: { "Content-Type": "application/json" },
    body: JSON.stringify(payload)
  })
  .then(function (r) { return r.json(); })
  .then(function (res) {
    var out = [];
    out.push("<div class='label'>" + (res.match || "?") +
             (res.mastery ? "  ·  " + res.mastery.state : "") + "</div>");
    if (res.tanaka_message) {
      out.push("<div style='margin:4px 0 6px'>" + res.tanaka_message + "</div>");
    }
    if (res.step_feedback) {
      var sf = res.step_feedback;
      out.push("<div>Memo: " + sf.awarded + "/" + sf.total + "</div>");
      sf.steps.forEach(function (s) {
        if (s.match_status === "NOT_MATCHED") {
          out.push("<div>· skipped: " + (s.desc || "") +
                   "  (" + s.marks + ")  →  " + (s.first_form || "") + "</div>");
        }
      });
    }
    resultEl.innerHTML = out.join("");
  })
  .catch(function (err) {
    resultEl.innerHTML = "<div class='label'>server error</div><pre>" +
                         String(err) + "</pre>";
  });
}'''

if "grading…" in html:
    report.append("· submit() already patched, skipped")
else:
    html, ok = replace_function(html, "submit", new_submit)
    report.append("✓ submit() replaced" if ok else "✗ submit() not found")

# ── Edit 5: call loadQuestion() on boot, right before "renderTabs();" ──
boot_marker = "renderTabs();"
if "loadQuestion();" not in html and boot_marker in html:
    html = html.replace(boot_marker, "loadQuestion();\n  " + boot_marker, 1)
    report.append("✓ loadQuestion() called on boot")
elif "loadQuestion();" in html:
    report.append("· loadQuestion boot call already present, skipped")
else:
    report.append("✗ boot marker not found")

# ── Write only if the file changed ──
if len(html) != original_len:
    HTML_PATH.write_text(html, encoding="utf-8")
    print(f"  ✓ {HTML_PATH.name} updated ({len(html):,} chars)")
else:
    print("  · no changes made")

print()
for line in report:
    print("   ", line)

  · no changes made

    · question card already present, skipped
    · question card CSS already present, skipped
    · loadQuestion already present, skipped
    · submit() already patched, skipped
    · loadQuestion boot call already present, skipped


In [10]:
import sys
print("tutor.drill_server in sys.modules:", "tutor.drill_server" in sys.modules)
print("SERVER_THREAD in globals:        ", "SERVER_THREAD" in globals())
try:
    print("SERVER_THREAD alive:         ", SERVER_THREAD.is_alive())
except Exception as e:
    print("SERVER_THREAD not available:", e)

tutor.drill_server in sys.modules: False
SERVER_THREAD in globals:         True
SERVER_THREAD alive:          True


In [11]:
import socket, urllib.request

# 1) Is anything bound to port 8765?
s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
s.settimeout(1)
try:
    s.connect(("127.0.0.1", 8765))
    print("port 8765: CONNECTED (something is listening)")
    s.close()
except Exception as e:
    print("port 8765: NOT LISTENING —", e)

# 2) Try the health endpoint directly
try:
    with urllib.request.urlopen("http://127.0.0.1:8765/health", timeout=2) as r:
        print("health:", r.read().decode())
except Exception as e:
    print("health failed:", e)

# 3) What does the thread's traceback look like if it died?
import traceback, sys
print()
print("SERVER_THREAD.is_alive():", SERVER_THREAD.is_alive())
print("SERVER_THREAD.ident:     ", SERVER_THREAD.ident)

port 8765: CONNECTED (something is listening)
health: {"ok": true, "anchor": "2023_P1_Q1.1.1"}

SERVER_THREAD.is_alive(): True
SERVER_THREAD.ident:      20124


In [12]:
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
TARGET = ROOT / "src" / "tutor" / "drill_server.py"

src = TARGET.read_text(encoding="utf-8")

# 1) swap the import
src = src.replace(
    "from http.server import BaseHTTPRequestHandler, HTTPServer",
    "from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer",
)

# 2) swap the server class in serve_forever
src = src.replace(
    "    HTTPServer((host, port), Handler).serve_forever()",
    "    server = ThreadingHTTPServer((host, port), Handler)\n"
    "    server.daemon_threads = True\n"
    "    server.serve_forever()",
)

TARGET.write_text(src, encoding="utf-8")
print("✓ drill_server.py patched to use ThreadingHTTPServer")
print("  size:", TARGET.stat().st_size, "bytes")

# Verify the changes landed
check = TARGET.read_text(encoding="utf-8")
print("  ThreadingHTTPServer imported:", "ThreadingHTTPServer" in check)
print("  single-threaded HTTPServer still present:",
      "HTTPServer((host, port), Handler)" in check)

✓ drill_server.py patched to use ThreadingHTTPServer
  size: 4273 bytes
  ThreadingHTTPServer imported: True
  single-threaded HTTPServer still present: True


In [13]:
from pathlib import Path
ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
src = (ROOT / "src" / "tutor" / "drill_server.py").read_text(encoding="utf-8")

# Print the import line
for line in src.splitlines():
    if "http.server import" in line:
        print("IMPORT:", line.strip())

# Print the serve_forever body
import re
m = re.search(r"def serve_forever.*?(?=\ndef |\Z)", src, re.DOTALL)
if m:
    print()
    print("serve_forever body:")
    for line in m.group(0).splitlines():
        print("  ", line)

IMPORT: from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

serve_forever body:
   def serve_forever(host="127.0.0.1", port=8765):
       print(f"drill_server at http://{host}:{port}")
       print(f"  anchor: {SESSION['anchor'].problem_id}  |  learner: {LEARNER_ID}")
       server = ThreadingHTTPServer((host, port), Handler)
       server.daemon_threads = True
       server.serve_forever()
   


In [14]:
import sys, time, urllib.request, importlib
from pathlib import Path

sys.path.insert(0, str(Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\src")))

if "tutor.drill_server" in sys.modules:
    del sys.modules["tutor.drill_server"]
importlib.invalidate_caches()

from tutor import drill_server
SERVER_THREAD = drill_server.serve_in_thread()
time.sleep(0.6)

try:
    with urllib.request.urlopen("http://127.0.0.1:8765/health", timeout=3) as r:
        print("health:", r.read().decode())
except Exception as e:
    print("health failed:", e)

drill_server at http://127.0.0.1:8765drill_server running in background thread on http://127.0.0.1:8765

  keyboard: http://127.0.0.1:8765/
  anchor:   2023_P1_Q1.1.1  |  learner: tanaka
health: {"ok": true, "anchor": "2023_P1_Q1.1.1"}


In [15]:
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
TARGET = ROOT / "src" / "tutor" / "drill_server.py"

SERVER = r'''"""
drill_server.py — HTTP shell over drill.grade_submission.
Also serves the keyboard HTML + libs so everything is same-origin.
"""
from __future__ import annotations

import json
import sys
import threading
import traceback
import uuid
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
SRC  = ROOT / "src"
DOCS = ROOT / "docs"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from tutor import drill
from tutor.learner_store import open_store
from tutor.schemas import Problem

STORE = open_store(ROOT / "data" / "learner_store" / "learners.db")
ENGINE = None
LEARNER_ID = "tanaka"

ANCHORS = [
    Problem(
        problem_id="2023_P1_Q1.1.1",
        skill_id="algebra.quadratic.solve",
        topic="Algebra",
        subtopic="Quadratic equations",
        structure_type="routine_calculation",
        prompt="Solve for x:  x^2 + x - 12 = 0",
        expected_answer="x = -4 or x = 3",
    ),
]

SESSION = {
    "id": str(uuid.uuid4()),
    "anchor": ANCHORS[0],
}


class Handler(BaseHTTPRequestHandler):
    def log_message(self, *a):
        pass

    # ---------- json response ----------
    def _send(self, code, obj):
        body = json.dumps(obj, ensure_ascii=False).encode("utf-8")
        self.send_response(code)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    # ---------- static file ----------
    def _serve_file(self, rel_path):
        target = (DOCS / rel_path).resolve()
        try:
            target.relative_to(DOCS.resolve())
        except ValueError:
            return self._send(403, {"error": "forbidden"})
        if not target.exists() or not target.is_file():
            return self._send(404, {"error": "not found"})
        ctype = {
            ".html": "text/html; charset=utf-8",
            ".css":  "text/css; charset=utf-8",
            ".js":   "application/javascript; charset=utf-8",
            ".json": "application/json; charset=utf-8",
            ".png":  "image/png",
            ".svg":  "image/svg+xml",
            ".ico":  "image/x-icon",
        }.get(target.suffix.lower(), "application/octet-stream")
        data = target.read_bytes()
        self.send_response(200)
        self.send_header("Content-Type", ctype)
        self.send_header("Content-Length", str(len(data)))
        self.send_header("Cache-Control", "no-store")
        self.end_headers()
        self.wfile.write(data)

    def do_OPTIONS(self):
        self.send_response(204)
        self.send_header("Access-Control-Allow-Origin", "*")
        self.send_header("Access-Control-Allow-Headers", "Content-Type")
        self.send_header("Access-Control-Allow-Methods", "GET,POST,OPTIONS")
        self.end_headers()

    def do_GET(self):
        path = self.path.split("?", 1)[0]

        if path in ("/", "/index.html", "/keyboard_preview.html"):
            return self._serve_file("keyboard_preview.html")

        if path.startswith("/libs/"):
            return self._serve_file(path[1:])

        if path.startswith("/next_question"):
            a = SESSION["anchor"]
            return self._send(200, {
                "problem_id":      a.problem_id,
                "skill_id":        a.skill_id,
                "prompt":          a.prompt,
                "expected_answer": a.expected_answer,
                "session_id":      SESSION["id"],
            })

        if path.startswith("/health"):
            return self._send(200, {"ok": True,
                                    "anchor": SESSION["anchor"].problem_id})

        return self._send(404, {"error": "not found"})

    def do_POST(self):
        length = int(self.headers.get("Content-Length", "0") or 0)
        try:
            body = json.loads(self.rfile.read(length) or b"{}")
        except Exception:
            return self._send(400, {"error": "bad json"})

        if self.path.startswith("/reset_session"):
            SESSION["id"] = str(uuid.uuid4())
            return self._send(200, {"session_id": SESSION["id"]})

        if self.path.startswith("/grade"):
            lines = body.get("learner_lines_latex") or []
            if not isinstance(lines, list):
                return self._send(400, {"error": "learner_lines_latex must be a list"})
            sbc = body.get("sbc_message") or ""
            try:
                result = drill.grade_submission(
                    store=STORE,
                    learner_id=LEARNER_ID,
                    session_id=SESSION["id"],
                    anchor=SESSION["anchor"],
                    learner_lines=lines,
                    sbc_message=sbc,
                )
                return self._send(200, result)
            except Exception as e:
                traceback.print_exc()
                return self._send(500, {"error": str(e)})

        return self._send(404, {"error": "not found"})


def serve_forever(host="127.0.0.1", port=8765):
    print(f"drill_server at http://{host}:{port}")
    print(f"  keyboard: http://{host}:{port}/")
    print(f"  anchor:   {SESSION['anchor'].problem_id}  |  learner: {LEARNER_ID}")
    server = ThreadingHTTPServer((host, port), Handler)
    server.daemon_threads = True
    server.serve_forever()


def serve_in_thread(host="127.0.0.1", port=8765):
    t = threading.Thread(target=serve_forever, args=(host, port), daemon=True)
    t.start()
    print(f"drill_server running in background thread on http://{host}:{port}")
    return t


if __name__ == "__main__":
    serve_forever()
'''

TARGET.write_text(SERVER, encoding="utf-8")
print(f"✓ {TARGET.name} rewritten ({TARGET.stat().st_size:,} bytes)")

✓ drill_server.py rewritten (5,846 bytes)


In [16]:
import sys, time, urllib.request, importlib
from pathlib import Path

sys.path.insert(0, str(Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\src")))

if "tutor.drill_server" in sys.modules:
    del sys.modules["tutor.drill_server"]
importlib.invalidate_caches()

from tutor import drill_server
SERVER_THREAD = drill_server.serve_in_thread()
time.sleep(0.6)

try:
    with urllib.request.urlopen("http://127.0.0.1:8765/health", timeout=3) as r:
        print("health:", r.read().decode())
except Exception as e:
    print("health failed:", e)

drill_server at http://127.0.0.1:8765drill_server running in background thread on http://127.0.0.1:8765

  keyboard: http://127.0.0.1:8765/
  anchor:   2023_P1_Q1.1.1  |  learner: tanaka
health: {"ok": true, "anchor": "2023_P1_Q1.1.1"}


In [17]:
import urllib.request, urllib.error, socket, sys

def probe(path):
    url = "http://127.0.0.1:8765" + path
    try:
        with urllib.request.urlopen(url, timeout=3) as r:
            data = r.read()
            print(f"{path:<25} {r.status}  {len(data):>6} bytes  ct={r.headers.get('Content-Type')}")
            return data
    except urllib.error.HTTPError as e:
        print(f"{path:<25} HTTP {e.code}")
        return None
    except Exception as e:
        print(f"{path:<25} FAIL  {type(e).__name__}: {e}")
        return None

# Is anything on the port?
s = socket.socket(); s.settimeout(1)
try:
    s.connect(("127.0.0.1", 8765))
    print("port 8765: CONNECTED")
    s.close()
except Exception as e:
    print("port 8765: NOT LISTENING —", e)

print()

# Does the server respond to each endpoint?
probe("/health")
probe("/next_question")
html = probe("/")
probe("/libs/mathquill.min.js")
probe("/libs/mathquill.min.css")

print()
if html:
    text = html.decode("utf-8", errors="replace")
    print("--- first 400 chars of what / served ---")
    print(text[:400])
    print()
    print("Contains 'qcardBody':", "qcardBody" in text)
    print("Contains 'API_BASE':  ", "API_BASE" in text)
    print("Contains 'libs/':     ", "libs/" in text)

port 8765: CONNECTED

/health                   200      40 bytes  ct=application/json; charset=utf-8
/next_question            200     207 bytes  ct=application/json; charset=utf-8
/                         200   34287 bytes  ct=text/html; charset=utf-8
/libs/mathquill.min.js    200   74920 bytes  ct=application/javascript; charset=utf-8
/libs/mathquill.min.css   200    9095 bytes  ct=text/css; charset=utf-8

--- first 400 chars of what / served ---

<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8" />
<meta name="viewport" content="width=device-width, initial-scale=1, maximum-scale=1, user-scalable=no" />
<title>MethodMarker Keyboard</title>
<link rel="stylesheet" href="./libs/mathquill.min.css" />
<script src="./libs/jquery.min.js"></script>
<script src="./libs/mathquill.min.js"></script>
<style>
:root {
  --bg:#eef2

Contains 'qcardBody': True
Contains 'API_BASE':   True
Contains 'libs/':      True


In [18]:
import sys, time, urllib.request, urllib.error, importlib
from pathlib import Path

# 1) fresh import of the file-serving version
sys.path.insert(0, str(Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\src")))
if "tutor.drill_server" in sys.modules:
    del sys.modules["tutor.drill_server"]
importlib.invalidate_caches()

from tutor import drill_server
print("loaded:", drill_server.__file__)

# 2) start
try:
    SERVER_THREAD = drill_server.serve_in_thread()
except OSError as e:
    print("port busy — killing and retrying:", e)
    import socket, subprocess
    # find PID holding the port (Windows)
    out = subprocess.run(
        ["netstat", "-ano"], capture_output=True, text=True
    ).stdout
    for line in out.splitlines():
        if ":8765" in line and "LISTENING" in line:
            pid = line.split()[-1]
            print("killing PID:", pid)
            subprocess.run(["taskkill", "/F", "/PID", pid], capture_output=True)
    time.sleep(0.8)
    SERVER_THREAD = drill_server.serve_in_thread()

time.sleep(0.8)

# 3) probe every endpoint
def probe(path):
    url = "http://127.0.0.1:8765" + path
    try:
        with urllib.request.urlopen(url, timeout=3) as r:
            data = r.read()
            print(f"  {path:<25} {r.status}  {len(data):>6} bytes  ct={r.headers.get('Content-Type')}")
            return data
    except urllib.error.HTTPError as e:
        print(f"  {path:<25} HTTP {e.code}")
        return None
    except Exception as e:
        print(f"  {path:<25} FAIL  {type(e).__name__}: {e}")
        return None

print()
print("probing endpoints:")
probe("/health")
probe("/next_question")
html = probe("/")
probe("/libs/mathquill.min.js")
probe("/libs/mathquill.min.css")

if html:
    text = html.decode("utf-8", errors="replace")
    print()
    print("served HTML contains:")
    print("  qcardBody:    ", "qcardBody" in text)
    print("  API_BASE:     ", "API_BASE" in text)
    print("  libs refs:    ", "./libs/" in text or "libs/" in text)

print()
print("If /health returned ok and / returned 200 with the HTML,",)
print("open http://127.0.0.1:8765/ in Chrome (not the file:// URL).")

loaded: C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\src\tutor\drill_server.py
drill_server at http://127.0.0.1:8765
  keyboard: http://127.0.0.1:8765/
  anchor:   2023_P1_Q1.1.1  |  learner: tanaka
drill_server running in background thread on http://127.0.0.1:8765

probing endpoints:
  /health                   200      40 bytes  ct=application/json; charset=utf-8
  /next_question            200     207 bytes  ct=application/json; charset=utf-8
  /                         200   34287 bytes  ct=text/html; charset=utf-8
  /libs/mathquill.min.js    200   74920 bytes  ct=application/javascript; charset=utf-8
  /libs/mathquill.min.css   200    9095 bytes  ct=text/css; charset=utf-8

served HTML contains:
  qcardBody:     True
  API_BASE:      True
  libs refs:     True

If /health returned ok and / returned 200 with the HTML,
open http://127.0.0.1:8765/ in Chrome (not the file:// URL).


In [19]:
import urllib.request

with urllib.request.urlopen("http://127.0.0.1:8765/", timeout=3) as r:
    html = r.read().decode("utf-8", errors="replace")

checks = {
    "qcard div present":      'id="qcard"' in html,
    "qcardBody present":      'id="qcardBody"' in html,
    "loading placeholder":    'Loading question' in html,
    "API_BASE defined":       'API_BASE' in html,
    "loadQuestion defined":   'function loadQuestion' in html,
    "loadQuestion called":    'loadQuestion();' in html,
    "submit patched":         'grading' in html,
    "mathquill lib ref":      './libs/mathquill' in html or 'libs/mathquill' in html,
}
for k, v in checks.items():
    print(f"  {'✓' if v else '✗'}  {k}")

print()
print("--- last 500 chars of served HTML ---")
print(html[-500:])

  ✓  qcard div present
  ✓  qcardBody present
  ✓  loading placeholder
  ✓  API_BASE defined
  ✓  loadQuestion defined
  ✓  loadQuestion called
  ✓  submit patched
  ✓  mathquill lib ref

--- last 500 chars of served HTML ---
el.className = "act";
      if (k.value === "__done__") el.classList.add("done");
      if (k.value === "__abc__" && state.abcMode) el.classList.add("mode-on");
      el.textContent = k.label;
      el.onmousedown = function (e) { e.preventDefault(); };
      el.onclick = function () { handleKey(k); };
      actsEl.appendChild(el);
    });
  }

  renderTabs();
  renderActs();
  renderKb();
  addLine(true);
  updateTargetUI();
  loadQuestion();
})();
</script>
</body>
</html>



In [20]:
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
HTML = ROOT / "docs" / "keyboard_preview.html"

html = HTML.read_text(encoding="utf-8")
before = len(html)

# The boot sequence is: renderTabs(); renderActs(); renderKb(); addLine(true); updateTargetUI();
# We want to fire loadQuestion() right at the end of that block, just before the
# IIFE closes. Anchor on the unique 5-line block.
boot_anchor = (
    "  renderTabs();\n"
    "  renderActs();\n"
    "  renderKb();\n"
    "  addLine(true);\n"
    "  updateTargetUI();\n"
    "})();"
)

boot_patched = (
    "  renderTabs();\n"
    "  renderActs();\n"
    "  renderKb();\n"
    "  addLine(true);\n"
    "  updateTargetUI();\n"
    "  loadQuestion();\n"
    "})();"
)

if boot_patched in html:
    print("· boot call already present — no change")
elif boot_anchor in html:
    html = html.replace(boot_anchor, boot_patched, 1)
    print("✓ loadQuestion() added to boot sequence")
else:
    print("✗ boot anchor not found — need to inspect manually")

# Also remove the spurious call inside the tab onclick handler, if it's there.
# That handler looks like:  renderTabs(); renderActs(); renderKb();
spurious = "loadQuestion();\n      renderTabs(); renderActs(); renderKb();"
if spurious in html:
    html = html.replace(spurious, "renderTabs(); renderActs(); renderKb();", 1)
    print("✓ removed spurious loadQuestion() from tab onclick")

if len(html) != before:
    HTML.write_text(html, encoding="utf-8")
    print(f"  file written ({len(html):,} chars)")

# Verify
check = HTML.read_text(encoding="utf-8")
# The boot call should be the LAST occurrence
import re
calls = [m.start() for m in re.finditer(r"loadQuestion\(\);", check)]
print()
print(f"loadQuestion(); appears {len(calls)} time(s) at positions {calls}")
if calls:
    tail = check[calls[-1]:calls[-1]+200]
    print("  last occurrence context:")
    print("  " + tail.replace("\n", "\n  ")[:200])

· boot call already present — no change

loadQuestion(); appears 2 time(s) at positions [28555, 33170]
  last occurrence context:
  loadQuestion();
  })();
  </script>
  </body>
  </html>
  


In [21]:
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
TARGET = ROOT / "src" / "tutor" / "drill.py"

# ── the converter block ──
CONVERTER = r'''# ============================================================
# LaTeX → plain text (MathQuill → memo-gradeable strings)
# ============================================================
import re as _re_latex


_LATEX_MAP = [
    (r"\left", ""), (r"\right", ""),
    (r"\times", "*"), (r"\div", "/"), (r"\cdot", "*"),
    (r"\pm", "+-"),
    (r"\leq", "<="), (r"\geq", ">="), (r"\neq", "!="),
    (r"\le", "<="), (r"\ge", ">="), (r"\ne", "!="),
    (r"\infty", "inf"), (r"\to", "->"),
    (r"\Rightarrow", "=>"), (r"\Leftarrow", "<="),
    (r"\therefore", "therefore"), (r"\because", "because"),
    (r"\alpha", "alpha"), (r"\beta", "beta"), (r"\gamma", "gamma"),
    (r"\theta", "theta"), (r"\pi", "pi"), (r"\sigma", "sigma"),
    (r"\Delta", "Delta"), (r"\Omega", "Omega"),
    (r"\sum", "sum"), (r"\int", "int"),
    (r"\sin", "sin"), (r"\cos", "cos"), (r"\tan", "tan"),
    (r"\log", "log"), (r"\ln", "ln"),
    (r"\,", " "), (r"\;", " "), (r"\!", ""), (r"\ ", " "),
]


def _latex_to_plain(s: str) -> str:
    """Turn MathQuill-style LaTeX into text the memo grader can compare."""
    if not s:
        return s
    out = s

    # \frac{a}{b}  →  (a)/(b)   — repeat for nested fractions
    frac = _re_latex.compile(r"\\frac\{([^{}]*)\}\{([^{}]*)\}")
    while frac.search(out):
        out = frac.sub(r"(\1)/(\2)", out)

    # \sqrt{a}  or  \sqrt[n]{a}  →  sqrt(a)
    rt = _re_latex.compile(r"\\sqrt(?:\[[^\]]*\])?\{([^{}]*)\}")
    while rt.search(out):
        out = rt.sub(r"sqrt(\1)", out)

    # ^{x} → ^x  and  _{x} → _x
    out = _re_latex.sub(r"\^\{([^{}]*)\}", r"^\1", out)
    out = _re_latex.sub(r"_\{([^{}]*)\}", r"_\1", out)

    # \text{...} → the text itself
    out = _re_latex.sub(r"\\text\{([^{}]*)\}", r"\1", out)

    # \hat{A} → A_hat  ·  \bar{x} → x_bar
    out = _re_latex.sub(r"\\hat\{([^{}]*)\}", r"\1_hat", out)
    out = _re_latex.sub(r"\\bar\{([^{}]*)\}", r"\1_bar", out)

    # simple command replacements
    for pat, rep in _LATEX_MAP:
        out = out.replace(pat, rep)

    # collapse whitespace
    out = _re_latex.sub(r"\s+", " ", out).strip()
    return out

'''

src = TARGET.read_text(encoding="utf-8")

if "_latex_to_plain" in src:
    print("· drill.py already has the converter — no change")
else:
    # ── 1. Insert converter right before SITTING_SIZE ──
    anchor = "SITTING_SIZE = 5\n"
    if anchor not in src:
        raise SystemExit("✗ anchor SITTING_SIZE not found")
    src = src.replace(anchor, CONVERTER + "\n" + anchor, 1)
    print("✓ converter inserted")

    # ── 2. Patch grade_submission to convert before grading ──
    old_join = '    learner_text = " ".join(expanded).strip()'
    new_join = (
        '    plain_lines = [_latex_to_plain(x) for x in expanded]\n'
        '    learner_text = " ".join(plain_lines).strip()'
    )
    if old_join in src:
        src = src.replace(old_join, new_join, 1)
        print("✓ grade_submission: learner_text now uses plain")
    else:
        print("✗ could not find join line")

    # ── 3. Also feed the step grader the plain lines ──
    old_step = '        step_result = _step_feedback(anchor.problem_id, expanded)'
    new_step = '        step_result = _step_feedback(anchor.problem_id, plain_lines)'
    if old_step in src:
        src = src.replace(old_step, new_step, 1)
        print("✓ step grader uses plain lines")

    old_inline = '                step_result = grade_steps_inline(anchor.step_scheme, expanded)'
    new_inline = '                step_result = grade_steps_inline(anchor.step_scheme, plain_lines)'
    if old_inline in src:
        src = src.replace(old_inline, new_inline, 1)
        print("✓ inline step grader uses plain lines")

    TARGET.write_text(src, encoding="utf-8")
    print(f"✓ {TARGET.name} written ({len(src):,} chars)")

# ── verify with quick tests ──
import sys, importlib
sys.path.insert(0, str(ROOT / "src"))
if "tutor.drill" in sys.modules:
    del sys.modules["tutor.drill"]
importlib.invalidate_caches()
from tutor import drill

print()
print("converter tests:")
tests = [
    r"\left(x+4\right)\left(x-3\right)=0",
    r"x^{2}+x-12=0",
    r"\frac{1}{2}",
    r"\sqrt{b^2-4ac}",
    r"x = -4 \text{ or } x = 3",
    r"\sin^{2}\theta + \cos^{2}\theta = 1",
]
for t in tests:
    print(f"  {t!r:55} → {drill._latex_to_plain(t)!r}")

✓ converter inserted
✓ grade_submission: learner_text now uses plain
✓ step grader uses plain lines
✓ inline step grader uses plain lines
✓ drill.py written (35,529 chars)

converter tests:
  '\\left(x+4\\right)\\left(x-3\\right)=0'                → '(x+4)(x-3)=0'
  'x^{2}+x-12=0'                                          → 'x^2+x-12=0'
  '\\frac{1}{2}'                                          → '(1)/(2)'
  '\\sqrt{b^2-4ac}'                                       → 'sqrt(b^2-4ac)'
  'x = -4 \\text{ or } x = 3'                             → 'x = -4 or x = 3'
  '\\sin^{2}\\theta + \\cos^{2}\\theta = 1'               → 'sin^2theta + cos^2theta = 1'


In [22]:
import sys, importlib
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
sys.path.insert(0, str(ROOT / "src"))

# Clear BOTH the sys.modules entry and the parent-package attribute
import tutor
if hasattr(tutor, "drill"):
    delattr(tutor, "drill")
if "tutor.drill" in sys.modules:
    del sys.modules["tutor.drill"]
importlib.invalidate_caches()

from tutor import drill
print("reloaded:", drill.__file__)
print("has _latex_to_plain:", hasattr(drill, "_latex_to_plain"))

if hasattr(drill, "_latex_to_plain"):
    print()
    print("converter tests:")
    tests = [
        r"\left(x+4\right)\left(x-3\right)=0",
        r"x^{2}+x-12=0",
        r"\frac{1}{2}",
        r"\sqrt{b^2-4ac}",
        r"x = -4 \text{ or } x = 3",
        r"\sin^{2}\theta + \cos^{2}\theta = 1",
    ]
    for t in tests:
        print(f"  {t!r:55} → {drill._latex_to_plain(t)!r}")
else:
    print("File doesn't have the converter — need to re-run the previous cell.")

reloaded: C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\src\tutor\drill.py
has _latex_to_plain: True

converter tests:
  '\\left(x+4\\right)\\left(x-3\\right)=0'                → '(x+4)(x-3)=0'
  'x^{2}+x-12=0'                                          → 'x^2+x-12=0'
  '\\frac{1}{2}'                                          → '(1)/(2)'
  '\\sqrt{b^2-4ac}'                                       → 'sqrt(b^2-4ac)'
  'x = -4 \\text{ or } x = 3'                             → 'x = -4 or x = 3'
  '\\sin^{2}\\theta + \\cos^{2}\\theta = 1'               → 'sin^2theta + cos^2theta = 1'


In [23]:
from pathlib import Path
import sys

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
HTML = ROOT / "docs" / "keyboard_preview.html"

html = HTML.read_text(encoding="utf-8")
before = len(html)

# ── 1. Pull the result div out of its current position ──
result_block = '  <div class="result" id="result"></div>'
if result_block in html:
    html = html.replace(result_block + "\n", "", 1)
    html = html.replace(result_block, "", 1)
    print("✓ removed result strip from current position")
else:
    print("· result strip not found where expected — inspecting")
    # try to locate any result div
    import re
    m = re.search(r'<div class="result"[^>]*>\s*</div>', html)
    if m:
        result_block = m.group(0)
        html = html[:m.start()] + html[m.end():]
        print("✓ removed (matched loosely):", result_block[:60])

# ── 2. Put it right after the working area, before Ask Tanaka ──
sbc_marker = '  <div class="sbc" id="sbcWrap">'
if sbc_marker in html:
    html = html.replace(
        sbc_marker,
        '  ' + result_block.strip() + '\n\n' + sbc_marker,
        1,
    )
    print("✓ result strip placed directly under working area")
else:
    # fallback: any .sbc div
    sbc_marker2 = '  <div class="sbc"'
    if sbc_marker2 in html:
        html = html.replace(
            sbc_marker2,
            '  ' + result_block.strip() + '\n\n' + sbc_marker2,
            1,
        )
        print("✓ result strip placed under working area (fallback marker)")
    else:
        print("✗ could not find .sbc marker to insert before")

# ── 3. Adjust result CSS so it's visible when shown, sits compactly ──
# Make it look like part of the working area rather than a footer.
css_old = """.result{padding:8px 12px;background:#0b1f44;color:#fff;font-family:ui-monospace,Menlo,monospace;font-size:10px;line-height:1.45;
  max-height:0;overflow:hidden;transition:max-height .25s ease;flex:none}"""

css_new = """.result{margin:0;padding:10px 14px;background:#0b1f44;color:#fff;
  font-family:ui-monospace,Menlo,monospace;font-size:11px;line-height:1.5;
  max-height:0;overflow:hidden;transition:max-height .25s ease;flex:none;
  border-top:1px solid #0b1f44}
.result.shown{max-height:260px;overflow-y:auto}"""

if css_old in html:
    html = html.replace(css_old, css_new, 1)
    print("✓ result CSS tuned for inline display")
else:
    # try to find and rewrite the .result block loosely
    import re
    pattern = r"\.result\{[^}]*\}"
    m = re.search(pattern, html)
    if m:
        old = m.group(0)
        new = (".result{margin:0;padding:10px 14px;background:#0b1f44;color:#fff;"
               "font-family:ui-monospace,Menlo,monospace;font-size:11px;line-height:1.5;"
               "max-height:0;overflow:hidden;transition:max-height .25s ease;flex:none;"
               "border-top:1px solid #0b1f44}")
        html = html.replace(old, new, 1)
        print("✓ result CSS rewritten (loose match)")

# ── 4. Also fix .shown rule if it's separate ──
html = html.replace(
    ".result.shown{max-height:200px;overflow-y:auto}",
    ".result.shown{max-height:260px;overflow-y:auto}",
)

HTML.write_text(html, encoding="utf-8")
print(f"  ✓ {HTML.name} written ({len(html):,} chars)")

# ── 5. Show the current server-thread state ──
print()
print("Server status:")
if "tutor.drill_server" in sys.modules:
    mod = sys.modules["tutor.drill_server"]
    print("  module loaded from:", getattr(mod, "__file__", "?"))
else:
    print("  drill_server module NOT loaded in this kernel")

from tutor import drill
print("  drill module has _latex_to_plain:", hasattr(drill, "_latex_to_plain"))
print("  drill module file:             ", drill.__file__)

# Is the old thread still running?
import socket
s = socket.socket(); s.settimeout(1)
try:
    s.connect(("127.0.0.1", 8765))
    print("  port 8765: BOUND (server still up — needs restart)")
    s.close()
except Exception:
    print("  port 8765: FREE (server down)")

✓ removed result strip from current position
✓ result strip placed directly under working area
✓ result CSS rewritten (loose match)
  ✓ keyboard_preview.html written (33,219 chars)

Server status:
  drill_server module NOT loaded in this kernel
  drill module has _latex_to_plain: True
  drill module file:              C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\src\tutor\drill.py
  port 8765: BOUND (server still up — needs restart)


In [24]:
import sys
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
SRC  = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("sys.path entry added:", str(SRC))
print("tutor package at:   ", (SRC / "tutor" / "__init__.py").exists())

sys.path entry added: C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\src
tutor package at:    True


In [25]:
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
TARGET = ROOT / "src" / "tutor" / "drill.py"

src = TARGET.read_text(encoding="utf-8")

# Add a normaliser helper after the _latex_to_plain block
NORM = r'''

# ============================================================
# Character-level normaliser — makes Unicode math match ASCII
# ============================================================
_UNICODE_FIX = {
    "\u2212": "-",   # − minus sign
    "\u2013": "-",   # – en dash
    "\u2014": "-",   # — em dash
    "\u00d7": "*",   # × multiplication
    "\u00f7": "/",   # ÷ division
    "\u2264": "<=",  # ≤
    "\u2265": ">=",  # ≥
    "\u2260": "!=",  # ≠
    "\u2192": "->",  # →
    "\u21d2": "=>",  # ⇒
    "\u2234": "therefore",
    "\u2235": "because",
    "\u03b1": "alpha", "\u03b2": "beta", "\u03b8": "theta",
    "\u03c0": "pi",    "\u03c3": "sigma", "\u221e": "inf",
    "\u221a": "sqrt",
}


def _normalise(s: str) -> str:
    """Flatten to a canonical form the memo grader can compare."""
    if not s:
        return ""
    out = s
    for k, v in _UNICODE_FIX.items():
        out = out.replace(k, v)
    # collapse whitespace, lowercase, drop decorative spaces around symbols
    import re as _rn
    out = _rn.sub(r"\s+", "", out).lower()
    out = out.replace("or", "|")
    return out

'''

if "_normalise" in src:
    print("· normaliser already present")
else:
    anchor = "def _latex_to_plain(s: str) -> str:"
    idx = src.find(anchor)
    if idx < 0:
        raise SystemExit("✗ cannot find _latex_to_plain to insert after")
    # find the end of _latex_to_plain (next top-level def or class after it)
    rest = src[idx:]
    nxt = rest.find("\ndef ", 10)
    if nxt < 0:
        nxt = rest.find("\nclass ", 10)
    if nxt < 0:
        raise SystemExit("✗ cannot find end of _latex_to_plain block")
    insert_at = idx + nxt
    src = src[:insert_at] + NORM + src[insert_at:]
    print("✓ normaliser inserted")

# Patch the retry block in grade_submission to use _normalise
old_retry = '''        if result.kind != MatchKind.MATCH:
            import re as _re2
            n_learner = _re2.sub(r"\\s+", "", learner_text).lower()
            n_expect  = _re2.sub(r"\\s+", "", anchor.expected_answer).lower()
            result2 = grade(n_learner, n_expect)
            if result2.kind == MatchKind.MATCH:
                result = result2'''

new_retry = '''        if result.kind != MatchKind.MATCH:
            n_learner = _normalise(learner_text)
            n_expect  = _normalise(anchor.expected_answer)
            # direct string match on the normalised forms
            if n_learner and n_expect and n_learner == n_expect:
                from .math_grader import MatchResult
                try:
                    result = MatchResult(kind=MatchKind.MATCH,
                                         matcher="normalised")
                except Exception:
                    # MatchResult constructor differs — fall back
                    result2 = grade(n_learner, n_expect)
                    if result2.kind == MatchKind.MATCH:
                        result = result2
            else:
                result2 = grade(n_learner, n_expect)
                if result2.kind == MatchKind.MATCH:
                    result = result2'''

if "n_learner = _normalise" in src:
    print("· retry block already uses _normalise")
elif old_retry in src:
    src = src.replace(old_retry, new_retry, 1)
    print("✓ retry block uses _normalise")
else:
    print("· retry block not found in expected form")

TARGET.write_text(src, encoding="utf-8")
print(f"✓ {TARGET.name} written ({len(src):,} chars)")

# Verify the normaliser works
import sys, importlib
sys.path.insert(0, str(ROOT / "src"))
import tutor
if hasattr(tutor, "drill"):
    delattr(tutor, "drill")
if "tutor.drill" in sys.modules:
    del sys.modules["tutor.drill"]
importlib.invalidate_caches()
from tutor import drill

print()
print("normaliser tests:")
tests = [
    ("x=\u22124 or x=3",  "x = -4 or x = 3"),
    ("x=-4 or x=3",        "x = -4 or x = 3"),
    ("(x+4)(x\u22123)=0",  "(x+4)(x-3)=0"),
    ("x\u00b2+x\u221212=0","x^2 + x - 12 = 0"),
]
for a, b in tests:
    na, nb = drill._normalise(a), drill._normalise(b)
    ok = "MATCH" if na == nb else "diff"
    print(f"  {a!r:30} vs {b!r:25} → {ok}")
    if na != nb:
        print(f"     normalised: {na!r}")
        print(f"     normalised: {nb!r}")

✓ normaliser inserted
· retry block not found in expected form
✓ drill.py written (36,635 chars)

normaliser tests:
  'x=−4 or x=3'                  vs 'x = -4 or x = 3'         → MATCH
  'x=-4 or x=3'                  vs 'x = -4 or x = 3'         → MATCH
  '(x+4)(x−3)=0'                 vs '(x+4)(x-3)=0'            → MATCH
  'x²+x−12=0'                    vs 'x^2 + x - 12 = 0'        → diff
     normalised: 'x²+x-12=0'
     normalised: 'x^2+x-12=0'


In [26]:
import sys, importlib, json
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
sys.path.insert(0, str(ROOT / "src"))

# Fresh import
import tutor
for mod in ["tutor.drill", "tutor.math_grader"]:
    if mod in sys.modules:
        del sys.modules[mod]
importlib.invalidate_caches()

from tutor import drill
from tutor.math_grader import grade, MatchKind

# ── exactly what the browser sends ──
learner_lines = [
    "(x+4)(x-3) = 0",
    "x = -4 or x = 3",
]
expected = "x = -4 or x = 3"

print("=== 1. RAW INPUT (what browser sends) ===")
for i, l in enumerate(learner_lines):
    print(f"  line {i+1}: {l!r}")

print()
print("=== 2. AFTER _latex_to_plain ===")
for l in learner_lines:
    print(f"  {l!r}")
    print(f"    → {drill._latex_to_plain(l)!r}")

print()
print("=== 3. AFTER _normalise ===")
print(f"  learner last line: {drill._normalise(learner_lines[-1])!r}")
print(f"  expected:          {drill._normalise(expected)!r}")
print(f"  equal?             {drill._normalise(learner_lines[-1]) == drill._normalise(expected)}")

print()
print("=== 4. grade() calls, in order of what should be tried ===")
learner_text = " ".join(learner_lines)
for label, a, b in [
    ("full join vs expected",        learner_text, expected),
    ("last line vs expected",        learner_lines[-1], expected),
    ("last line plain vs expected",  drill._latex_to_plain(learner_lines[-1]), expected),
    ("last line norm vs expected",   drill._normalise(learner_lines[-1]), drill._normalise(expected)),
]:
    try:
        r = grade(a, b)
        print(f"  {label:35} → {r.kind}")
    except Exception as e:
        print(f"  {label:35} → EXCEPTION {type(e).__name__}: {e}")

print()
print("=== 5. grade_submission (the full real path) ===")
from tutor.learner_store import open_store
from tutor.schemas import Problem

store = open_store(ROOT / "data" / "learner_store" / "learners.db")
anchor = Problem(
    problem_id="2023_P1_Q1.1.1",
    skill_id="algebra.quadratic.solve",
    topic="Algebra",
    subtopic="Quadratic equations",
    structure_type="routine_calculation",
    prompt="Solve for x:  x^2 + x - 12 = 0",
    expected_answer=expected,
)
result = drill.grade_submission(
    store=store,
    learner_id="tanaka",
    session_id="diagnostic-session",
    anchor=anchor,
    learner_lines=learner_lines,
    sbc_message="",
)
print(json.dumps(result, indent=2, ensure_ascii=False))

=== 1. RAW INPUT (what browser sends) ===
  line 1: '(x+4)(x-3) = 0'
  line 2: 'x = -4 or x = 3'

=== 2. AFTER _latex_to_plain ===
  '(x+4)(x-3) = 0'
    → '(x+4)(x-3) = 0'
  'x = -4 or x = 3'
    → 'x = -4 or x = 3'

=== 3. AFTER _normalise ===
  learner last line: 'x=-4|x=3'
  expected:          'x=-4|x=3'
  equal?             True

=== 4. grade() calls, in order of what should be tried ===
  full join vs expected               → MatchKind.UNCERTAIN
  last line vs expected               → MatchKind.MATCH
  last line plain vs expected         → MatchKind.MATCH
  last line norm vs expected          → MatchKind.MATCH

=== 5. grade_submission (the full real path) ===
{
  "match": "UNCERTAIN",
  "matcher": "none",
  "raw_lines": [
    "(x+4)(x-3) = 0",
    "x = -4 or x = 3"
  ],
  "step_feedback": null,
  "tanaka_message": "I want to be sure this is the shape the memo expects. Check each step against the method.",
  "mastery": {
    "state": "near_mastery",
    "passed": 1
  }
}


In [27]:
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
TARGET = ROOT / "src" / "tutor" / "drill.py"

src = TARGET.read_text(encoding="utf-8")

# Replace the join-and-grade with last-line-and-grade
old = '''    plain_lines = [_latex_to_plain(x) for x in expanded]
    learner_text = " ".join(plain_lines).strip()
    try:
        result = grade(learner_text, anchor.expected_answer)
        # If uncertain, retry with normalised whitespace
        if result.kind != MatchKind.MATCH:
            n_learner = _normalise(learner_text)
            n_expect  = _normalise(anchor.expected_answer)
            # direct string match on the normalised forms
            if n_learner and n_expect and n_learner == n_expect:
                from .math_grader import MatchResult
                try:
                    result = MatchResult(kind=MatchKind.MATCH,
                                         matcher="normalised")
                except Exception:
                    # MatchResult constructor differs — fall back
                    result2 = grade(n_learner, n_expect)
                    if result2.kind == MatchKind.MATCH:
                        result = result2
            else:
                result2 = grade(n_learner, n_expect)
                if result2.kind == MatchKind.MATCH:
                    result = result2
    except Exception as e:'''

new = '''    plain_lines = [_latex_to_plain(x) for x in expanded]
    # The answer is the LAST line of the working — the rest is method.
    final_answer = plain_lines[-1].strip() if plain_lines else ""
    learner_text = final_answer  # what we hand to the matcher
    try:
        result = grade(final_answer, anchor.expected_answer)
        # If uncertain, retry against normalised forms
        if result.kind != MatchKind.MATCH:
            n_learner = _normalise(final_answer)
            n_expect  = _normalise(anchor.expected_answer)
            if n_learner and n_expect and n_learner == n_expect:
                try:
                    from .math_grader import MatchResult
                    result = MatchResult(kind=MatchKind.MATCH,
                                         matcher="normalised")
                except Exception:
                    result2 = grade(n_learner, n_expect)
                    if result2.kind == MatchKind.MATCH:
                        result = result2
            else:
                result2 = grade(n_learner, n_expect)
                if result2.kind == MatchKind.MATCH:
                    result = result2
    except Exception as e:'''

if "final_answer = plain_lines[-1]" in src:
    print("· already patched")
elif old in src:
    src = src.replace(old, new, 1)
    TARGET.write_text(src, encoding="utf-8")
    print("✓ grade_submission now grades the LAST line only")
    print(f"  {TARGET.name} written ({len(src):,} chars)")
else:
    print("✗ could not find the anchor block")
    # dump the relevant section so we can see what's there
    idx = src.find("plain_lines = ")
    if idx > 0:
        print(src[idx-100:idx+600])

✗ could not find the anchor block
      context={"source": "keyboard"},
            )
        except Exception:
            pass

    plain_lines = [_latex_to_plain(x) for x in expanded]
    learner_text = " ".join(plain_lines).strip()
    try:
        result = grade(learner_text, anchor.expected_answer)
    except Exception as e:
        return {"match": "error", "message": f"grader failed: {e}",
                "step_feedback": None, "raw_lines": expanded,
                "tanaka_message": "", "mastery": None}

    outcome = {
        "match": result.kind.name if hasattr(result.kind, "name") else str(result.kind),
        "matcher": getattr(result, "matcher", ""),
        "raw_lines": expanded,
        "step_feedback": Non


In [28]:
import sys, importlib, json, re
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
sys.path.insert(0, str(ROOT / "src"))

# ── 1. What's on disk? ──
print("=== 1. disk state of drill.py ===")
src = (ROOT / "src" / "tutor" / "drill.py").read_text(encoding="utf-8")
print("  has _latex_to_plain:      ", "_latex_to_plain" in src)
print("  has _normalise:           ", "def _normalise" in src)
print("  has 'final_answer =':     ", "final_answer = plain_lines[-1]" in src)
print("  has 'last line only':     ", "last line only" in src.lower())

# show the exact lines around the grade call
idx = src.find("plain_lines = ")
if idx > 0:
    snippet = src[idx:idx+700]
    print()
    print("  --- current grade_submission grading block ---")
    for line in snippet.split("\n")[:20]:
        print("  " + line)

# ── 2. Fresh import ──
print()
print("=== 2. fresh import ===")
import tutor
if hasattr(tutor, "drill"):
    delattr(tutor, "drill")
for m in list(sys.modules):
    if m.startswith("tutor.drill"):
        del sys.modules[m]
importlib.invalidate_caches()
from tutor import drill
print("  loaded from:", drill.__file__)
print("  has _latex_to_plain:  ", hasattr(drill, "_latex_to_plain"))
print("  has _normalise:       ", hasattr(drill, "_normalise"))

# ── 3. Run grade_submission in-process ──
print()
print("=== 3. grade_submission in-process (no server) ===")
from tutor.learner_store import open_store
from tutor.schemas import Problem

store = open_store(ROOT / "data" / "learner_store" / "learners.db")
anchor = Problem(
    problem_id="2023_P1_Q1.1.1",
    skill_id="algebra.quadratic.solve",
    topic="Algebra",
    subtopic="Quadratic equations",
    structure_type="routine_calculation",
    prompt="Solve for x:  x^2 + x - 12 = 0",
    expected_answer="x = -4 or x = 3",
)

# exactly what the browser sent (from your earlier diagnostic)
learner_lines = [
    "(x+4)(x-3) = 0",
    "x = -4 or x = 3",
]

result = drill.grade_submission(
    store=store,
    learner_id="tanaka",
    session_id="diag-" + str(id(anchor)),
    anchor=anchor,
    learner_lines=learner_lines,
    sbc_message="",
)
print(json.dumps(result, indent=2, ensure_ascii=False))

# ── 4. Also call grade() directly the way grade_submission should ──
print()
print("=== 4. direct grade() call with what SHOULD be graded ===")
from tutor.math_grader import grade
plain_lines = [drill._latex_to_plain(x) for x in learner_lines]
final = plain_lines[-1].strip()
print(f"  final line plain: {final!r}")
print(f"  expected:         {anchor.expected_answer!r}")
r = grade(final, anchor.expected_answer)
print(f"  grade() says:     {r.kind}")

=== 1. disk state of drill.py ===
  has _latex_to_plain:       True
  has _normalise:            True
  has 'final_answer =':      False
  has 'last line only':      False

  --- current grade_submission grading block ---
  plain_lines = [_latex_to_plain(x) for x in expanded]
      learner_text = " ".join(plain_lines).strip()
      try:
          result = grade(learner_text, anchor.expected_answer)
      except Exception as e:
          return {"match": "error", "message": f"grader failed: {e}",
                  "step_feedback": None, "raw_lines": expanded,
                  "tanaka_message": "", "mastery": None}
  
      outcome = {
          "match": result.kind.name if hasattr(result.kind, "name") else str(result.kind),
          "matcher": getattr(result, "matcher", ""),
          "raw_lines": expanded,
          "step_feedback": None,
          "tanaka_message": "",
          "mastery": None,
      }
  
      if result.kind == MatchKind.MAT

=== 2. fresh import ===
  loaded from:

{
  "match": "UNCERTAIN",
  "matcher": "none",
  "raw_lines": [
    "(x+4)(x-3) = 0",
    "x = -4 or x = 3"
  ],
  "step_feedback": null,
  "tanaka_message": "I want to be sure this is the shape the memo expects. Check each step against the method.",
  "mastery": {
    "state": "near_mastery",
    "passed": 1
  }
}

=== 4. direct grade() call with what SHOULD be graded ===
  final line plain: 'x = -4 or x = 3'
  expected:         'x = -4 or x = 3'
  grade() says:     MatchKind.MATCH


In [29]:
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
TARGET = ROOT / "src" / "tutor" / "drill.py"

src = TARGET.read_text(encoding="utf-8")

OLD = (
    '    plain_lines = [_latex_to_plain(x) for x in expanded]\n'
    '    learner_text = " ".join(plain_lines).strip()\n'
)

NEW = (
    '    plain_lines = [_latex_to_plain(x) for x in expanded]\n'
    '    # The answer is the LAST line of the working. Earlier lines are method.\n'
    '    learner_text = plain_lines[-1].strip() if plain_lines else ""\n'
)

if OLD in src:
    src = src.replace(OLD, NEW, 1)
    TARGET.write_text(src, encoding="utf-8")
    print("✓ patched: grade_submission now grades the last line only")
    print(f"  {TARGET.name} written ({len(src):,} chars)")
elif NEW in src:
    print("· already patched")
else:
    print("✗ anchor not found — dumping what's there")
    idx = src.find("plain_lines = ")
    print(src[idx-100:idx+200] if idx > 0 else "plain_lines not found")

# ── verify in-process, no server, no browser ──
import sys, importlib, json
sys.path.insert(0, str(ROOT / "src"))

import tutor
if hasattr(tutor, "drill"): delattr(tutor, "drill")
for m in list(sys.modules):
    if m.startswith("tutor.drill"): del sys.modules[m]
importlib.invalidate_caches()

from tutor import drill
from tutor.learner_store import open_store
from tutor.schemas import Problem

store = open_store(ROOT / "data" / "learner_store" / "learners.db")
anchor = Problem(
    problem_id="2023_P1_Q1.1.1",
    skill_id="algebra.quadratic.solve",
    topic="Algebra",
    subtopic="Quadratic equations",
    structure_type="routine_calculation",
    prompt="Solve for x:  x^2 + x - 12 = 0",
    expected_answer="x = -4 or x = 3",
)

result = drill.grade_submission(
    store=store, learner_id="tanaka", session_id="verify-1",
    anchor=anchor,
    learner_lines=["(x+4)(x-3) = 0", "x = -4 or x = 3"],
    sbc_message="",
)
print()
print("in-process grade_submission result:")
print("  match:", result["match"])
print("  tanaka_message:", result["tanaka_message"])

✓ patched: grade_submission now grades the last line only
  drill.py written (36,728 chars)



in-process grade_submission result:
  match: MATCH
  tanaka_message: Nice — the shape is right.


In [30]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\src")))

In [31]:
from tutor import drill_server
drill_server.serve_in_thread()
import time, urllib.request
time.sleep(0.6)
print(urllib.request.urlopen("http://127.0.0.1:8765/health", timeout=3).read().decode())

drill_server at http://127.0.0.1:8765drill_server running in background thread on http://127.0.0.1:8765

  keyboard: http://127.0.0.1:8765/
  anchor:   2023_P1_Q1.1.1  |  learner: tanaka
{"ok": true, "anchor": "2023_P1_Q1.1.1"}


In [32]:
from pathlib import Path
src = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\src\tutor\drill.py").read_text(encoding="utf-8")
print("last-line patch present:", "learner_text = plain_lines[-1].strip()" in src)

last-line patch present: True


In [33]:
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
HTML = ROOT / "docs" / "keyboard_preview.html"

html = HTML.read_text(encoding="utf-8")

OLD = '''    if (res.step_feedback) {
      var sf = res.step_feedback;
      out.push("<div>Memo: " + sf.awarded + "/" + sf.total + "</div>");
      sf.steps.forEach(function (s) {
        if (s.match_status === "NOT_MATCHED") {
          out.push("<div>· skipped: " + (s.desc || "") +
                   "  (" + s.marks + ")  →  " + (s.first_form || "") + "</div>");
        }
      });
    }'''

NEW = '''    if (res.step_feedback) {
      var sf = res.step_feedback;
      out.push("<div><strong>Memo: " + sf.awarded + "/" + sf.total +
               "</strong>  (this is what a marker would award)</div>");
      sf.steps.forEach(function (s) {
        if (s.match_status === "NOT_MATCHED") {
          var line = "<div>· missing: <em>" + (s.desc || "a memo step") +
                     "</em>  (" + s.marks +
                     (s.marks === 1 ? " mark" : " marks") + ")";
          if (s.first_form) {
            line += "  →  memo shows: " + s.first_form;
          }
          line += "</div>";
          out.push(line);
        }
      });
    }'''

if OLD in html:
    html = html.replace(OLD, NEW, 1)
    HTML.write_text(html, encoding="utf-8")
    print("✓ step feedback JS patched")
    print(f"  {HTML.name} written ({len(html):,} chars)")
elif "missing: <em>" in html:
    print("· already patched")
else:
    print("✗ block not found — dumping what's there")
    idx = html.find("if (res.step_feedback)")
    if idx > 0:
        print(html[idx:idx+400])

✗ block not found — dumping what's there


In [34]:
from pathlib import Path
import re

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
DRILL = ROOT / "src" / "tutor" / "drill.py"
HTML  = ROOT / "docs" / "keyboard_preview.html"

# ══════════════════════════════════════════════════════════════
# 1) Python: add sibling lookup to grade_submission
# ══════════════════════════════════════════════════════════════
src = DRILL.read_text(encoding="utf-8")

# Add a helper right after _show_sibling so we can call it from grade_submission
helper = '''

def _pick_sibling_for_anchor(anchor, exclude_keys=None):
    """Return one worked variant for this anchor's template, or None.

    Used by the web path: when the learner gets it wrong, they see a
    fully worked sibling problem from the same template — problem,
    steps, and note — exactly as the CLI does.
    """
    try:
        template_id = route_template(anchor.skill_id, anchor.prompt)
        if not template_id:
            return None
        pool = _load_pool(template_id)
        if not pool:
            return None
        exclude = exclude_keys or set()
        candidates = [a for a in pool
                      if analogue_library.analogue_key(a) not in exclude]
        if not candidates:
            candidates = pool
        import random as _r
        ex = _r.choice(candidates)
        return {
            "problem": ex.get("problem", ""),
            "steps":   list(ex.get("steps", [])),
            "note":    ex.get("note", ""),
            "key":     analogue_library.analogue_key(ex),
        }
    except Exception:
        return None
'''

if "_pick_sibling_for_anchor" in src:
    print("· sibling helper already present")
else:
    # insert right after the _show_sibling function body
    marker = "# ============================================================\n# Warm diagnosis message"
    if marker in src:
        src = src.replace(marker, helper + "\n\n" + marker, 1)
        print("✓ sibling helper inserted")
    else:
        print("✗ could not find insertion marker")

# Now patch grade_submission to include the sibling on NO_MATCH and UNCERTAIN
old_uncertain = '''    elif result.kind == MatchKind.UNCERTAIN:
        outcome["tanaka_message"] = (
            "I want to be sure this is the shape the memo expects. "
            "Check each step against the method."
        )'''

new_uncertain = '''    elif result.kind == MatchKind.UNCERTAIN:
        outcome["tanaka_message"] = (
            "I want to be sure this is the shape the memo expects. "
            "Here is a similar one worked out — compare it line by line."
        )
        sib = _pick_sibling_for_anchor(anchor)
        if sib:
            outcome["sibling"] = sib'''

if "outcome[\"sibling\"] = sib" in src and "UNCERTAIN" in src:
    print("· UNCERTAIN sibling already patched")
elif old_uncertain in src:
    src = src.replace(old_uncertain, new_uncertain, 1)
    print("✓ UNCERTAIN now returns a sibling")
else:
    print("✗ could not find UNCERTAIN block")

# NO_MATCH branch — append sibling after tanaka_message assembly
old_nomatch_tail = '''        if misconception_id:
            outcome["tanaka_message"] += f"  Noticed: {misconception_id}"'''

new_nomatch_tail = '''        if misconception_id:
            outcome["tanaka_message"] += f"  Noticed: {misconception_id}"
        outcome["tanaka_message"] += "  Here is a similar one worked out — compare it line by line."
        sib = _pick_sibling_for_anchor(anchor)
        if sib:
            outcome["sibling"] = sib'''

if "Here is a similar one worked out" in src and src.count("_pick_sibling_for_anchor") > 1:
    print("· NO_MATCH sibling already patched")
elif old_nomatch_tail in src:
    src = src.replace(old_nomatch_tail, new_nomatch_tail, 1)
    print("✓ NO_MATCH now returns a sibling")
else:
    print("✗ could not find NO_MATCH tail")

DRILL.write_text(src, encoding="utf-8")
print(f"  drill.py written ({len(src):,} chars)")

# ══════════════════════════════════════════════════════════════
# 2) HTML: render the sibling in the result strip
# ══════════════════════════════════════════════════════════════
html = HTML.read_text(encoding="utf-8")

old_js = '''    if (res.step_feedback) {
      var sf = res.step_feedback;
      out.push("<div><strong>Memo: " + sf.awarded + "/" + sf.total +
               "</strong>  (this is what a marker would award)</div>");
      sf.steps.forEach(function (s) {
        if (s.match_status === "NOT_MATCHED") {
          var line = "<div>· missing: <em>" + (s.desc || "a memo step") +
                     "</em>  (" + s.marks +
                     (s.marks === 1 ? " mark" : " marks") + ")";
          if (s.first_form) {
            line += "  →  memo shows: " + s.first_form;
          }
          line += "</div>";
          out.push(line);
        }
      });
    }'''

new_js = '''    if (res.step_feedback) {
      var sf = res.step_feedback;
      out.push("<div><strong>Memo: " + sf.awarded + "/" + sf.total +
               "</strong>  (this is what a marker would award)</div>");
      sf.steps.forEach(function (s) {
        if (s.match_status === "NOT_MATCHED") {
          var line = "<div>· missing: <em>" + (s.desc || "a memo step") +
                     "</em>  (" + s.marks +
                     (s.marks === 1 ? " mark" : " marks") + ")";
          if (s.first_form) {
            line += "  →  memo shows: " + s.first_form;
          }
          line += "</div>";
          out.push(line);
        }
      });
    }
    if (res.sibling) {
      out.push("<div style='margin-top:8px;padding-top:8px;border-top:1px solid #2a3f66'>" +
               "<strong>Similar one, worked out:</strong></div>");
      out.push("<div style='margin-top:4px'>" + (res.sibling.problem || "") + "</div>");
      (res.sibling.steps || []).forEach(function (s) {
        out.push("<div style='margin-left:12px;color:#b8d0ff'>" + s + "</div>");
      });
      if (res.sibling.note) {
        out.push("<div style='margin-top:4px;color:#9ab'>Note: " +
                 res.sibling.note + "</div>");
      }
    }'''

if "Similar one, worked out:" in html:
    print("· sibling rendering already in HTML")
elif old_js in html:
    html = html.replace(old_js, new_js, 1)
    HTML.write_text(html, encoding="utf-8")
    print("✓ sibling rendering added to result strip")
    print(f"  {HTML.name} written ({len(html):,} chars)")
elif "if (res.step_feedback)" in html:
    # fall back: append sibling rendering right after step_feedback block
    print("· step feedback block differs — using regex append")
    pattern = re.compile(
        r"(if \(res\.step_feedback\) \{.*?\n    \})",
        re.DOTALL
    )
    m = pattern.search(html)
    if m:
        insertion = m.group(1) + "\n    " + '''if (res.sibling) {
      out.push("<div style='margin-top:8px;padding-top:8px;border-top:1px solid #2a3f66'><strong>Similar one, worked out:</strong></div>");
      out.push("<div style='margin-top:4px'>" + (res.sibling.problem || "") + "</div>");
      (res.sibling.steps || []).forEach(function (s) {
        out.push("<div style='margin-left:12px;color:#b8d0ff'>" + s + "</div>");
      });
      if (res.sibling.note) {
        out.push("<div style='margin-top:4px;color:#9ab'>Note: " + res.sibling.note + "</div>");
      }
    }'''
        html = html[:m.start()] + insertion + html[m.end():]
        HTML.write_text(html, encoding="utf-8")
        print("✓ sibling rendering added via regex")
        print(f"  {HTML.name} written ({len(html):,} chars)")
    else:
        print("✗ could not find step_feedback block")
else:
    print("✗ neither anchor found for JS patch")

✓ sibling helper inserted
✓ UNCERTAIN now returns a sibling
· NO_MATCH sibling already patched
  drill.py written (37,927 chars)
✗ neither anchor found for JS patch


In [35]:
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
src = (ROOT / "src" / "tutor" / "drill.py").read_text(encoding="utf-8")

# Confirm the patch is really on disk
print("disk has new UNCERTAIN text:",
      "Here is a similar one worked out" in src)
print("disk has sibling helper:      ",
      "_pick_sibling_for_anchor" in src)
print("disk has UNCERTAIN sibling:   ",
      'outcome["sibling"] = sib' in src)

# Show the actual UNCERTAIN block so we know what's on disk
import re
m = re.search(r"elif result\.kind == MatchKind\.UNCERTAIN:.*?(?=\n    else:|\n    else\b)",
              src, re.DOTALL)
if m:
    print()
    print("--- UNCERTAIN block on disk ---")
    print(m.group(0))
else:
    print()
    print("Could not find UNCERTAIN block — dumping region")
    idx = src.find("UNCERTAIN")
    print(src[idx-200:idx+800])

disk has new UNCERTAIN text: True
disk has sibling helper:       True
disk has UNCERTAIN sibling:    True

--- UNCERTAIN block on disk ---
elif result.kind == MatchKind.UNCERTAIN:
        outcome["tanaka_message"] = (
            "I want to be sure this is the shape the memo expects. "
            "Here is a similar one worked out — compare it line by line."
        )
        sib = _pick_sibling_for_anchor(anchor)
        if sib:
            outcome["sibling"] = sib



In [36]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\src")))

from tutor import drill
print("loaded from:", drill.__file__)
# Check the loaded module has the new text (not just the disk)
src = open(drill.__file__, "r", encoding="utf-8").read()
print("new UNCERTAIN text in loaded module:", "Here is a similar one worked out" in src)

loaded from: C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\src\tutor\drill.py
new UNCERTAIN text in loaded module: True


In [37]:
from tutor import drill_server
drill_server.serve_in_thread()
import time, urllib.request
time.sleep(0.6)
print(urllib.request.urlopen("http://127.0.0.1:8765/health", timeout=3).read().decode())

drill_server at http://127.0.0.1:8765drill_server running in background thread on http://127.0.0.1:8765

  keyboard: http://127.0.0.1:8765/
  anchor:   2023_P1_Q1.1.1  |  learner: tanaka
{"ok": true, "anchor": "2023_P1_Q1.1.1"}


In [38]:
import re
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
DRILL = ROOT / "src" / "tutor" / "drill.py"
HTML  = ROOT / "docs" / "keyboard_preview.html"

# ══════════════════════════════════════════════════════════════
# A. drill.py — strict sign detection + marker-style verdict payload
# ══════════════════════════════════════════════════════════════
src = DRILL.read_text(encoding="utf-8")
orig_len = len(src)

HELPERS = r'''def _sign_mismatch_only(learner, expected):
    """True if learner and expected differ ONLY in the signs of some numbers."""
    import re as _rs
    def nums(s):
        return _rs.findall(r"-?\d+(?:\.\d+)?", s)
    l = nums(learner); e = nums(expected)
    if len(l) != len(e) or not l:
        return False
    diffs = 0
    for a, b in zip(l, e):
        if a == b:
            continue
        if a.lstrip("-") == b.lstrip("-"):
            diffs += 1
        else:
            return False
    return diffs >= 1


def _make_sign_flip_diagnosis(learner, expected):
    return (
        f"You wrote: {learner}.  The memo has: {expected}.  "
        "The numbers match — one sign is wrong. "
        "When a term crosses the equals sign, its sign flips."
    )


def _make_diagnosis_sentence(learner, expected, kind, misid):
    if kind == "sign_flip":
        return _make_sign_flip_diagnosis(learner, expected)
    if kind == "partial":
        return "One root is right. The other is missing or wrong."
    base = f"Your answer ({learner}) does not match the memo ({expected})."
    if misid:
        base += f"  Noticed: {misid}."
    return base


'''

if "_sign_mismatch_only" in src:
    print("· helpers already on disk")
else:
    idx = src.find("def grade_submission(")
    if idx < 0:
        raise SystemExit("grade_submission not found")
    line_start = src.rfind("\n", 0, idx) + 1
    src = src[:line_start] + HELPERS + src[line_start:]
    print("✓ helpers inserted")

NEW_GRADE = r'''def grade_submission(store, learner_id, session_id, anchor,
                     learner_lines, sbc_message=""):
    """Grade one keyboard submission. Return a marker-style verdict."""
    from . import slash_commands as _sc

    expanded = [_sc.expand(line) if line else "" for line in learner_lines]

    if sbc_message.strip():
        try:
            store.interventions.log(
                learner_id=learner_id, skill_id=anchor.skill_id,
                action_type="sbc_message",
                action_desc=sbc_message.strip()[:400],
                question_id=anchor.problem_id, session_id=session_id,
                context={"source": "keyboard"},
            )
        except Exception:
            pass

    plain_lines = [_latex_to_plain(x) for x in expanded]
    final_answer = plain_lines[-1].strip() if plain_lines else ""

    # strict: sign-only mismatch is a real error, not ambiguous form
    sign_only = _sign_mismatch_only(final_answer, anchor.expected_answer)

    try:
        result = grade(final_answer, anchor.expected_answer)
    except Exception as e:
        return {"match": "error", "message": f"grader failed: {e}",
                "raw_lines": expanded, "your_working": plain_lines,
                "mark": {"awarded": 0, "total": 0}, "memo_steps": [],
                "diagnosis": f"Grader error: {e}",
                "tanaka_message": "", "mastery": None, "sibling": None}

    outcome = {
        "match": None,
        "raw_lines": expanded,
        "your_working": plain_lines,
        "step_feedback": None,
        "memo_steps": [],
        "mark": {"awarded": 0, "total": 0},
        "diagnosis": "",
        "tanaka_message": "",
        "mastery": None,
        "sibling": None,
    }

    # step grader runs for MATCH and NO_MATCH both
    step_result = _step_feedback(anchor.problem_id, plain_lines)
    if step_result is None and getattr(anchor, "step_scheme", None):
        try:
            step_result = grade_steps_inline(anchor.step_scheme, plain_lines)
        except Exception:
            step_result = None

    if step_result is not None:
        outcome["mark"] = {
            "awarded": step_result["awarded_marks"],
            "total":   step_result["total_marks"],
        }
        outcome["memo_steps"] = [
            {"index": s.get("index"), "desc": s.get("desc"),
             "marks": s.get("marks"), "status": s.get("match_status"),
             "first_form": (s.get("forms") or [None])[0]}
            for s in step_result.get("steps", [])
        ]

    if sign_only:
        outcome["match"] = "NO_MATCH"
        _save_attempt(store, session_id, learner_id, anchor.skill_id,
                      final_answer, "D_sign_flip", "sign_flip")
        outcome["diagnosis"] = _make_sign_flip_diagnosis(final_answer,
                                                         anchor.expected_answer)
        try:
            store.discoveries.record(
                learner_id=learner_id, skill_id=anchor.skill_id,
                question_id=anchor.problem_id, learner_lines=plain_lines,
                expected_answer=anchor.expected_answer, session_id=session_id,
            )
        except Exception:
            pass
        sib = _pick_sibling_for_anchor(anchor)
        if sib: outcome["sibling"] = sib

    elif result.kind == MatchKind.MATCH:
        outcome["match"] = "MATCH"
        _save_attempt(store, session_id, learner_id, anchor.skill_id,
                      final_answer, "D_match", "none")

        if step_result is not None:
            try:
                for m in step_result.get("steps", []):
                    if m.get("match_status") == "NOT_MATCHED":
                        store.interventions.log(
                            learner_id=learner_id, skill_id=anchor.skill_id,
                            action_type="memo_line_shown",
                            action_desc="Showed memo version of missed step: " + m["desc"],
                            question_id=anchor.problem_id, session_id=session_id,
                            context={"step_index": m["index"], "step_desc": m["desc"]},
                        )
            except Exception:
                pass
            _close_open_interventions(store, learner_id, anchor.skill_id,
                                      step_result, anchor.problem_id, session_id)

        if step_result is None:
            outcome["diagnosis"] = "Answer matches the memo."
        elif step_result["awarded_marks"] == step_result["total_marks"]:
            outcome["diagnosis"] = "Full method. All memo steps shown."
        else:
            missing = [s["desc"] for s in step_result.get("steps", [])
                       if s.get("match_status") == "NOT_MATCHED"]
            outcome["diagnosis"] = (
                "Answer is right, but the method is incomplete. Missing: "
                + "; ".join(missing) + "."
            )

    elif result.kind == MatchKind.UNCERTAIN:
        outcome["match"] = "UNCERTAIN"
        outcome["diagnosis"] = (
            "Your working does not clearly match the memo's expected form. "
            "Check the notation."
        )
        sib = _pick_sibling_for_anchor(anchor)
        if sib: outcome["sibling"] = sib

    else:
        outcome["match"] = "NO_MATCH"
        try:
            kind = diagnose_wrong(final_answer, anchor.expected_answer) or "unknown"
        except Exception:
            kind = "unknown"
        misid = None
        try:
            diag = diagnose(anchor.skill_id, final_answer, anchor.expected_answer)
            if diag and diag.misconception_id:
                misid = diag.misconception_id
        except Exception:
            pass
        _save_attempt(store, session_id, learner_id, anchor.skill_id,
                      final_answer, "D_NO_MATCH", kind, misid)
        if kind == "unknown":
            try:
                store.discoveries.record(
                    learner_id=learner_id, skill_id=anchor.skill_id,
                    question_id=anchor.problem_id, learner_lines=plain_lines,
                    expected_answer=anchor.expected_answer, session_id=session_id,
                )
            except Exception:
                pass
        outcome["diagnosis"] = _make_diagnosis_sentence(
            final_answer, anchor.expected_answer, kind, misid)
        sib = _pick_sibling_for_anchor(anchor)
        if sib: outcome["sibling"] = sib

    _try_recompute_mastery(store, learner_id, anchor.skill_id)
    try:
        m = store.get_mastery(learner_id, anchor.skill_id)
        outcome["mastery"] = {"state": m.get("mastery_state"),
                              "passed": m.get("sittings_passed", 0)}
    except Exception:
        outcome["mastery"] = None

    return outcome
'''

m_start = re.search(r"^def grade_submission\(", src, re.MULTILINE)
if not m_start:
    raise SystemExit("grade_submission not found (2nd pass)")
start = m_start.start()
m_next = re.search(r"^def \w+\(", src[start+10:], re.MULTILINE)
end = start + 10 + m_next.start() if m_next else len(src)
src = src[:start] + NEW_GRADE + "\n\n" + src[end:]

DRILL.write_text(src, encoding="utf-8")
print(f"✓ grade_submission replaced  ({orig_len} → {len(src)} chars)")

# ══════════════════════════════════════════════════════════════
# B. keyboard_preview.html — the four-section strip render
# ══════════════════════════════════════════════════════════════
html = HTML.read_text(encoding="utf-8")
orig_html_len = len(html)

NEW_JS = r'''.then(function (res) {
    var out = [];

    // 1. Mark — the headline
    if (res.mark && res.mark.total > 0) {
      var ok = res.mark.awarded === res.mark.total;
      out.push("<div class='mark " + (ok ? "mark-full" : "mark-partial") + "'>" +
               res.mark.awarded + "/" + res.mark.total + "</div>");
    } else if (res.match === "MATCH") {
      out.push("<div class='mark mark-full'>ok</div>");
    } else if (res.match === "NO_MATCH") {
      out.push("<div class='mark mark-zero'>0</div>");
    }

    // 2. Your working
    if (res.your_working && res.your_working.length) {
      out.push("<div class='strip-head'>Your working</div>");
      res.your_working.forEach(function (l, i) {
        out.push("<div class='strip-line'>" + (i + 1) + ". " + l + "</div>");
      });
    }

    // 3. Memo expected — per step, with mark status
    if (res.memo_steps && res.memo_steps.length) {
      out.push("<div class='strip-head'>Memo expected</div>");
      res.memo_steps.forEach(function (s) {
        var sym = s.status === "MATCHED" ? "ok"
                : s.status === "NOT_MATCHED" ? "x"
                : "?";
        var cls = s.status === "MATCHED" ? "s-ok"
                : s.status === "NOT_MATCHED" ? "s-no"
                : "s-maybe";
        var line = "<div class='strip-line'>" +
                   "<span class='" + cls + "'>" + sym + "</span> " +
                   (s.desc || "");
        if (s.marks != null) {
          line += " <span class='marks'>(" + s.marks +
                  (s.marks === 1 ? " mark" : " marks") + ")</span>";
        }
        line += "</div>";
        if (s.first_form) {
          line += "<div class='strip-example'>" + s.first_form + "</div>";
        }
        out.push(line);
      });
    }

    // 4. Diagnosis — one concrete sentence
    if (res.diagnosis) {
      out.push("<div class='strip-diag'>" + res.diagnosis + "</div>");
    }

    // 5. Sibling — only after diagnosis
    if (res.sibling) {
      out.push("<div class='strip-head sibling-head'>Similar one, worked out</div>");
      out.push("<div class='strip-line'>" + (res.sibling.problem || "") + "</div>");
      (res.sibling.steps || []).forEach(function (s) {
        out.push("<div class='strip-line sibling-step'>" + s + "</div>");
      });
      if (res.sibling.note) {
        out.push("<div class='strip-note'>" + res.sibling.note + "</div>");
      }
    }

    resultEl.innerHTML = out.join("");
  })'''

m1 = html.find(".then(function (res) {")
m2 = html.find('resultEl.innerHTML = out.join("");', m1)
if m1 > 0 and m2 > m1:
    tail_start = m2 + len('resultEl.innerHTML = out.join("");')
    tail_match = re.search(r"\n\s*\}\)\s*\n", html[tail_start:])
    end_of = tail_start + tail_match.end() if tail_match else tail_start
    html = html[:m1] + NEW_JS + "\n  " + html[end_of:]
    print("✓ submit render replaced")
else:
    print(f"✗ render anchors not found (m1={m1}, m2={m2})")

CSS_NEW = """
.mark { font-size: 26px; font-weight: 800; letter-spacing: -0.02em;
        padding: 8px 0 10px; }
.mark-full    { color: #6f6; }
.mark-partial { color: #fbb; }
.mark-zero    { color: #f88; }
.strip-head   { font-size: 10px; font-weight: 700; text-transform: uppercase;
                letter-spacing: .06em; color: #9ab; margin: 10px 0 4px; }
.strip-line   { font-size: 12.5px; padding: 2px 0; color: #dde; line-height: 1.55; }
.strip-example{ font-size: 12px; padding: 1px 0 4px 22px; color: #8fa8d6;
                font-family: 'Cambria Math', Georgia, serif; }
.strip-diag   { margin-top: 12px; padding: 9px 11px; background: rgba(255,220,120,.08);
                border-left: 3px solid #fc8; font-size: 12.5px; color: #ffe8b8;
                line-height: 1.55; }
.strip-note   { font-size: 11.5px; color: #9ab; margin-top: 6px;
                padding-left: 14px; font-style: italic; }
.s-ok, .s-no, .s-maybe { font-weight: 700; margin-right: 4px; }
.s-ok    { color: #6f6; }
.s-no    { color: #f88; }
.s-maybe { color: #fb8; }
.marks   { opacity: 0.6; font-size: 11px; }
.sibling-head { margin-top: 16px; padding-top: 12px;
                border-top: 1px solid #2a3f66; }
.sibling-step { color: #b8d0ff; padding-left: 16px; }
"""

if ".strip-diag" not in html:
    idx = html.rfind("</style>")
    if idx > 0:
        html = html[:idx] + CSS_NEW + "\n" + html[idx:]
        print("✓ strip CSS added")

HTML.write_text(html, encoding="utf-8")
print(f"  keyboard: {orig_html_len} → {len(html)} chars")

# ══════════════════════════════════════════════════════════════
# C. Verify in-process, no server, no browser
# ══════════════════════════════════════════════════════════════
import sys, importlib
sys.path.insert(0, str(ROOT / "src"))
import tutor
if hasattr(tutor, "drill"): delattr(tutor, "drill")
for m in list(sys.modules):
    if m.startswith("tutor.drill"): del sys.modules[m]
importlib.invalidate_caches()
from tutor import drill
from tutor.learner_store import open_store
from tutor.schemas import Problem

store = open_store(ROOT / "data" / "learner_store" / "learners.db")
anchor = Problem(
    problem_id="2023_P1_Q1.1.1",
    skill_id="algebra.quadratic.solve",
    topic="Algebra", subtopic="Quadratic equations",
    structure_type="routine_calculation",
    prompt="Solve for x:  x^2 + x - 12 = 0",
    expected_answer="x = -4 or x = 3",
)

print()
print("── test: wrong sign (x = 4 or x = 3) ──")
r = drill.grade_submission(store, "tanaka", "v-sign", anchor,
                           ["x = 4 or x = 3"], "")
print("  match:     ", r["match"])
print("  mark:      ", r["mark"])
print("  diagnosis: ", r["diagnosis"])
print("  sibling?   ", "yes" if r.get("sibling") else "no")

print()
print("── test: correct (x = -4 or x = 3) ──")
r = drill.grade_submission(store, "tanaka", "v-ok", anchor,
                           ["x = -4 or x = 3"], "")
print("  match:     ", r["match"])
print("  mark:      ", r["mark"])
print("  diagnosis: ", r["diagnosis"])

✓ helpers inserted
✓ grade_submission replaced  (37927 → 39776 chars)
✓ submit render replaced
  keyboard: 33219 → 33221 chars

── test: wrong sign (x = 4 or x = 3) ──
  match:      NO_MATCH
  mark:       {'awarded': 0, 'total': 3}
  diagnosis:  You wrote: x = 4 or x = 3.  The memo has: x = -4 or x = 3.  The numbers match — one sign is wrong. When a term crosses the equals sign, its sign flips.
  sibling?    yes

── test: correct (x = -4 or x = 3) ──
  match:      MATCH
  mark:       {'awarded': 1, 'total': 3}
  diagnosis:  Answer is right, but the method is incomplete. Missing: Factorise; Set each factor to zero.


In [39]:
import urllib.request, json

# 1) What is the running server actually returning?
req = urllib.request.Request(
    "http://127.0.0.1:8765/grade",
    data=json.dumps({
        "learner_lines_latex": ["x = 4 or x = 3"],
        "sbc_message": "",
        "tab_used": "Algebra",
    }).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)
try:
    with urllib.request.urlopen(req, timeout=5) as r:
        body = json.loads(r.read())
    print("server response keys:", sorted(body.keys()))
    print()
    print(json.dumps(body, indent=2)[:1200])
except Exception as e:
    print("server call failed:", e)

# 2) What does the served HTML look like right now?
try:
    with urllib.request.urlopen("http://127.0.0.1:8765/", timeout=5) as r:
        html = r.read().decode("utf-8")
    print()
    print("served HTML has new classes:")
    for cls in ["strip-diag", "mark-full", "memo_steps", "res.mark"]:
        print(f"  {cls:<16} {'yes' if cls in html else 'NO'}")
except Exception as e:
    print("HTML fetch failed:", e)

server response keys: ['mastery', 'match', 'matcher', 'raw_lines', 'step_feedback', 'tanaka_message']

{
  "match": "UNCERTAIN",
  "matcher": "none",
  "raw_lines": [
    "x = 4 or x = 3"
  ],
  "step_feedback": null,
  "tanaka_message": "I want to be sure this is the shape the memo expects. Check each step against the method.",
  "mastery": null
}

served HTML has new classes:
  strip-diag       yes
  mark-full        yes
  memo_steps       yes
  res.mark         yes


In [40]:
import sys
from pathlib import Path
ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
sys.path.insert(0, str(ROOT / "src"))

# Load a FRESH copy of everything
import importlib, tutor
for m in list(sys.modules):
    if m.startswith("tutor."): 
        del sys.modules[m]
importlib.invalidate_caches()

from tutor import drill_server
print("drill_server file:  ", drill_server.__file__)
print("drill_server loads: OK")

# Start the server
SERVER_THREAD = drill_server.serve_in_thread()

import time, urllib.request, json
time.sleep(0.8)

# Verify the server has the new code
with urllib.request.urlopen("http://127.0.0.1:8765/health", timeout=3) as r:
    print("health:", r.read().decode())

req = urllib.request.Request(
    "http://127.0.0.1:8765/grade",
    data=json.dumps({"learner_lines_latex": ["x = 4 or x = 3"],
                     "sbc_message": "", "tab_used": "Algebra"}).encode(),
    headers={"Content-Type": "application/json"}, method="POST")
with urllib.request.urlopen(req, timeout=5) as r:
    body = json.loads(r.read())
print()
print("server returns these keys:", sorted(body.keys()))
print("has 'mark':       ", "mark" in body)
print("has 'diagnosis':  ", "diagnosis" in body)
print("has 'your_working':", "your_working" in body)

drill_server file:   C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\src\tutor\drill_server.py
drill_server loads: OK
drill_server at http://127.0.0.1:8765
  keyboard: http://127.0.0.1:8765/
  anchor:   2023_P1_Q1.1.1  |  learner: tanaka
drill_server running in background thread on http://127.0.0.1:8765
health: {"ok": true, "anchor": "2023_P1_Q1.1.1"}

server returns these keys: ['mastery', 'match', 'matcher', 'raw_lines', 'step_feedback', 'tanaka_message']
has 'mark':        False
has 'diagnosis':   False
has 'your_working': False


In [41]:
import sys
from pathlib import Path
ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
sys.path.insert(0, str(ROOT / "src"))

import importlib, tutor
for m in list(sys.modules):
    if m.startswith("tutor."):
        del sys.modules[m]
importlib.invalidate_caches()

from tutor import analogue_library
from tutor.template_router import route_template
from tutor.schemas import Problem

anchor = Problem(
    problem_id="2023_P1_Q1.1.1",
    skill_id="algebra.quadratic.solve",
    topic="Algebra", subtopic="Quadratic equations",
    structure_type="routine_calculation",
    prompt="Solve for x:  x^2 + x - 12 = 0",
    expected_answer="x = -4 or x = 3",
)

tid = route_template(anchor.skill_id, anchor.prompt)
print("template_id:", tid)
print()

pool = analogue_library.POOL.get(tid, [])
print("pool size:", len(pool))
print()

print("── first 3 analogues, raw steps ──")
for i, ex in enumerate(pool[:3]):
    print(f"=== analogue {i+1} ===")
    print("  problem:", ex.get("problem"))
    steps = ex.get("steps", [])
    print(f"  steps: {len(steps)}")
    for j, s in enumerate(steps):
        print(f"    {j+1}. {s}")
    print("  note:", ex.get("note"))
    print()

template_id: quadratic_standard_factorable

pool size: 50

── first 3 analogues, raw steps ──
=== analogue 1 ===
  problem: Solve x^2 - 4x - 32 = 0
  steps: 3
    1. (x - 8)(x + 4) = 0
    2. x - 8 = 0  or  x + 4 = 0
    3. x = 8 or x = -4
  note: b = -(sum of roots); c = (product of roots).

=== analogue 2 ===
  problem: Solve x^2 - 8x + 7 = 0
  steps: 3
    1. (x - 7)(x - 1) = 0
    2. x - 7 = 0  or  x - 1 = 0
    3. x = 7 or x = 1
  note: b = -(sum of roots); c = (product of roots).

=== analogue 3 ===
  problem: Solve x^2 + 5x - 36 = 0
  steps: 3
    1. (x - 4)(x + 9) = 0
    2. x - 4 = 0  or  x + 9 = 0
    3. x = 4 or x = -9
  note: b = -(sum of roots); c = (product of roots).



In [42]:
import sys, inspect
from pathlib import Path
ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
sys.path.insert(0, str(ROOT / "src"))

import importlib, tutor
for m in list(sys.modules):
    if m.startswith("tutor."):
        del sys.modules[m]
importlib.invalidate_caches()

# ── 1. Where are the generators? ──
try:
    from tutor import analogue_generators
    print("analogue_generators module:", analogue_generators.__file__)
    print()
    print("public functions:")
    for name, obj in inspect.getmembers(analogue_generators, inspect.isfunction):
        if name.startswith("_"):
            continue
        try:
            sig = inspect.signature(obj)
            print(f"  {name}{sig}")
        except Exception:
            print(f"  {name}")
except Exception as e:
    print("import failed:", e)

print()

# ── 2. Search the whole src tree for "quadratic_standard_factorable" ──
print("── files mentioning quadratic_standard_factorable ──")
for path in (ROOT / "src").rglob("*.py"):
    try:
        text = path.read_text(encoding="utf-8")
    except Exception:
        continue
    if "quadratic_standard_factorable" in text or "Factorable" in text:
        rel = path.relative_to(ROOT)
        lines = [i+1 for i, l in enumerate(text.split("\n"))
                 if "quadratic_standard_factorable" in l]
        print(f"  {rel}  lines: {lines[:10]}")

print()

# ── 3. Show the source of any function that mentions "Set each factor" ──
print("── functions that mention 'Set each factor' ──")
for path in (ROOT / "src").rglob("*.py"):
    try:
        text = path.read_text(encoding="utf-8")
    except Exception:
        continue
    if "Set each factor" in text:
        rel = path.relative_to(ROOT)
        print(f"  {rel}")
        # show any lines around it
        for i, line in enumerate(text.split("\n")):
            if "Set each factor" in line:
                print(f"    line {i+1}: {line.strip()[:120]}")

analogue_generators module: C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\src\tutor\analogue_generators.py

public functions:
  gen_analytical_area(count=50, seed=68)
  gen_analytical_circle_equation(count=50, seed=65)
  gen_analytical_circle_radius(count=50, seed=66)
  gen_analytical_distance(count=200, seed=61)
  gen_analytical_gradient(count=200, seed=62)
  gen_analytical_inclination(count=50, seed=69)
  gen_analytical_line_equation(count=200, seed=64)
  gen_analytical_midpoint(count=50, seed=63)
  gen_analytical_tangent_equation(count=50, seed=67)
  gen_analytical_translation_v2(count=50, seed=206)
  gen_ap_sum(count=200, seed=22)
  gen_ap_term(count=200, seed=21)
  gen_calculus_tangent(count=50, seed=34)
  gen_cubic_concave_v2(count=50, seed=201)
  gen_cubic_increasing_v2(count=50, seed=202)
  gen_cubic_intercept_v2(count=50, seed=203)
  gen_cubic_turning(count=50, seed=33)
  gen_derivative_first_principles(count=50, seed=32)
  gen_derivative_rules(count=200, seed=

In [43]:
import sys, json
from pathlib import Path
ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
sys.path.insert(0, str(ROOT / "src"))

import importlib, tutor
for m in list(sys.modules):
    if m.startswith("tutor."):
        del sys.modules[m]
importlib.invalidate_caches()

from tutor import analogue_library
from tutor import step_schemes

# ── grab a spread of templates across topics ──
pool = analogue_library.POOL
all_tids = sorted(pool.keys())
print(f"total templates in pool: {len(all_tids)}")
print()

# sample: first, last, and 8 spread across
import random
random.seed(42)
sample = list(set([all_tids[0], all_tids[len(all_tids)//2], all_tids[-1]] +
                  random.sample(all_tids, min(7, len(all_tids)))))

for tid in sample:
    variants = pool[tid]
    if not variants:
        continue
    ex = variants[0]
    print("=" * 70)
    print(f"template: {tid}")
    print(f"  variants: {len(variants)}")
    print(f"  problem:  {ex.get('problem','')[:80]}")

    # what does the memo scheme say?
    scheme = None
    try:
        scheme = step_schemes.build_scheme(tid, ex, ex.get("steps", [""])[-1])
    except Exception:
        pass

    print()
    print("  --- MEMO expects ---")
    if scheme:
        for s in scheme:
            print(f"    {s.get('index','?')}. {s.get('desc','?')}  ({s.get('marks','?')} mark)")
    else:
        print("    (no scheme found)")

    print()
    print("  --- ANALOGUE produces ---")
    for i, st in enumerate(ex.get("steps", []), 1):
        print(f"    {i}. {st}")
    print()

total templates in pool: 64

template: derivative_first_principles
  variants: 50
  problem:  Differentiate f(x) = -6x^2 from first principles.

  --- MEMO expects ---
    (no scheme found)

  --- ANALOGUE produces ---
    1. Formula: f'(x) = lim_{h->0} [f(x + h) - f(x)] / h
    2. f(x + h) = -6(x + h)^2 = -6x^2 + -12xh + -6h^2
    3. f(x + h) - f(x) = -12xh + -6h^2
    4. Divide by h: -12x + -6h
    5. Take limit as h -> 0:  f'(x) = -12x

template: hyperbola_equation
  variants: 50
  problem:  Find the hyperbola with asymptotes x = 1 and y = -5, passing through x = 0.

  --- MEMO expects ---
    (no scheme found)

  --- ANALOGUE produces ---
    1. Form: y = a/(x - p) + q
    2. From asymptotes: y = a/(x - (1)) + -5
    3. Substitute a given point to solve for a.
    4. Answer:  y = -6/(x - (1)) + -5

template: trig_written
  variants: 11
  problem:  Prove that sin(180 - x) = sin x.

  --- MEMO expects ---
    (no scheme found)

  --- ANALOGUE produces ---
    1. sin(180 - x) is in th

In [44]:
import sys, inspect
from pathlib import Path
ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
sys.path.insert(0, str(ROOT / "src"))

import importlib, tutor
for m in list(sys.modules):
    if m.startswith("tutor."):
        del sys.modules[m]
importlib.invalidate_caches()

from tutor import step_schemes, analogue_library

# ── 1. What's the signature? ──
print("── build_scheme signature ──")
print(inspect.signature(step_schemes.build_scheme))
print()
print("── build_scheme source ──")
try:
    src = inspect.getsource(step_schemes.build_scheme)
    for line in src.split("\n"):
        print("  " + line)
except Exception as e:
    print("  could not get source:", e)

print()
print("── what else is in step_schemes? ──")
for name in dir(step_schemes):
    if name.startswith("_"):
        continue
    obj = getattr(step_schemes, name)
    kind = type(obj).__name__
    print(f"  {name}  ({kind})")

# ── 2. What does it actually return? ──
print()
print("── test call with correct signature ──")
tid = "quadratic_standard_factorable"
pool = analogue_library.POOL.get(tid, [])
if pool:
    ex = pool[0]
    expected = ex.get("steps", [""])[-1].split(":")[-1].strip()
    print(f"  template_id: {tid}")
    print(f"  expected:    {expected!r}")
    try:
        result = step_schemes.build_scheme(tid, ex, expected)
        print(f"  returned:    {result!r}")
        if result:
            print(f"  scheme length: {len(result)}")
            for s in result:
                print(f"    {s}")
    except Exception as e:
        print(f"  EXCEPTION: {type(e).__name__}: {e}")

# ── 3. Does it read a data file? ──
print()
print("── files in the step_schemes module's directory ──")
for p in (ROOT / "src" / "tutor").rglob("*scheme*"):
    print(f"  {p.relative_to(ROOT)}")
for p in (ROOT / "data" / "processed" / "tutor").rglob("*scheme*"):
    print(f"  {p.relative_to(ROOT)}")

── build_scheme signature ──
(template_id: 'str', analogue: 'dict', expected_answer: 'str') -> 'Optional[list[dict]]'

── build_scheme source ──
  def build_scheme(template_id: str, analogue: dict, expected_answer: str) -> Optional[list[dict]]:
      """Return a step scheme for the given analogue, or None if unsupported."""
      builder = _BUILDERS.get(template_id)
      if builder is None:
          return None
      problem = analogue.get("problem", "")
      try:
          return builder(problem, expected_answer)
      except Exception:
          return None
  

── what else is in step_schemes? ──
  Optional  (_SpecialForm)
  annotations  (_Feature)
  build_scheme  (function)
  re  (module)

── test call with correct signature ──
  template_id: quadratic_standard_factorable
  expected:    'x = 8 or x = -4'
  returned:    [{'index': 1, 'desc': 'Factorise the quadratic', 'marks': 1, 'forms': ['(x - 8)(x - -4) = 0', '(x - -4)(x - 8) = 0', '(x-8)(x--4)=0', '(x--4)(x-8)=0', '(x + -8)(x 

In [45]:
import sys
from pathlib import Path
ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
sys.path.insert(0, str(ROOT / "src"))

import importlib, tutor
for m in list(sys.modules):
    if m.startswith("tutor."):
        del sys.modules[m]
importlib.invalidate_caches()

from tutor import step_schemes, analogue_library

builders = getattr(step_schemes, "_BUILDERS", {})
print(f"templates with a registered scheme builder: {len(builders)}")
for tid in sorted(builders.keys()):
    print(f"  {tid}")
print()

pool_tids = sorted(analogue_library.POOL.keys())
print(f"templates in the pool: {len(pool_tids)}")
print()

have = [t for t in pool_tids if t in builders]
missing = [t for t in pool_tids if t not in builders]
print(f"pool templates WITH a scheme:  {len(have)}")
print(f"pool templates WITHOUT:        {len(missing)}")
print()

if missing:
    print("── templates WITHOUT a scheme (grader can't mark these) ──")
    for t in missing:
        n = len(analogue_library.POOL[t])
        print(f"  {t:<40} {n} variants")

templates with a registered scheme builder: 2
  quadratic_factored
  quadratic_standard_factorable

templates in the pool: 64

pool templates WITH a scheme:  2
pool templates WITHOUT:        62

── templates WITHOUT a scheme (grader can't mark these) ──
  analytical_area                          43 variants
  analytical_circle_equation               50 variants
  analytical_circle_radius                 50 variants
  analytical_distance                      199 variants
  analytical_gradient                      168 variants
  analytical_inclination                   50 variants
  analytical_line_equation                 182 variants
  analytical_midpoint                      50 variants
  analytical_tangent_equation              50 variants
  analytical_translation                   52 variants
  ap_sum                                   200 variants
  ap_term                                  200 variants
  calculus_tangent                         50 variants
  cubic_concave           

In [46]:
import sys, re, json, importlib
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
GEN  = ROOT / "src" / "tutor" / "analogue_generators.py"
sys.path.insert(0, str(ROOT / "src"))

# ══════════════════════════════════════════════════════════════
# 1. Show what's being replaced
# ══════════════════════════════════════════════════════════════
src = GEN.read_text(encoding="utf-8")
lines = src.splitlines()

start = None
end = None
for i, line in enumerate(lines):
    if line.startswith("def gen_quadratic_standard_factorable"):
        start = i
    elif start is not None and (line.startswith("def ") or line.startswith("class ")) and i > start:
        end = i
        break
if end is None:
    end = len(lines)

print("── current gen_quadratic_standard_factorable ──")
for i in range(start, min(start + 8, end)):
    print(f"  {lines[i]}")
print(f"  ... ({end - start} lines total)")
print()

# ══════════════════════════════════════════════════════════════
# 2. Build the replacement
# ══════════════════════════════════════════════════════════════
NEW_FN = '''def gen_quadratic_standard_factorable(count=50, seed=101):
    """Canonical 3-step memo output for factorable quadratics.

    Steps produced (match the memo exactly):
      1. factored form           (x + 4)(x - 3) = 0
      2. set each factor to zero x + 4 = 0  or  x - 3 = 0
      3. both roots              x = -4  or  x = 3
    """
    import random as _r
    rng = _r.Random(seed)
    out = []
    seen = set()
    attempts = 0

    def signed_coeff(v):
        if v == 0:  return ""
        if v == 1:  return " + "
        if v == -1: return " - "
        if v > 0:   return f" + {v}"
        return f" - {abs(v)}"

    while len(out) < count and attempts < count * 8:
        attempts += 1
        r1 = rng.randint(-10, 10)
        r2 = rng.randint(-10, 10)
        if r1 == 0 or r2 == 0 or r1 == r2:
            continue
        key = tuple(sorted((r1, r2)))
        if key in seen:
            continue
        seen.add(key)

        b = -(r1 + r2)
        c = r1 * r2

        eq = f"x^2{signed_coeff(b)}x{signed_coeff(c)} = 0"
        eq = eq.replace("  ", " ").strip()

        def factor(r):
            return f"(x - {r})" if r > 0 else f"(x + {abs(r)})"

        def setzero(r):
            return f"x - {r} = 0" if r > 0 else f"x + {abs(r)} = 0"

        factored   = f"{factor(r1)}{factor(r2)} = 0"
        setzeros   = f"{setzero(r1)}  or  {setzero(r2)}"
        roots_line = f"x = {r1} or x = {r2}"

        out.append({
            "problem": f"Solve {eq}",
            "steps": [factored, setzeros, roots_line],
            "note": "b = -(sum of roots); c = (product of roots).",
        })
    return out
'''

# Replace the old function with the new one
new_src = "\n".join(lines[:start]) + "\n" + NEW_FN + "\n" + "\n".join(lines[end:])
GEN.write_text(new_src, encoding="utf-8")
print(f"✓ generator rewritten ({len(lines)} → {len(new_src.splitlines())} lines)")

# ══════════════════════════════════════════════════════════════
# 3. Reload and rebuild the pool
# ══════════════════════════════════════════════════════════════
for m in list(sys.modules):
    if m.startswith("tutor."):
        del sys.modules[m]
importlib.invalidate_caches()

from tutor import analogue_generators, analogue_library

new_variants = analogue_generators.gen_quadratic_standard_factorable(count=50, seed=101)
print(f"✓ generated {len(new_variants)} fresh variants")

# First three look like this:
print()
print("── first 3 variants ──")
for ex in new_variants[:3]:
    print(f"  problem: {ex['problem']}")
    for i, s in enumerate(ex["steps"], 1):
        print(f"    {i}. {s}")
    print()

# Update the in-memory pool
analogue_library.POOL["quadratic_standard_factorable"] = new_variants
print(f"✓ in-memory pool updated")

# Persist to whatever JSON the library loads from, if it exists
persisted = False
for candidate in [
    ROOT / "data" / "processed" / "tutor" / "analogue_library.json",
    ROOT / "data" / "processed" / "tutor" / "analogues.json",
    ROOT / "data" / "processed" / "tutor" / "pool.json",
]:
    if candidate.exists():
        try:
            data = json.loads(candidate.read_text(encoding="utf-8"))
            if isinstance(data, dict) and "quadratic_standard_factorable" in data:
                data["quadratic_standard_factorable"] = new_variants
                candidate.write_text(json.dumps(data, ensure_ascii=False, indent=2),
                                     encoding="utf-8")
                print(f"✓ persisted to {candidate.relative_to(ROOT)}")
                persisted = True
        except Exception as e:
            print(f"  could not update {candidate.name}: {e}")

if not persisted:
    print("  (no JSON file found to update — pool is in-memory only for now)")

# ══════════════════════════════════════════════════════════════
# 4. Acceptance test
# ══════════════════════════════════════════════════════════════
from tutor.learner_store import open_store
from tutor.schemas import Problem
from tutor import drill

store = open_store(ROOT / "data" / "learner_store" / "learners.db")
anchor = Problem(
    problem_id="2023_P1_Q1.1.1",
    skill_id="algebra.quadratic.solve",
    topic="Algebra", subtopic="Quadratic equations",
    structure_type="routine_calculation",
    prompt="Solve for x:  x^2 + x - 12 = 0",
    expected_answer="x = -4 or x = 3",
)

def show(title, result):
    print(f"── {title} ──")
    print(f"  match:      {result['match']}")
    print(f"  mark:       {result['mark']}")
    print(f"  your work:  {result.get('your_working')}")
    print(f"  memo steps:")
    for s in result.get("memo_steps", []):
        print(f"    {s['status']:12} {s['desc']}  ({s['marks']})  form: {s.get('first_form','')}")
    print(f"  diagnosis:  {result.get('diagnosis','')}")
    print(f"  sibling:    {'yes' if result.get('sibling') else 'no'}")
    print()

# Test 1: wrong sign on first root, missing method
r1 = drill.grade_submission(store, "tanaka", "acc-1", anchor,
                            ["x = 4 or x = 3"], "")
show("TEST 1  wrong sign (x = 4 or x = 3)", r1)

# Test 2: full correct method
r2 = drill.grade_submission(store, "tanaka", "acc-2", anchor,
                            ["(x + 4)(x - 3) = 0",
                             "x + 4 = 0  or  x - 3 = 0",
                             "x = -4 or x = 3"], "")
show("TEST 2  full correct method", r2)

# Test 3: right answer, missing middle step
r3 = drill.grade_submission(store, "tanaka", "acc-3", anchor,
                            ["(x + 4)(x - 3) = 0",
                             "x = -4 or x = 3"], "")
show("TEST 3  right answer, missing set-each-factor", r3)

# ══════════════════════════════════════════════════════════════
# 5. Verdict
# ══════════════════════════════════════════════════════════════
print("═" * 60)
print("ACCEPTANCE")
print("═" * 60)

def check(cond, label):
    print(f"  {'✓' if cond else '✗'}  {label}")

check(len(new_variants) >= 40, "at least 40 variants generated")
check(all(len(v["steps"]) == 3 for v in new_variants[:20]),
      "variants have exactly 3 steps each")
check(r1["match"] == "NO_MATCH", "test 1: wrong sign → NO_MATCH")
check(r1["mark"]["total"] > 0, "test 1: mark populated (not zero total)")
check("sign" in r1.get("diagnosis", "").lower() or "wrong" in r1.get("diagnosis","").lower(),
      "test 1: diagnosis names the sign problem")
check(bool(r1.get("sibling")), "test 1: sibling present")
check(r2["match"] == "MATCH", "test 2: full method → MATCH")
check(r2["mark"]["awarded"] == r2["mark"]["total"] if r2["mark"]["total"] else False,
      "test 2: full marks")
check(not r2.get("sibling"), "test 2: no sibling on clean full match")
check(r3["match"] == "MATCH", "test 3: right answer → MATCH")
check(r3["mark"]["awarded"] < r3["mark"]["total"] if r3["mark"]["total"] else False,
      "test 3: partial marks (missing middle step)")

── current gen_quadratic_standard_factorable ──
  def gen_quadratic_standard_factorable(count=50, seed=101):
      """Canonical 3-step memo output with reasoning lines and diagnostic meta."""
      import random as _r
      rng = _r.Random(seed)
      out, seen = [], set()
      tries = 0
      while len(out) < count and tries < count * 40:
          tries += 1
  ... (51 lines total)

✓ generator rewritten (1949 → 1953 lines)
✓ generated 50 fresh variants

── first 3 variants ──
  problem: Solve x^2 - 4x - 32 = 0
    1. (x - 8)(x + 4) = 0
    2. x - 8 = 0  or  x + 4 = 0
    3. x = 8 or x = -4

  problem: Solve x^2 - 8x + 7 = 0
    1. (x - 7)(x - 1) = 0
    2. x - 7 = 0  or  x - 1 = 0
    3. x = 7 or x = 1

  problem: Solve x^2 + 5x - 36 = 0
    1. (x - 4)(x + 9) = 0
    2. x - 4 = 0  or  x + 9 = 0
    3. x = 4 or x = -9

✓ in-memory pool updated
  (no JSON file found to update — pool is in-memory only for now)


── TEST 1  wrong sign (x = 4 or x = 3) ──
  match:      NO_MATCH
  mark:       {'awarded': 0, 'total': 3}
  your work:  ['x = 4 or x = 3']
  memo steps:
    NOT_MATCHED  Factorise  (1)  form: None
    NOT_MATCHED  Set each factor to zero  (1)  form: None
    NOT_MATCHED  Write both roots  (1)  form: None
  diagnosis:  You wrote: x = 4 or x = 3.  The memo has: x = -4 or x = 3.  The numbers match — one sign is wrong. When a term crosses the equals sign, its sign flips.
  sibling:    yes

── TEST 2  full correct method ──
  match:      MATCH
  mark:       {'awarded': 3, 'total': 3}
  your work:  ['(x + 4)(x - 3) = 0', 'x + 4 = 0 or x - 3 = 0', 'x = -4 or x = 3']
  memo steps:
    MATCHED      Factorise  (1)  form: None
    MATCHED      Set each factor to zero  (1)  form: None
    MATCHED      Write both roots  (1)  form: None
  diagnosis:  Full method. All memo steps shown.
  sibling:    no

── TEST 3  right answer, missing set-each-factor ──
  match:      MATCH
  mark:       {'awarded': 

In [47]:
import sys, importlib, inspect
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
sys.path.insert(0, str(ROOT / "src"))

# ── 1. Clear BOTH sys.modules entries AND parent-package attributes ──
import tutor
for name in list(vars(tutor).keys()):
    if not name.startswith("__"):
        try:
            delattr(tutor, name)
        except Exception:
            pass

for m in list(sys.modules):
    if m == "tutor" or m.startswith("tutor."):
        del sys.modules[m]

importlib.invalidate_caches()

# ── 2. Fresh import ──
from tutor import analogue_generators, analogue_library

# ── 3. Confirm the new generator is loaded ──
src_fn = inspect.getsource(analogue_generators.gen_quadratic_standard_factorable)
print("loaded function contains 'set each factor':",
      "set each factor" in src_fn.lower())
print("loaded function line count:", len(src_fn.splitlines()))
print()

# ── 4. Regenerate ──
new_variants = analogue_generators.gen_quadratic_standard_factorable(count=50, seed=101)
print(f"generated {len(new_variants)} variants")
print()
for ex in new_variants[:3]:
    print(f"  {ex['problem']}")
    for i, s in enumerate(ex["steps"], 1):
        print(f"    {i}. {s}")
    print()

# ── 5. Verify shape ──
all_three = all(len(v["steps"]) == 3 for v in new_variants)
print(f"all variants have exactly 3 steps: {all_three}")
if not all_three:
    counts = {}
    for v in new_variants:
        n = len(v["steps"])
        counts[n] = counts.get(n, 0) + 1
    print(f"  step-count distribution: {counts}")

# ── 6. Update in-memory pool ──
analogue_library.POOL["quadratic_standard_factorable"] = new_variants
print("POOL updated in memory")

# ── 7. Re-run the failing test in-process ──
from tutor.learner_store import open_store
from tutor.schemas import Problem
from tutor import drill

store = open_store(ROOT / "data" / "learner_store" / "learners.db")
anchor = Problem(
    problem_id="2023_P1_Q1.1.1",
    skill_id="algebra.quadratic.solve",
    topic="Algebra", subtopic="Quadratic equations",
    structure_type="routine_calculation",
    prompt="Solve for x:  x^2 + x - 12 = 0",
    expected_answer="x = -4 or x = 3",
)

print()
r = drill.grade_submission(store, "tanaka", "verify-2", anchor,
                            ["x = 4 or x = 3"], "")
print("sibling present:", bool(r.get("sibling")))
if r.get("sibling"):
    print("sibling problem:", r["sibling"]["problem"])
    print("sibling steps:")
    for i, s in enumerate(r["sibling"]["steps"], 1):
        print(f"  {i}. {s}")
    print()
    print("sibling has middle step:",
          any("= 0" in s and "or" in s for s in r["sibling"]["steps"]))

loaded function contains 'set each factor': True
loaded function line count: 54

generated 50 variants

  Solve x^2 - 4x - 32 = 0
    1. (x - 8)(x + 4) = 0
    2. x - 8 = 0  or  x + 4 = 0
    3. x = 8 or x = -4

  Solve x^2 - 8x + 7 = 0
    1. (x - 7)(x - 1) = 0
    2. x - 7 = 0  or  x - 1 = 0
    3. x = 7 or x = 1

  Solve x^2 + 5x - 36 = 0
    1. (x - 4)(x + 9) = 0
    2. x - 4 = 0  or  x + 9 = 0
    3. x = 4 or x = -9

all variants have exactly 3 steps: True
POOL updated in memory

sibling present: True
sibling problem: Solve x^2 + 4x - 32 = 0
sibling steps:
  1. (x - 4)(x + 8) = 0
  2. x - 4 = 0  or  x + 8 = 0
  3. x = 4 or x = -8

sibling has middle step: True


In [48]:
import sys, importlib
from pathlib import Path
ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
sys.path.insert(0, str(ROOT / "src"))

# Fresh import — simulate post-restart state
import tutor
for name in list(vars(tutor).keys()):
    if not name.startswith("__"):
        try: delattr(tutor, name)
        except Exception: pass
for m in list(sys.modules):
    if m == "tutor" or m.startswith("tutor."):
        del sys.modules[m]
importlib.invalidate_caches()

from tutor import analogue_library

pool = analogue_library.POOL.get("quadratic_standard_factorable", [])
print("pool size after fresh import:", len(pool))
if pool:
    print("first variant steps:")
    for i, s in enumerate(pool[0].get("steps", []), 1):
        print(f"  {i}. {s}")
    print()
    print("3-step canonical?", len(pool[0]["steps"]) == 3)

pool size after fresh import: 50
first variant steps:
  1. (x - 8)(x + 4) = 0
  2. x - 8 = 0  or  x + 4 = 0
  3. x = 8 or x = -4

3-step canonical? True


In [49]:
import urllib.request, json, re

print("── 1. What the server is returning right now ──")
req = urllib.request.Request(
    "http://127.0.0.1:8765/grade",
    data=json.dumps({
        "learner_lines_latex": ["x = 4 or x = 3"],
        "sbc_message": "",
        "tab_used": "Algebra",
    }).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)
try:
    with urllib.request.urlopen(req, timeout=5) as r:
        body = json.loads(r.read())
    print("  keys:", sorted(body.keys()))
    print("  has 'mark':       ", "mark" in body)
    print("  has 'diagnosis':  ", "diagnosis" in body)
    print("  has 'memo_steps': ", "memo_steps" in body)
    print("  has 'your_working':", "your_working" in body)
    print("  has 'sibling':    ", "sibling" in body)
    if "mark" in body:
        print("  mark value:       ", body["mark"])
    if "memo_steps" in body:
        print("  memo_steps count: ", len(body["memo_steps"]))
    if "sibling" in body and body["sibling"]:
        print("  sibling steps:    ", len(body["sibling"].get("steps", [])))
except Exception as e:
    print("  FAILED:", e)

print()
print("── 2. What the served HTML expects ──")
try:
    with urllib.request.urlopen("http://127.0.0.1:8765/", timeout=5) as r:
        html = r.read().decode("utf-8")
    for probe, label in [
        ("res.mark",       "reads res.mark"),
        ("res.memo_steps", "reads res.memo_steps"),
        ("res.diagnosis",  "reads res.diagnosis"),
        ("res.your_working","reads res.your_working"),
        ("res.sibling",    "reads res.sibling"),
        ("loadQuestion();","calls loadQuestion at boot"),
        ("strip-diag",     "has strip-diag CSS"),
        ("qcardBody",      "has question card div"),
    ]:
        print(f"  {'✓' if probe in html else '✗'}  {label}")
except Exception as e:
    print("  FAILED:", e)

── 1. What the server is returning right now ──
  keys: ['mastery', 'match', 'matcher', 'raw_lines', 'step_feedback', 'tanaka_message']
  has 'mark':        False
  has 'diagnosis':   False
  has 'memo_steps':  False
  has 'your_working': False
  has 'sibling':     False

── 2. What the served HTML expects ──
  ✓  reads res.mark
  ✓  reads res.memo_steps
  ✓  reads res.diagnosis
  ✓  reads res.your_working
  ✓  reads res.sibling
  ✓  calls loadQuestion at boot
  ✓  has strip-diag CSS
  ✓  has question card div


In [50]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\src")))

# Rebuild the pool with the new generator (in-memory, from fresh import)
from tutor import analogue_generators, analogue_library
analogue_library.POOL["quadratic_standard_factorable"] = \
    analogue_generators.gen_quadratic_standard_factorable(count=50, seed=101)
print("pool rebuilt:", len(analogue_library.POOL["quadratic_standard_factorable"]), "variants")
print("first variant steps:", analogue_library.POOL["quadratic_standard_factorable"][0]["steps"])

# Start the server
from tutor import drill_server
drill_server.serve_in_thread()

import time, urllib.request, json
time.sleep(0.8)

# Health check
with urllib.request.urlopen("http://127.0.0.1:8765/health", timeout=3) as r:
    print("health:", r.read().decode())

# Verify the server returns the new schema
req = urllib.request.Request(
    "http://127.0.0.1:8765/grade",
    data=json.dumps({"learner_lines_latex": ["x = 4 or x = 3"],
                     "sbc_message": "", "tab_used": "Algebra"}).encode(),
    headers={"Content-Type": "application/json"}, method="POST")
with urllib.request.urlopen(req, timeout=5) as r:
    body = json.loads(r.read())
print()
print("server keys:", sorted(body.keys()))
print("has 'mark':      ", "mark" in body)
print("has 'diagnosis': ", "diagnosis" in body)
print("has 'memo_steps':", "memo_steps" in body)
print("has 'sibling':   ", "sibling" in body)
if "mark" in body:
    print("mark:", body["mark"])
if "memo_steps" in body:
    for s in body["memo_steps"]:
        print(f"  {s['status']:12} {s['desc']}  ({s['marks']})")

pool rebuilt: 50 variants
first variant steps: ['(x - 8)(x + 4) = 0', 'x - 8 = 0  or  x + 4 = 0', 'x = 8 or x = -4']
drill_server at http://127.0.0.1:8765drill_server running in background thread on http://127.0.0.1:8765

  keyboard: http://127.0.0.1:8765/
  anchor:   2023_P1_Q1.1.1  |  learner: tanaka
health: {"ok": true, "anchor": "2023_P1_Q1.1.1"}

server keys: ['mastery', 'match', 'matcher', 'raw_lines', 'step_feedback', 'tanaka_message']
has 'mark':       False
has 'diagnosis':  False
has 'memo_steps': False
has 'sibling':    False


In [51]:
import sys, re, json, importlib
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
GEN  = ROOT / "src" / "tutor" / "analogue_generators.py"
DRILL = ROOT / "src" / "tutor" / "drill.py"
sys.path.insert(0, str(ROOT / "src"))

# ══════════════════════════════════════════════════════════════
# 1. Rewrite the generator with working + meta
# ══════════════════════════════════════════════════════════════
src = GEN.read_text(encoding="utf-8")
lines = src.splitlines()

start = None; end = None
for i, line in enumerate(lines):
    if line.startswith("def gen_quadratic_standard_factorable"):
        start = i
    elif start is not None and (line.startswith("def ") or line.startswith("class ")) and i > start:
        end = i; break
if end is None: end = len(lines)

NEW_FN = '''def gen_quadratic_standard_factorable(count=50, seed=101):
    """Canonical 3-step memo output with reasoning lines and diagnostic meta."""
    import random as _r
    rng = _r.Random(seed)
    out, seen = [], set()
    tries = 0
    while len(out) < count and tries < count * 40:
        tries += 1
        r1 = rng.randint(-10, 10)
        r2 = rng.randint(-10, 10)
        if r1 == 0 or r2 == 0 or r1 == r2: continue
        key = tuple(sorted((r1, r2)))
        if key in seen: continue
        seen.add(key)

        # Equation: x^2 + bx + c where roots are r1, r2
        b = -(r1 + r2)
        c = r1 * r2
        def sg(v):
            if v == 0: return ""
            if v == 1: return " + "
            if v == -1: return " - "
            return f" + {v}" if v > 0 else f" - {abs(v)}"
        eq = f"x^2{sg(b)}x{sg(c)} = 0".replace("  ", " ").strip()

        # Factored form: (x - r1)(x - r2) = 0 with sign-clean brackets
        def br(r):
            return f"(x - {r})" if r > 0 else f"(x + {abs(r)})"
        factored = f"{br(r1)}{br(r2)} = 0"

        def sz(r):
            return f"x - {r} = 0" if r > 0 else f"x + {abs(r)} = 0"
        setzeros = f"{sz(r1)}  or  {sz(r2)}"
        roots = f"x = {r1} or x = {r2}"

        # Reasoning: two bracket constants p, q with p+q=b and pq=c
        p, q = -r1, -r2
        working = [
            f"Need two numbers that multiply to {c} and add to {b}.",
            f"Those are {p} and {q}.  {p} * {q} = {c}.  {p} + {q} = {b}.",
        ]

        out.append({
            "problem": f"Solve {eq}",
            "steps":   [factored, setzeros, roots],
            "working": working,
            "note":    "b = -(sum of roots); c = (product of roots).",
            "meta":    {"r1": r1, "r2": r2, "b": b, "c": c, "p": p, "q": q},
        })
    return out
'''

new_src = "\n".join(lines[:start]) + "\n" + NEW_FN + "\n" + "\n".join(lines[end:])
GEN.write_text(new_src, encoding="utf-8")
print(f"✓ generator rewritten")

# ══════════════════════════════════════════════════════════════
# 2. Diagnostic engine — add to drill.py
# ══════════════════════════════════════════════════════════════
diag_src = DRILL.read_text(encoding="utf-8")

DIAG = r'''# ============================================================
# Per-line diagnostic engine (factorable quadratics)
# ============================================================
import re as _diag_re


_FACTORED = _diag_re.compile(
    r"\(\s*x\s*([+-])\s*(\d+)\s*\)\s*\(\s*x\s*([+-])\s*(\d+)\s*\)\s*=\s*0"
)
_SET_ZERO = _diag_re.compile(
    r"x\s*([+-])\s*(\d+)\s*=\s*0.*?x\s*([+-])\s*(\d+)\s*=\s*0"
)
_ROOTS = _diag_re.compile(
    r"x\s*=\s*(-?\d+)\s*(?:or|,)\s*x\s*=\s*(-?\d+)"
)
_REASONING = _diag_re.compile(
    r"(?:multiply|add|product|sum|times|×|×).*(?:\d).*(?:\d)",
    _diag_re.IGNORECASE,
)


def _sign_const(sign, num):
    return int(num) if sign == "+" else -int(num)


def _diagnose_factorable_line(text, meta):
    """Return a per-line diagnosis dict for one learner line."""
    s = (text or "").strip()
    if not s:
        return None

    # Try factored form
    m = _FACTORED.search(s)
    if m:
        p_l = _sign_const(m.group(1), m.group(2))
        q_l = _sign_const(m.group(3), m.group(4))
        sum_l = p_l + q_l
        prod_l = p_l * q_l
        b = meta.get("b"); c = meta.get("c")
        if sum_l == b and prod_l == c:
            msg = (f"Correct factorisation. "
                   f"Bracket constants {p_l} and {q_l} "
                   f"sum to {b} and multiply to {c}.")
            return {"line": s, "as": "factorise", "verdict": "correct", "message": msg}
        # Distinguish sign-only vs total mismatch
        msg = (f"You used bracket constants {p_l} and {q_l}. "
               f"Their sum is {sum_l} (should be {b}). "
               f"Their product is {prod_l} (should be {c}).")
        if abs(p_l) == abs(meta["p"]) or abs(q_l) == abs(meta["q"]):
            msg += " One of the constants has the right size but the wrong sign."
        return {"line": s, "as": "factorise", "verdict": "wrong", "message": msg}

    # Try set-each-factor
    m = _SET_ZERO.search(s)
    if m:
        p_l = _sign_const(m.group(1), m.group(2))
        q_l = _sign_const(m.group(3), m.group(4))
        if p_l == meta["p"] and q_l == meta["q"]:
            return {"line": s, "as": "set-each-factor", "verdict": "correct",
                    "message": "Correct — each bracket set to zero."}
        return {"line": s, "as": "set-each-factor", "verdict": "wrong",
                "message": f"These do not match the memo factorisation. "
                           f"The memo brackets have constants {meta['p']} and {meta['q']}."}

    # Try roots
    m = _ROOTS.search(s)
    if m:
        lr1, lr2 = int(m.group(1)), int(m.group(2))
        if {lr1, lr2} == {meta["r1"], meta["r2"]}:
            return {"line": s, "as": "roots", "verdict": "correct",
                    "message": "Correct roots."}
        return {"line": s, "as": "roots", "verdict": "wrong",
                "message": (f"Your roots are {lr1} and {lr2}. "
                            f"The memo has {meta['r1']} and {meta['r2']}.")}

    # Reasoning / working line
    if _REASONING.search(s):
        return {"line": s, "as": "reasoning", "verdict": "note",
                "message": "Working noted — this is not a memo step, no mark attached."}

    return {"line": s, "as": "unknown", "verdict": "note",
            "message": "Cannot classify this line."}


def _diagnose_factorable(learner_lines, meta):
    """Run per-line diagnosis for a factorable-quadratic attempt."""
    out = []
    factorise_was_wrong = False
    for line in learner_lines:
        d = _diagnose_factorable_line(line, meta)
        if not d:
            continue
        # If factorise was wrong, note the downstream effect
        if d["as"] == "roots" and d["verdict"] == "wrong" and factorise_was_wrong:
            d["message"] += (" Your roots match your own factorisation, "
                             "but both inherit the error from the line above.")
        if d["as"] == "factorise" and d["verdict"] == "wrong":
            factorise_was_wrong = True
        out.append(d)
    return out


'''

if "_diagnose_factorable" not in diag_src:
    # insert before grade_submission
    m = re.search(r"^def grade_submission\(", diag_src, re.MULTILINE)
    if not m:
        raise SystemExit("grade_submission not found")
    line_start = diag_src.rfind("\n", 0, m.start()) + 1
    diag_src = diag_src[:line_start] + DIAG + diag_src[line_start:]
    print("✓ diagnostic engine inserted")
else:
    print("· diagnostic engine already present")

# Hook into grade_submission — attach line_diagnosis to the outcome
hook_old = '''    outcome = {
        "match": None,
        "raw_lines": expanded,
        "your_working": plain_lines,'''
hook_new = '''    # per-line diagnostic — runs regardless of match verdict
    meta = None
    try:
        _tid = route_template(anchor.skill_id, anchor.prompt)
        if _tid == "quadratic_standard_factorable":
            _pool = _load_pool(_tid)
            for _v in _pool:
                if _v.get("problem","").strip().endswith(anchor.prompt.strip().split("Solve",1)[-1].strip()):
                    meta = _v.get("meta"); break
            if meta is None and _pool:
                meta = _pool[0].get("meta")
    except Exception:
        meta = None

    line_diagnosis = []
    if meta:
        try:
            line_diagnosis = _diagnose_factorable(plain_lines, meta)
        except Exception:
            line_diagnosis = []

    outcome = {
        "match": None,
        "raw_lines": expanded,
        "your_working": plain_lines,
        "line_diagnosis": line_diagnosis,'''

if "line_diagnosis" not in diag_src:
    if hook_old in diag_src:
        diag_src = diag_src.replace(hook_old, hook_new, 1)
        print("✓ line_diagnosis wired into grade_submission")
    else:
        print("✗ hook anchor not found")
else:
    print("· line_diagnosis already wired")

DRILL.write_text(diag_src, encoding="utf-8")
print(f"✓ drill.py written ({len(diag_src):,} chars)")

# ══════════════════════════════════════════════════════════════
# 3. Reload + in-process acceptance test
# ══════════════════════════════════════════════════════════════
import tutor
for name in list(vars(tutor).keys()):
    if not name.startswith("__"):
        try: delattr(tutor, name)
        except Exception: pass
for m in list(sys.modules):
    if m == "tutor" or m.startswith("tutor."):
        del sys.modules[m]
importlib.invalidate_caches()

from tutor import analogue_generators, analogue_library, drill
from tutor.learner_store import open_store
from tutor.schemas import Problem

analogue_library.POOL["quadratic_standard_factorable"] = \
    analogue_generators.gen_quadratic_standard_factorable(count=50, seed=101)

store = open_store(ROOT / "data" / "learner_store" / "learners.db")

# Force a specific variant: x^2 + 5x - 36 = 0, roots 4 and -9
variant = next(v for v in analogue_library.POOL["quadratic_standard_factorable"]
               if v["meta"]["r1"] == 4 and v["meta"]["r2"] == -9)
print(f"\nusing variant: {variant['problem']}")
print(f"  memo steps: {variant['steps']}")
print(f"  working:    {variant['working']}")

anchor = Problem(
    problem_id="test-1",
    skill_id="algebra.quadratic.solve",
    topic="Algebra", subtopic="Quadratic equations",
    structure_type="routine_calculation",
    prompt=variant["problem"],
    expected_answer=f"x = 4 or x = -9",
)

# ══════════════════════════════════════════════════════════════
# TEST A: child picks WRONG factors — writes (x + 4)(x - 8) = 0
# ══════════════════════════════════════════════════════════════
print()
print("═" * 60)
print("TEST A — child picks WRONG factors")
print("═" * 60)
result = drill.grade_submission(
    store, "tanaka", "dia-a", anchor,
    ["(x + 4)(x - 8) = 0", "x = -4 or x = 8"], "")
print(f"match: {result['match']}   mark: {result['mark']}")
print()
print("Per-line diagnosis:")
for d in result.get("line_diagnosis", []):
    print(f"  {d['verdict'].upper():8}  [{d['as']}]")
    print(f"    line: {d['line']}")
    print(f"    → {d['message']}")
    print()

# ══════════════════════════════════════════════════════════════
# TEST B: child gets it right, all 3 steps
# ══════════════════════════════════════════════════════════════
print("═" * 60)
print("TEST B — full correct method")
print("═" * 60)
result = drill.grade_submission(
    store, "tanaka", "dia-b", anchor,
    ["(x - 4)(x + 9) = 0", "x - 4 = 0 or x + 9 = 0", "x = 4 or x = -9"], "")
print(f"match: {result['match']}   mark: {result['mark']}")
print()
for d in result.get("line_diagnosis", []):
    print(f"  {d['verdict'].upper():8}  [{d['as']}]: {d['message']}")

✓ generator rewritten
✓ diagnostic engine inserted
✓ line_diagnosis wired into grade_submission
✓ drill.py written (44,545 chars)

using variant: Solve x^2 + 5x - 36 = 0
  memo steps: ['(x - 4)(x + 9) = 0', 'x - 4 = 0  or  x + 9 = 0', 'x = 4 or x = -9']
  working:    ['Need two numbers that multiply to -36 and add to 5.', 'Those are -4 and 9.  -4 * 9 = -36.  -4 + 9 = 5.']

════════════════════════════════════════════════════════════
TEST A — child picks WRONG factors
════════════════════════════════════════════════════════════
match: UNCERTAIN   mark: {'awarded': 0, 'total': 0}

Per-line diagnosis:
  WRONG     [factorise]
    line: (x + 4)(x - 8) = 0
    → You used bracket constants 4 and -8. Their sum is -4 (should be 5). Their product is -32 (should be -36). One of the constants has the right size but the wrong sign.

  WRONG     [roots]
    line: x = -4 or x = 8
    → Your roots are -4 and 8. The memo has 4 and -9. Your roots match your own factorisation, but both inherit the error 

In [52]:
import sys, re, json, importlib
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
GEN  = ROOT / "src" / "tutor" / "analogue_generators.py"
DRILL = ROOT / "src" / "tutor" / "drill.py"
sys.path.insert(0, str(ROOT / "src"))

# ══════════════════════════════════════════════════════════════
# 1. Rewrite the generator with working + meta
# ══════════════════════════════════════════════════════════════
src = GEN.read_text(encoding="utf-8")
lines = src.splitlines()

start = None; end = None
for i, line in enumerate(lines):
    if line.startswith("def gen_quadratic_standard_factorable"):
        start = i
    elif start is not None and (line.startswith("def ") or line.startswith("class ")) and i > start:
        end = i; break
if end is None: end = len(lines)

NEW_FN = '''def gen_quadratic_standard_factorable(count=50, seed=101):
    """Canonical 3-step memo output with reasoning lines and diagnostic meta."""
    import random as _r
    rng = _r.Random(seed)
    out, seen = [], set()
    tries = 0
    while len(out) < count and tries < count * 40:
        tries += 1
        r1 = rng.randint(-10, 10)
        r2 = rng.randint(-10, 10)
        if r1 == 0 or r2 == 0 or r1 == r2: continue
        key = tuple(sorted((r1, r2)))
        if key in seen: continue
        seen.add(key)

        # Equation: x^2 + bx + c where roots are r1, r2
        b = -(r1 + r2)
        c = r1 * r2
        def sg(v):
            if v == 0: return ""
            if v == 1: return " + "
            if v == -1: return " - "
            return f" + {v}" if v > 0 else f" - {abs(v)}"
        eq = f"x^2{sg(b)}x{sg(c)} = 0".replace("  ", " ").strip()

        # Factored form: (x - r1)(x - r2) = 0 with sign-clean brackets
        def br(r):
            return f"(x - {r})" if r > 0 else f"(x + {abs(r)})"
        factored = f"{br(r1)}{br(r2)} = 0"

        def sz(r):
            return f"x - {r} = 0" if r > 0 else f"x + {abs(r)} = 0"
        setzeros = f"{sz(r1)}  or  {sz(r2)}"
        roots = f"x = {r1} or x = {r2}"

        # Reasoning: two bracket constants p, q with p+q=b and pq=c
        p, q = -r1, -r2
        working = [
            f"Need two numbers that multiply to {c} and add to {b}.",
            f"Those are {p} and {q}.  {p} * {q} = {c}.  {p} + {q} = {b}.",
        ]

        out.append({
            "problem": f"Solve {eq}",
            "steps":   [factored, setzeros, roots],
            "working": working,
            "note":    "b = -(sum of roots); c = (product of roots).",
            "meta":    {"r1": r1, "r2": r2, "b": b, "c": c, "p": p, "q": q},
        })
    return out
'''

new_src = "\n".join(lines[:start]) + "\n" + NEW_FN + "\n" + "\n".join(lines[end:])
GEN.write_text(new_src, encoding="utf-8")
print(f"✓ generator rewritten")

# ══════════════════════════════════════════════════════════════
# 2. Diagnostic engine — add to drill.py
# ══════════════════════════════════════════════════════════════
diag_src = DRILL.read_text(encoding="utf-8")

DIAG = r'''# ============================================================
# Per-line diagnostic engine (factorable quadratics)
# ============================================================
import re as _diag_re


_FACTORED = _diag_re.compile(
    r"\(\s*x\s*([+-])\s*(\d+)\s*\)\s*\(\s*x\s*([+-])\s*(\d+)\s*\)\s*=\s*0"
)
_SET_ZERO = _diag_re.compile(
    r"x\s*([+-])\s*(\d+)\s*=\s*0.*?x\s*([+-])\s*(\d+)\s*=\s*0"
)
_ROOTS = _diag_re.compile(
    r"x\s*=\s*(-?\d+)\s*(?:or|,)\s*x\s*=\s*(-?\d+)"
)
_REASONING = _diag_re.compile(
    r"(?:multiply|add|product|sum|times|×|×).*(?:\d).*(?:\d)",
    _diag_re.IGNORECASE,
)


def _sign_const(sign, num):
    return int(num) if sign == "+" else -int(num)


def _diagnose_factorable_line(text, meta):
    """Return a per-line diagnosis dict for one learner line."""
    s = (text or "").strip()
    if not s:
        return None

    # Try factored form
    m = _FACTORED.search(s)
    if m:
        p_l = _sign_const(m.group(1), m.group(2))
        q_l = _sign_const(m.group(3), m.group(4))
        sum_l = p_l + q_l
        prod_l = p_l * q_l
        b = meta.get("b"); c = meta.get("c")
        if sum_l == b and prod_l == c:
            msg = (f"Correct factorisation. "
                   f"Bracket constants {p_l} and {q_l} "
                   f"sum to {b} and multiply to {c}.")
            return {"line": s, "as": "factorise", "verdict": "correct", "message": msg}
        # Distinguish sign-only vs total mismatch
        msg = (f"You used bracket constants {p_l} and {q_l}. "
               f"Their sum is {sum_l} (should be {b}). "
               f"Their product is {prod_l} (should be {c}).")
        if abs(p_l) == abs(meta["p"]) or abs(q_l) == abs(meta["q"]):
            msg += " One of the constants has the right size but the wrong sign."
        return {"line": s, "as": "factorise", "verdict": "wrong", "message": msg}

    # Try set-each-factor
    m = _SET_ZERO.search(s)
    if m:
        p_l = _sign_const(m.group(1), m.group(2))
        q_l = _sign_const(m.group(3), m.group(4))
        if p_l == meta["p"] and q_l == meta["q"]:
            return {"line": s, "as": "set-each-factor", "verdict": "correct",
                    "message": "Correct — each bracket set to zero."}
        return {"line": s, "as": "set-each-factor", "verdict": "wrong",
                "message": f"These do not match the memo factorisation. "
                           f"The memo brackets have constants {meta['p']} and {meta['q']}."}

    # Try roots
    m = _ROOTS.search(s)
    if m:
        lr1, lr2 = int(m.group(1)), int(m.group(2))
        if {lr1, lr2} == {meta["r1"], meta["r2"]}:
            return {"line": s, "as": "roots", "verdict": "correct",
                    "message": "Correct roots."}
        return {"line": s, "as": "roots", "verdict": "wrong",
                "message": (f"Your roots are {lr1} and {lr2}. "
                            f"The memo has {meta['r1']} and {meta['r2']}.")}

    # Reasoning / working line
    if _REASONING.search(s):
        return {"line": s, "as": "reasoning", "verdict": "note",
                "message": "Working noted — this is not a memo step, no mark attached."}

    return {"line": s, "as": "unknown", "verdict": "note",
            "message": "Cannot classify this line."}


def _diagnose_factorable(learner_lines, meta):
    """Run per-line diagnosis for a factorable-quadratic attempt."""
    out = []
    factorise_was_wrong = False
    for line in learner_lines:
        d = _diagnose_factorable_line(line, meta)
        if not d:
            continue
        # If factorise was wrong, note the downstream effect
        if d["as"] == "roots" and d["verdict"] == "wrong" and factorise_was_wrong:
            d["message"] += (" Your roots match your own factorisation, "
                             "but both inherit the error from the line above.")
        if d["as"] == "factorise" and d["verdict"] == "wrong":
            factorise_was_wrong = True
        out.append(d)
    return out


'''

if "_diagnose_factorable" not in diag_src:
    # insert before grade_submission
    m = re.search(r"^def grade_submission\(", diag_src, re.MULTILINE)
    if not m:
        raise SystemExit("grade_submission not found")
    line_start = diag_src.rfind("\n", 0, m.start()) + 1
    diag_src = diag_src[:line_start] + DIAG + diag_src[line_start:]
    print("✓ diagnostic engine inserted")
else:
    print("· diagnostic engine already present")

# Hook into grade_submission — attach line_diagnosis to the outcome
hook_old = '''    outcome = {
        "match": None,
        "raw_lines": expanded,
        "your_working": plain_lines,'''
hook_new = '''    # per-line diagnostic — runs regardless of match verdict
    meta = None
    try:
        _tid = route_template(anchor.skill_id, anchor.prompt)
        if _tid == "quadratic_standard_factorable":
            _pool = _load_pool(_tid)
            for _v in _pool:
                if _v.get("problem","").strip().endswith(anchor.prompt.strip().split("Solve",1)[-1].strip()):
                    meta = _v.get("meta"); break
            if meta is None and _pool:
                meta = _pool[0].get("meta")
    except Exception:
        meta = None

    line_diagnosis = []
    if meta:
        try:
            line_diagnosis = _diagnose_factorable(plain_lines, meta)
        except Exception:
            line_diagnosis = []

    outcome = {
        "match": None,
        "raw_lines": expanded,
        "your_working": plain_lines,
        "line_diagnosis": line_diagnosis,'''

if "line_diagnosis" not in diag_src:
    if hook_old in diag_src:
        diag_src = diag_src.replace(hook_old, hook_new, 1)
        print("✓ line_diagnosis wired into grade_submission")
    else:
        print("✗ hook anchor not found")
else:
    print("· line_diagnosis already wired")

DRILL.write_text(diag_src, encoding="utf-8")
print(f"✓ drill.py written ({len(diag_src):,} chars)")

# ══════════════════════════════════════════════════════════════
# 3. Reload + in-process acceptance test
# ══════════════════════════════════════════════════════════════
import tutor
for name in list(vars(tutor).keys()):
    if not name.startswith("__"):
        try: delattr(tutor, name)
        except Exception: pass
for m in list(sys.modules):
    if m == "tutor" or m.startswith("tutor."):
        del sys.modules[m]
importlib.invalidate_caches()

from tutor import analogue_generators, analogue_library, drill
from tutor.learner_store import open_store
from tutor.schemas import Problem

analogue_library.POOL["quadratic_standard_factorable"] = \
    analogue_generators.gen_quadratic_standard_factorable(count=50, seed=101)

store = open_store(ROOT / "data" / "learner_store" / "learners.db")

# Force a specific variant: x^2 + 5x - 36 = 0, roots 4 and -9
variant = next(v for v in analogue_library.POOL["quadratic_standard_factorable"]
               if v["meta"]["r1"] == 4 and v["meta"]["r2"] == -9)
print(f"\nusing variant: {variant['problem']}")
print(f"  memo steps: {variant['steps']}")
print(f"  working:    {variant['working']}")

anchor = Problem(
    problem_id="test-1",
    skill_id="algebra.quadratic.solve",
    topic="Algebra", subtopic="Quadratic equations",
    structure_type="routine_calculation",
    prompt=variant["problem"],
    expected_answer=f"x = 4 or x = -9",
)

# ══════════════════════════════════════════════════════════════
# TEST A: child picks WRONG factors — writes (x + 4)(x - 8) = 0
# ══════════════════════════════════════════════════════════════
print()
print("═" * 60)
print("TEST A — child picks WRONG factors")
print("═" * 60)
result = drill.grade_submission(
    store, "tanaka", "dia-a", anchor,
    ["(x + 4)(x - 8) = 0", "x = -4 or x = 8"], "")
print(f"match: {result['match']}   mark: {result['mark']}")
print()
print("Per-line diagnosis:")
for d in result.get("line_diagnosis", []):
    print(f"  {d['verdict'].upper():8}  [{d['as']}]")
    print(f"    line: {d['line']}")
    print(f"    → {d['message']}")
    print()

# ══════════════════════════════════════════════════════════════
# TEST B: child gets it right, all 3 steps
# ══════════════════════════════════════════════════════════════
print("═" * 60)
print("TEST B — full correct method")
print("═" * 60)
result = drill.grade_submission(
    store, "tanaka", "dia-b", anchor,
    ["(x - 4)(x + 9) = 0", "x - 4 = 0 or x + 9 = 0", "x = 4 or x = -9"], "")
print(f"match: {result['match']}   mark: {result['mark']}")
print()
for d in result.get("line_diagnosis", []):
    print(f"  {d['verdict'].upper():8}  [{d['as']}]: {d['message']}")

✓ generator rewritten
· diagnostic engine already present
· line_diagnosis already wired
✓ drill.py written (44,545 chars)

using variant: Solve x^2 + 5x - 36 = 0
  memo steps: ['(x - 4)(x + 9) = 0', 'x - 4 = 0  or  x + 9 = 0', 'x = 4 or x = -9']
  working:    ['Need two numbers that multiply to -36 and add to 5.', 'Those are -4 and 9.  -4 * 9 = -36.  -4 + 9 = 5.']

════════════════════════════════════════════════════════════
TEST A — child picks WRONG factors
════════════════════════════════════════════════════════════
match: UNCERTAIN   mark: {'awarded': 0, 'total': 0}

Per-line diagnosis:
  WRONG     [factorise]
    line: (x + 4)(x - 8) = 0
    → You used bracket constants 4 and -8. Their sum is -4 (should be 5). Their product is -32 (should be -36). One of the constants has the right size but the wrong sign.

  WRONG     [roots]
    line: x = -4 or x = 8
    → Your roots are -4 and 8. The memo has 4 and -9. Your roots match your own factorisation, but both inherit the error from th

In [53]:
import sys, re, importlib
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
DRILL = ROOT / "src" / "tutor" / "drill.py"
sys.path.insert(0, str(ROOT / "src"))

src = DRILL.read_text(encoding="utf-8")

# ── 1. Remove the OLD diagnostic engine block ──
# The old block starts with the banner comment and ends right before grade_submission.
banner = "# Per-line diagnostic engine (factorable quadratics)"
i_banner = src.find(banner)
if i_banner > 0:
    block_start = src.rfind("# ============================================================",
                            0, i_banner)
    if block_start < 0: block_start = i_banner
    gs_pos = src.find("def grade_submission(", i_banner)
    gs_banner = src.rfind("# ============================================================",
                          0, gs_pos) if gs_pos > 0 else -1
    cut_end = gs_banner if gs_banner > block_start else gs_pos
    if cut_end > block_start:
        src = src[:block_start] + src[cut_end:]
        print(f"✓ old DIAG block removed ({cut_end - block_start} chars)")
    else:
        print("✗ could not determine DIAG block boundaries")
else:
    print("· no old DIAG block on disk")

# ── 2. Insert the new step-oriented engine ──
NEW_DIAG = r'''# ============================================================
# Step-oriented diagnostic engine (factorable quadratics)
# ============================================================
import re as _diag_re

_FACTORED = _diag_re.compile(
    r"\(\s*x\s*([+-])\s*(\d+)\s*\)\s*\(\s*x\s*([+-])\s*(\d+)\s*\)\s*=\s*0"
)
_SET_ZERO = _diag_re.compile(
    r"x\s*([+-])\s*(\d+)\s*=\s*0\s*(?:or|,)\s*x\s*([+-])\s*(\d+)\s*=\s*0"
)
_ROOTS = _diag_re.compile(
    r"x\s*=\s*(-?\d+)\s*(?:or|,)\s*x\s*=\s*(-?\d+)"
)
_REASONING = _diag_re.compile(
    r"(?:multiply|add|product|sum|times|\u00d7).*(?:\d).*(?:\d)",
    _diag_re.IGNORECASE,
)


def _sign_const(sign, num):
    return int(num) if sign == "+" else -int(num)


def _classify_all(text, meta):
    """Every memo-step classification a single learner line contains."""
    s = (text or "").strip()
    if not s:
        return []
    results = []

    m = _FACTORED.search(s)
    if m:
        p_l = _sign_const(m.group(1), m.group(2))
        q_l = _sign_const(m.group(3), m.group(4))
        if p_l + q_l == meta["b"] and p_l * q_l == meta["c"]:
            results.append({"kind": "factorise", "correct": True, "line": s})
        else:
            results.append({
                "kind": "factorise", "correct": False, "line": s,
                "reason": (f"Your bracket constants are {p_l} and {q_l}. "
                           f"Their sum is {p_l + q_l} (should be {meta['b']}). "
                           f"Their product is {p_l * q_l} (should be {meta['c']})."),
            })

    m = _SET_ZERO.search(s)
    if m:
        p_l = _sign_const(m.group(1), m.group(2))
        q_l = _sign_const(m.group(3), m.group(4))
        if {p_l, q_l} == {meta["p"], meta["q"]}:
            results.append({"kind": "setzero", "correct": True, "line": s})
        else:
            results.append({
                "kind": "setzero", "correct": False, "line": s,
                "reason": (f"These do not match the memo's factorisation. "
                           f"The memo brackets have constants {meta['p']} and {meta['q']}."),
            })

    m = _ROOTS.search(s)
    if m:
        lr1, lr2 = int(m.group(1)), int(m.group(2))
        if {lr1, lr2} == {meta["r1"], meta["r2"]}:
            results.append({"kind": "roots", "correct": True, "line": s})
        else:
            results.append({
                "kind": "roots", "correct": False, "line": s,
                "reason": (f"Your roots are {lr1} and {lr2}. "
                           f"The memo has {meta['r1']} and {meta['r2']}."),
            })

    if not results and _REASONING.search(s):
        results.append({"kind": "reasoning", "line": s})
    return results


def _diagnose_factorable_steps(learner_lines, meta):
    """Per-memo-step verdict for a factorable-quadratic attempt."""
    all_c = []
    for line in learner_lines:
        all_c.extend(_classify_all(line, meta))

    def find(kind):
        return next((c for c in all_c if c.get("kind") == kind), None)

    steps = []

    # Step 1 — Factorise
    f = find("factorise")
    if f and f["correct"]:
        steps.append({"step": 1, "name": "Factorise", "marks": 1,
                      "status": "CORRECT", "message": "", "evidence": f["line"]})
    elif f:
        steps.append({"step": 1, "name": "Factorise", "marks": 1,
                      "status": "WRONG", "message": f["reason"], "evidence": f["line"]})
    else:
        steps.append({"step": 1, "name": "Factorise", "marks": 1,
                      "status": "MISSING",
                      "message": (f"You didn't show the factored form. "
                                  f"The memo wants a line like: {meta['expected_factorised']}."),
                      "evidence": ""})

    # Step 2 — Set each factor to zero
    z = find("setzero")
    r = find("roots")
    if z and z["correct"]:
        steps.append({"step": 2, "name": "Set each factor to zero", "marks": 1,
                      "status": "CORRECT", "message": "", "evidence": z["line"]})
    elif z:
        steps.append({"step": 2, "name": "Set each factor to zero", "marks": 1,
                      "status": "WRONG", "message": z["reason"], "evidence": z["line"]})
    elif r:
        steps.append({"step": 2, "name": "Set each factor to zero", "marks": 1,
                      "status": "MISSING",
                      "message": (f"You went from brackets straight to the roots. "
                                  f"The memo wants each factor set to zero, "
                                  f"e.g. {meta['expected_setzero']}."),
                      "evidence": ""})
    else:
        steps.append({"step": 2, "name": "Set each factor to zero", "marks": 1,
                      "status": "MISSING",
                      "message": (f"You didn't set each factor to zero. "
                                  f"The memo wants a line like: {meta['expected_setzero']}."),
                      "evidence": ""})

    # Step 3 — Both roots
    if r and r["correct"]:
        steps.append({"step": 3, "name": "Write both roots", "marks": 1,
                      "status": "CORRECT", "message": "", "evidence": r["line"]})
    elif r:
        msg = r["reason"]
        if f and not f["correct"]:
            msg += " These match your own factorisation, but the factorisation was wrong."
        steps.append({"step": 3, "name": "Write both roots", "marks": 1,
                      "status": "WRONG", "message": msg, "evidence": r["line"]})
    else:
        steps.append({"step": 3, "name": "Write both roots", "marks": 1,
                      "status": "MISSING",
                      "message": (f"The final roots line is missing or incomplete. "
                                  f"The memo wants both solutions, e.g. {meta['expected_roots']}."),
                      "evidence": ""})

    return steps


def _find_meta_for_anchor(anchor):
    """Look up the meta dict for the anchor from its template pool."""
    try:
        tid = route_template(anchor.skill_id, anchor.prompt)
        if tid != "quadratic_standard_factorable":
            return None
        pool = _load_pool(tid)
        if not pool:
            return None
        s = anchor.prompt.replace(" ", "")
        m = _diag_re.search(r"x\^2([+-]?\d*)x([+-]?\d+)=0", s)
        if not m:
            return pool[0].get("meta")
        b_str, c_str = m.group(1), m.group(2)
        b = 1 if b_str == "" else (-1 if b_str == "-" else int(b_str))
        c = int(c_str)
        for v in pool:
            meta = v.get("meta")
            if meta and meta.get("b") == b and meta.get("c") == c:
                return meta
        return pool[0].get("meta")
    except Exception:
        return None


'''

gs_pos = src.find("def grade_submission(")
gs_line = src.rfind("\n", 0, gs_pos) + 1
src = src[:gs_line] + NEW_DIAG + src[gs_line:]
print("✓ new diagnostic engine inserted")

# ── 3. Update the hook inside grade_submission ──
# Remove any leftover line_diagnosis block and replace with step_diagnosis.
pattern = re.compile(
    r"    # per-line diagnostic.*?(?=    outcome = \{)",
    re.DOTALL,
)
src = pattern.sub("", src)

src = src.replace(
    '"line_diagnosis": line_diagnosis,',
    '"step_diagnosis": step_diagnosis,',
)
src = src.replace(
    '"line_diagnosis": [],',
    '"step_diagnosis": [],',
)

# Insert step_diagnosis block before the outcome dict if not present
if '"step_diagnosis": step_diagnosis' not in src:
    src = src.replace(
        '    outcome = {\n        "match": None,\n        "raw_lines": expanded,\n        "your_working": plain_lines,',
        '    meta = _find_meta_for_anchor(anchor)\n'
        '    step_diagnosis = []\n'
        '    if meta:\n'
        '        try:\n'
        '            step_diagnosis = _diagnose_factorable_steps(plain_lines, meta)\n'
        '        except Exception:\n'
        '            step_diagnosis = []\n\n'
        '    outcome = {\n'
        '        "match": None,\n'
        '        "raw_lines": expanded,\n'
        '        "your_working": plain_lines,\n'
        '        "step_diagnosis": step_diagnosis,'
    )
    print("✓ step_diagnosis block inserted")

# ── 4. Tighter match logic — wrong numbers = NO_MATCH, not UNCERTAIN ──
# Add a strict check: if the final answer's numbers don't match expected numbers
# as a set (mod sign), force NO_MATCH.
strict_block = '''    # strict number-set check before trusting the matcher's UNCERTAIN
    def _nums(s):
        import re as _rn
        return [int(x) for x in _rn.findall(r"-?\\d+", s or "")]
    def _signless(s):
        return set(abs(n) for n in _nums(s))

    _learner_nums = _signless(final_answer)
    _expected_nums = _signless(anchor.expected_answer)
    _num_set_matches = _learner_nums == _expected_nums and bool(_expected_nums)
'''

if "_signless" not in src:
    # insert right before sign_only computation
    src = src.replace(
        "    # strict: sign-only mismatch is a real error, not ambiguous form\n"
        "    sign_only = _sign_mismatch_only(final_answer, anchor.expected_answer)",
        strict_block +
        "\n    # strict: sign-only mismatch is a real error, not ambiguous form\n"
        "    sign_only = _sign_mismatch_only(final_answer, anchor.expected_answer)",
    )
    print("✓ strict number-set check inserted")

# Override UNCERTAIN when the number-set doesn't match
src = src.replace(
    '''    elif result.kind == MatchKind.UNCERTAIN:
        outcome["match"] = "UNCERTAIN"
        outcome["diagnosis"] = (
            "Your working does not clearly match the memo's expected form. "
            "Check the notation."
        )''',
    '''    elif result.kind == MatchKind.UNCERTAIN and not _num_set_matches:
        # Numbers differ from expected → wrong answer, not ambiguous form
        outcome["match"] = "NO_MATCH"
        outcome["diagnosis"] = (
            f"Your final answer ({final_answer}) does not match the memo "
            f"({anchor.expected_answer})."
        )
        sib = _pick_sibling_for_anchor(anchor)
        if sib: outcome["sibling"] = sib

    elif result.kind == MatchKind.UNCERTAIN:
        outcome["match"] = "UNCERTAIN"
        outcome["diagnosis"] = (
            "Your working does not clearly match the memo's expected form. "
            "Check the notation."
        )''',
)

DRILL.write_text(src, encoding="utf-8")
print(f"✓ drill.py written ({len(src):,} chars)")

# ── 5. Reload + run the four tests ──
import tutor
for name in list(vars(tutor).keys()):
    if not name.startswith("__"):
        try: delattr(tutor, name)
        except Exception: pass
for m in list(sys.modules):
    if m == "tutor" or m.startswith("tutor."):
        del sys.modules[m]
importlib.invalidate_caches()

from tutor import analogue_generators, analogue_library, drill
from tutor.learner_store import open_store
from tutor.schemas import Problem

analogue_library.POOL["quadratic_standard_factorable"] = \
    analogue_generators.gen_quadratic_standard_factorable(count=50, seed=101)

store = open_store(ROOT / "data" / "learner_store" / "learners.db")

variant = next(v for v in analogue_library.POOL["quadratic_standard_factorable"]
               if v["meta"]["r1"] == 4 and v["meta"]["r2"] == -9)
print()
print(f"anchor: {variant['problem']}")

anchor = Problem(
    problem_id="test-1", skill_id="algebra.quadratic.solve",
    topic="Algebra", subtopic="Quadratic equations",
    structure_type="routine_calculation",
    prompt=variant["problem"],
    expected_answer="x = 4 or x = -9",
)

def show(title, result):
    print()
    print("═" * 60)
    print(title)
    print("═" * 60)
    print(f"match: {result['match']}   mark: {result['mark']}")
    print()
    for d in result.get("step_diagnosis", []):
        print(f"  [{d['status']:8}] Step {d['step']}: {d['name']}  ({d['marks']} mark)")
        if d['evidence']:
            print(f"    written: {d['evidence']}")
        if d['message']:
            print(f"    → {d['message']}")

show("TEST A — wrong factors",
     drill.grade_submission(store, "tanaka", "t-a", anchor,
        ["(x + 4)(x - 8) = 0", "x = -4 or x = 8"], ""))
show("TEST B — full correct",
     drill.grade_submission(store, "tanaka", "t-b", anchor,
        ["(x - 4)(x + 9) = 0", "x - 4 = 0  or  x + 9 = 0", "x = 4 or x = -9"], ""))
show("TEST C — compressed (no middle step)",
     drill.grade_submission(store, "tanaka", "t-c", anchor,
        ["(x - 4)(x + 9) = 0", "x = 4 or x = -9"], ""))
show("TEST D — wrong factorise, roots inherit",
     drill.grade_submission(store, "tanaka", "t-d", anchor,
        ["(x - 9)(x + 4) = 0", "x = 9 or x = -4"], ""))

✓ old DIAG block removed (116 chars)
✓ new diagnostic engine inserted
✓ strict number-set check inserted
✓ drill.py written (51,252 chars)

anchor: Solve x^2 + 5x - 36 = 0


NameError: name 'step_diagnosis' is not defined

In [ ]:
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
DRILL = ROOT / "src" / "tutor" / "drill.py"

src = DRILL.read_text(encoding="utf-8")

# Show exactly what we're about to modify
i_use = src.find('"step_diagnosis": step_diagnosis,')
print("usage found at char:", i_use)

# Print the 300 chars before the usage line so we see what's there
print()
print("── 300 chars before the usage line ──")
print(src[i_use-300:i_use+50])

usage found at char: 45557

── 300 chars before the usage line ──
       "mark": {"awarded": 0, "total": 0}, "memo_steps": [],
                "diagnosis": f"Grader error: {e}",
                "tanaka_message": "", "mastery": None, "sibling": None}

    outcome = {
        "match": None,
        "raw_lines": expanded,
        "your_working": plain_lines,
        "step_diagnosis": step_diagnosis,
        "step_fe


In [ ]:
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
DRILL = ROOT / "src" / "tutor" / "drill.py"

src = DRILL.read_text(encoding="utf-8")

NEEDLE = '''    outcome = {
        "match": None,
        "raw_lines": expanded,
        "your_working": plain_lines,
        "step_diagnosis": step_diagnosis,'''

REPLACEMENT = '''    meta = _find_meta_for_anchor(anchor)
    step_diagnosis = []
    if meta:
        try:
            step_diagnosis = _diagnose_factorable_steps(plain_lines, meta)
        except Exception:
            step_diagnosis = []

    outcome = {
        "match": None,
        "raw_lines": expanded,
        "your_working": plain_lines,
        "step_diagnosis": step_diagnosis,'''

if "meta = _find_meta_for_anchor(anchor)" in src:
    print("· definition already present")
elif NEEDLE in src:
    src = src.replace(NEEDLE, REPLACEMENT, 1)
    DRILL.write_text(src, encoding="utf-8")
    print("✓ definition block inserted")
    print(f"  drill.py written ({len(src):,} chars)")
else:
    print("✗ needle not found")

✓ definition block inserted
  drill.py written (51,477 chars)


In [ ]:
import sys, re, importlib
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
DRILL = ROOT / "src" / "tutor" / "drill.py"
sys.path.insert(0, str(ROOT / "src"))

src = DRILL.read_text(encoding="utf-8")

# Find the exact position of the "step_diagnosis": step_diagnosis line
needle = '"step_diagnosis": step_diagnosis,'
i_use = src.find(needle)
if i_use < 0:
    raise SystemExit("✗ cannot find step_diagnosis reference")

# Find the "outcome = {" that precedes it
i_outcome = src.rfind("    outcome = {", 0, i_use)
if i_outcome < 0:
    raise SystemExit("✗ cannot find '    outcome = {' before it")

# Check if the definition block is already there
i_def = src.find("_find_meta_for_anchor(anchor)", 0, i_outcome)
if i_def > 0:
    print("· definition block appears before outcome — checking it's inside grade_submission")
else:
    # Insert the block right before "outcome = {"
    BLOCK = (
        "    meta = _find_meta_for_anchor(anchor)\n"
        "    step_diagnosis = []\n"
        "    if meta:\n"
        "        try:\n"
        "            step_diagnosis = _diagnose_factorable_steps(plain_lines, meta)\n"
        "        except Exception:\n"
        "            step_diagnosis = []\n\n"
    )
    src = src[:i_outcome] + BLOCK + src[i_outcome:]
    DRILL.write_text(src, encoding="utf-8")
    print(f"✓ definition block inserted before outcome (line {src[:i_outcome].count(chr(10))+1})")
    print(f"✓ drill.py written ({len(src):,} chars)")

# Verify by inspecting the file
src2 = DRILL.read_text(encoding="utf-8")
def_line = "meta = _find_meta_for_anchor(anchor)"
use_line = '"step_diagnosis": step_diagnosis,'
i_def2 = src2.find(def_line)
i_use2 = src2.find(use_line)
print()
print("verification:")
print(f"  definition at char: {i_def2}")
print(f"  usage at char:      {i_use2}")
print(f"  definition before usage: {0 <= i_def2 < i_use2}")

# ── reload and run tests ──
import tutor
for name in list(vars(tutor).keys()):
    if not name.startswith("__"):
        try: delattr(tutor, name)
        except Exception: pass
for m in list(sys.modules):
    if m == "tutor" or m.startswith("tutor."):
        del sys.modules[m]
importlib.invalidate_caches()

from tutor import analogue_generators, analogue_library, drill
from tutor.learner_store import open_store
from tutor.schemas import Problem

analogue_library.POOL["quadratic_standard_factorable"] = \
    analogue_generators.gen_quadratic_standard_factorable(count=50, seed=101)

store = open_store(ROOT / "data" / "learner_store" / "learners.db")
variant = next(v for v in analogue_library.POOL["quadratic_standard_factorable"]
               if v["meta"]["r1"] == 4 and v["meta"]["r2"] == -9)

anchor = Problem(
    problem_id="test-1", skill_id="algebra.quadratic.solve",
    topic="Algebra", subtopic="Quadratic equations",
    structure_type="routine_calculation",
    prompt=variant["problem"], expected_answer="x = 4 or x = -9",
)

def show(title, result):
    print()
    print("═" * 60); print(title); print("═" * 60)
    print(f"match: {result['match']}   mark: {result['mark']}")
    for d in result.get("step_diagnosis", []):
        print(f"  [{d['status']:8}] Step {d['step']}: {d['name']}  ({d['marks']} mark)")
        if d['evidence']: print(f"    written: {d['evidence']}")
        if d['message']:  print(f"    → {d['message']}")

show("TEST A — wrong factors",
     drill.grade_submission(store, "tanaka", "t-a", anchor,
        ["(x + 4)(x - 8) = 0", "x = -4 or x = 8"], ""))
show("TEST B — full correct",
     drill.grade_submission(store, "tanaka", "t-b", anchor,
        ["(x - 4)(x + 9) = 0", "x - 4 = 0  or  x + 9 = 0", "x = 4 or x = -9"], ""))
show("TEST C — compressed (no middle step)",
     drill.grade_submission(store, "tanaka", "t-c", anchor,
        ["(x - 4)(x + 9) = 0", "x = 4 or x = -9"], ""))
show("TEST D — wrong factorise, roots inherit",
     drill.grade_submission(store, "tanaka", "t-d", anchor,
        ["(x - 9)(x + 4) = 0", "x = 9 or x = -4"], ""))

· definition block appears before outcome — checking it's inside grade_submission

verification:
  definition at char: -1
  usage at char:      45557
  definition before usage: False


NameError: name 'step_diagnosis' is not defined

In [ ]:
from pathlib import Path
ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
DRILL = ROOT / "src" / "tutor" / "drill.py"

lines = DRILL.read_text(encoding="utf-8").splitlines()

# Show lines 1240-1262 — the region the traceback points at
print("── disk, lines 1240 to 1262 ──")
for i in range(1239, min(1262, len(lines))):
    print(f"{i+1:5}  {lines[i]}")

# Also search the whole file for the definition block
src = "\n".join(lines)
i_def = src.find("meta = _find_meta_for_anchor(anchor)")
i_use = src.find('"step_diagnosis": step_diagnosis,')
print()
print("definition exists at char:", i_def)
print("usage exists at char:     ", i_use)
print("definition before usage:  ", 0 <= i_def < i_use if i_def >= 0 else False)

# Is the definition INSIDE grade_submission?
i_gs = src.find("def grade_submission(")
i_next_def = src.find("\ndef ", i_gs + 10)
print()
print("grade_submission starts at char:", i_gs)
print("next top-level def at char:     ", i_next_def)
if i_def > 0 and i_gs > 0 and i_next_def > 0:
    inside = i_gs < i_def < i_next_def
    print("definition is inside grade_submission:", inside)

── disk, lines 1240 to 1262 ──
 1240  
 1241      try:
 1242          result = grade(final_answer, anchor.expected_answer)
 1243      except Exception as e:
 1244          return {"match": "error", "message": f"grader failed: {e}",
 1245                  "raw_lines": expanded, "your_working": plain_lines,
 1246                  "mark": {"awarded": 0, "total": 0}, "memo_steps": [],
 1247                  "diagnosis": f"Grader error: {e}",
 1248                  "tanaka_message": "", "mastery": None, "sibling": None}
 1249  
 1250      outcome = {
 1251          "match": None,
 1252          "raw_lines": expanded,
 1253          "your_working": plain_lines,
 1254          "step_diagnosis": step_diagnosis,
 1255          "step_feedback": None,
 1256          "memo_steps": [],
 1257          "mark": {"awarded": 0, "total": 0},
 1258          "diagnosis": "",
 1259          "tanaka_message": "",
 1260          "mastery": None,
 1261          "sibling": None,
 1262      }

definition exists 

In [ ]:
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
DRILL = ROOT / "src" / "tutor" / "drill.py"

lines = DRILL.read_text(encoding="utf-8").split("\n")

# Find the outcome line inside grade_submission.
# We know it's at line 1250 (0-indexed: 1249). Verify before changing.
target = None
for i, line in enumerate(lines):
    if line == '    outcome = {' and i > 1200:
        # confirm the next line is "match": None
        if i + 1 < len(lines) and '"match": None,' in lines[i + 1]:
            target = i
            break

if target is None:
    raise SystemExit("✗ could not locate outcome block")

print(f"inserting before line {target+1}")

block = [
    '    meta = _find_meta_for_anchor(anchor)',
    '    step_diagnosis = []',
    '    if meta:',
    '        try:',
    '            step_diagnosis = _diagnose_factorable_steps(plain_lines, meta)',
    '        except Exception:',
    '            step_diagnosis = []',
    '',
]

new_lines = lines[:target] + block + lines[target:]
DRILL.write_text("\n".join(new_lines), encoding="utf-8")
print(f"✓ inserted 8 lines  ({len(lines)} → {len(new_lines)} total lines)")

# Verify immediately by reading back
verify = DRILL.read_text(encoding="utf-8")
i_def = verify.find("meta = _find_meta_for_anchor(anchor)")
i_use = verify.find('"step_diagnosis": step_diagnosis,')
print()
print("verification after write:")
print(f"  definition at char: {i_def}")
print(f"  usage at char:      {i_use}")
print(f"  definition before usage: {0 <= i_def < i_use}")
print()

# Show the region
lines2 = verify.split("\n")
for i in range(target - 2, min(target + 12, len(lines2))):
    print(f"{i+1:5}  {lines2[i]}")

inserting before line 1250
✓ inserted 8 lines  (1393 → 1401 total lines)

verification after write:
  definition at char: 45446
  usage at char:      45782
  definition before usage: True

 1248                  "tanaka_message": "", "mastery": None, "sibling": None}
 1249  
 1250      meta = _find_meta_for_anchor(anchor)
 1251      step_diagnosis = []
 1252      if meta:
 1253          try:
 1254              step_diagnosis = _diagnose_factorable_steps(plain_lines, meta)
 1255          except Exception:
 1256              step_diagnosis = []
 1257  
 1258      outcome = {
 1259          "match": None,
 1260          "raw_lines": expanded,
 1261          "your_working": plain_lines,


In [ ]:
import sys, re, importlib
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
DRILL = ROOT / "src" / "tutor" / "drill.py"
sys.path.insert(0, str(ROOT / "src"))

src = DRILL.read_text(encoding="utf-8")

# Find the exact position of the "step_diagnosis": step_diagnosis line
needle = '"step_diagnosis": step_diagnosis,'
i_use = src.find(needle)
if i_use < 0:
    raise SystemExit("✗ cannot find step_diagnosis reference")

# Find the "outcome = {" that precedes it
i_outcome = src.rfind("    outcome = {", 0, i_use)
if i_outcome < 0:
    raise SystemExit("✗ cannot find '    outcome = {' before it")

# Check if the definition block is already there
i_def = src.find("_find_meta_for_anchor(anchor)", 0, i_outcome)
if i_def > 0:
    print("· definition block appears before outcome — checking it's inside grade_submission")
else:
    # Insert the block right before "outcome = {"
    BLOCK = (
        "    meta = _find_meta_for_anchor(anchor)\n"
        "    step_diagnosis = []\n"
        "    if meta:\n"
        "        try:\n"
        "            step_diagnosis = _diagnose_factorable_steps(plain_lines, meta)\n"
        "        except Exception:\n"
        "            step_diagnosis = []\n\n"
    )
    src = src[:i_outcome] + BLOCK + src[i_outcome:]
    DRILL.write_text(src, encoding="utf-8")
    print(f"✓ definition block inserted before outcome (line {src[:i_outcome].count(chr(10))+1})")
    print(f"✓ drill.py written ({len(src):,} chars)")

# Verify by inspecting the file
src2 = DRILL.read_text(encoding="utf-8")
def_line = "meta = _find_meta_for_anchor(anchor)"
use_line = '"step_diagnosis": step_diagnosis,'
i_def2 = src2.find(def_line)
i_use2 = src2.find(use_line)
print()
print("verification:")
print(f"  definition at char: {i_def2}")
print(f"  usage at char:      {i_use2}")
print(f"  definition before usage: {0 <= i_def2 < i_use2}")

# ── reload and run tests ──
import tutor
for name in list(vars(tutor).keys()):
    if not name.startswith("__"):
        try: delattr(tutor, name)
        except Exception: pass
for m in list(sys.modules):
    if m == "tutor" or m.startswith("tutor."):
        del sys.modules[m]
importlib.invalidate_caches()

from tutor import analogue_generators, analogue_library, drill
from tutor.learner_store import open_store
from tutor.schemas import Problem

analogue_library.POOL["quadratic_standard_factorable"] = \
    analogue_generators.gen_quadratic_standard_factorable(count=50, seed=101)

store = open_store(ROOT / "data" / "learner_store" / "learners.db")
variant = next(v for v in analogue_library.POOL["quadratic_standard_factorable"]
               if v["meta"]["r1"] == 4 and v["meta"]["r2"] == -9)

anchor = Problem(
    problem_id="test-1", skill_id="algebra.quadratic.solve",
    topic="Algebra", subtopic="Quadratic equations",
    structure_type="routine_calculation",
    prompt=variant["problem"], expected_answer="x = 4 or x = -9",
)

def show(title, result):
    print()
    print("═" * 60); print(title); print("═" * 60)
    print(f"match: {result['match']}   mark: {result['mark']}")
    for d in result.get("step_diagnosis", []):
        print(f"  [{d['status']:8}] Step {d['step']}: {d['name']}  ({d['marks']} mark)")
        if d['evidence']: print(f"    written: {d['evidence']}")
        if d['message']:  print(f"    → {d['message']}")

show("TEST A — wrong factors",
     drill.grade_submission(store, "tanaka", "t-a", anchor,
        ["(x + 4)(x - 8) = 0", "x = -4 or x = 8"], ""))
show("TEST B — full correct",
     drill.grade_submission(store, "tanaka", "t-b", anchor,
        ["(x - 4)(x + 9) = 0", "x - 4 = 0  or  x + 9 = 0", "x = 4 or x = -9"], ""))
show("TEST C — compressed (no middle step)",
     drill.grade_submission(store, "tanaka", "t-c", anchor,
        ["(x - 4)(x + 9) = 0", "x = 4 or x = -9"], ""))
show("TEST D — wrong factorise, roots inherit",
     drill.grade_submission(store, "tanaka", "t-d", anchor,
        ["(x - 9)(x + 4) = 0", "x = 9 or x = -4"], ""))

· definition block appears before outcome — checking it's inside grade_submission

verification:
  definition at char: 45446
  usage at char:      45782
  definition before usage: True

════════════════════════════════════════════════════════════
TEST A — wrong factors
════════════════════════════════════════════════════════════
match: NO_MATCH   mark: {'awarded': 0, 'total': 0}

════════════════════════════════════════════════════════════
TEST B — full correct
════════════════════════════════════════════════════════════
match: MATCH   mark: {'awarded': 0, 'total': 0}
  [CORRECT ] Step 1: Factorise  (1 mark)
    written: (x - 4)(x + 9) = 0
  [CORRECT ] Step 2: Set each factor to zero  (1 mark)
    written: x - 4 = 0 or x + 9 = 0
  [CORRECT ] Step 3: Write both roots  (1 mark)
    written: x = 4 or x = -9

════════════════════════════════════════════════════════════
TEST C — compressed (no middle step)
════════════════════════════════════════════════════════════
match: MATCH   mark: {'a

In [ ]:
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
DRILL = ROOT / "src" / "tutor" / "drill.py"

src = DRILL.read_text(encoding="utf-8")

# ── 1. Add the derivation block at the top of _diagnose_factorable_steps ──
OLD_HEAD = '''def _diagnose_factorable_steps(learner_lines, meta):
    """Per-memo-step verdict for a factorable-quadratic attempt."""
    all_c = []'''

NEW_HEAD = '''def _diagnose_factorable_steps(learner_lines, meta):
    """Per-memo-step verdict for a factorable-quadratic attempt."""
    # Derive expected strings from primitives if not provided
    if "expected_factorised" not in meta:
        def _br(k):
            if k == 0: return "(x)"
            return f"(x - {k})" if k > 0 else f"(x + {abs(k)})"
        def _sz(k):
            if k == 0: return "x = 0"
            return f"x - {k} = 0" if k > 0 else f"x + {abs(k)} = 0"
        meta = dict(meta)
        meta["expected_factorised"] = f"{_br(meta['p'])}{_br(meta['q'])} = 0"
        meta["expected_setzero"]    = f"{_sz(meta['p'])}  or  {_sz(meta['q'])}"
        meta["expected_roots"]      = f"x = {meta['r1']} or x = {meta['r2']}"

    all_c = []'''

if "if \"expected_factorised\" not in meta:" in src:
    print("· derivation already present")
elif OLD_HEAD in src:
    src = src.replace(OLD_HEAD, NEW_HEAD, 1)
    DRILL.write_text(src, encoding="utf-8")
    print("✓ derivation block inserted into _diagnose_factorable_steps")
    print(f"  drill.py written ({len(src):,} chars)")
else:
    print("✗ anchor not found")

· derivation already present


In [ ]:
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
DRILL = ROOT / "src" / "tutor" / "drill.py"

src = DRILL.read_text(encoding="utf-8")

# Find the final "return outcome" inside grade_submission
i_gs = src.find("def grade_submission(")
if i_gs < 0:
    raise SystemExit("grade_submission not found")

# Find the last "return outcome" before the next top-level def
next_def = src.find("\ndef ", i_gs + 10)
if next_def < 0:
    next_def = len(src)

region = src[i_gs:next_def]
i_ret = region.rfind("    return outcome")
if i_ret < 0:
    raise SystemExit("return outcome not found")

abs_ret = i_gs + i_ret

OVERRIDE = '''    # ── override with step_diagnosis when available ──
    if step_diagnosis:
        awarded = sum(1 for d in step_diagnosis if d["status"] == "CORRECT")
        total   = sum(d["marks"] for d in step_diagnosis)
        outcome["mark"] = {"awarded": awarded, "total": total}

        wrong = [d for d in step_diagnosis
                 if d["status"] in ("WRONG", "MISSING")]
        if not wrong:
            outcome["diagnosis"] = "Full method. All memo steps shown."
        else:
            outcome["diagnosis"] = wrong[0]["message"]

        # Verdict: match depends on both the answer and the method
        if awarded == total:
            outcome["match"] = "MATCH"
        elif _num_set_matches and awarded >= 1:
            outcome["match"] = "MATCH"     # right answer, method incomplete
        else:
            outcome["match"] = "NO_MATCH"

        # No sibling on a clean full mark
        if awarded == total:
            outcome["sibling"] = None

'''

if "override with step_diagnosis" in src:
    print("· override already present")
else:
    src = src[:abs_ret] + OVERRIDE + src[abs_ret:]
    DRILL.write_text(src, encoding="utf-8")
    print("✓ override block inserted before final return")
    print(f"  drill.py written ({len(src):,} chars)")

· override already present


In [ ]:
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
DRILL = ROOT / "src" / "tutor" / "drill.py"

src = DRILL.read_text(encoding="utf-8")

# Replace each of the six "steps.append({...})" dicts to include an "expected" key.
# Do it via regex on the four patterns.

# 1. Step 1 CORRECT
src = src.replace(
    'steps.append({"step": 1, "name": "Factorise", "marks": 1,\n'
    '                      "status": "CORRECT", "message": "", "evidence": f["line"]})',
    'steps.append({"step": 1, "name": "Factorise", "marks": 1,\n'
    '                      "status": "CORRECT", "message": "", "evidence": f["line"],\n'
    '                      "expected": meta["expected_factorised"]})',
)

# 2. Step 1 WRONG
src = src.replace(
    'steps.append({"step": 1, "name": "Factorise", "marks": 1,\n'
    '                      "status": "WRONG", "message": f["reason"], "evidence": f["line"]})',
    'steps.append({"step": 1, "name": "Factorise", "marks": 1,\n'
    '                      "status": "WRONG", "message": f["reason"], "evidence": f["line"],\n'
    '                      "expected": meta["expected_factorised"]})',
)

# 3. Step 1 MISSING
src = src.replace(
    'steps.append({"step": 1, "name": "Factorise", "marks": 1,\n'
    '                      "status": "MISSING",\n'
    '                      "message": (f"You didn\'t show the factored form. "\n'
    '                                  f"The memo wants a line like: {meta[\'expected_factorised\']}."),\n'
    '                      "evidence": ""})',
    'steps.append({"step": 1, "name": "Factorise", "marks": 1,\n'
    '                      "status": "MISSING",\n'
    '                      "message": (f"You didn\'t show the factored form. "\n'
    '                                  f"The memo wants a line like: {meta[\'expected_factorised\']}."),\n'
    '                      "evidence": "", "expected": meta["expected_factorised"]})',
)

# 4. Step 2 CORRECT
src = src.replace(
    'steps.append({"step": 2, "name": "Set each factor to zero", "marks": 1,\n'
    '                      "status": "CORRECT", "message": "", "evidence": z["line"]})',
    'steps.append({"step": 2, "name": "Set each factor to zero", "marks": 1,\n'
    '                      "status": "CORRECT", "message": "", "evidence": z["line"],\n'
    '                      "expected": meta["expected_setzero"]})',
)

# 5. Step 2 WRONG
src = src.replace(
    'steps.append({"step": 2, "name": "Set each factor to zero", "marks": 1,\n'
    '                      "status": "WRONG", "message": z["reason"], "evidence": z["line"]})',
    'steps.append({"step": 2, "name": "Set each factor to zero", "marks": 1,\n'
    '                      "status": "WRONG", "message": z["reason"], "evidence": z["line"],\n'
    '                      "expected": meta["expected_setzero"]})',
)

# 6. Step 2 MISSING (both variants — "went from brackets" and "didn't set")
src = src.replace(
    '"status": "MISSING",\n'
    '                      "message": (f"You went from brackets straight to the roots. "\n'
    '                                  f"The memo wants each factor set to zero, "\n'
    '                                  f"e.g. {meta[\'expected_setzero\']}."),\n'
    '                      "evidence": ""})',
    '"status": "MISSING",\n'
    '                      "message": (f"You went from brackets straight to the roots. "\n'
    '                                  f"The memo wants each factor set to zero, "\n'
    '                                  f"e.g. {meta[\'expected_setzero\']}."),\n'
    '                      "evidence": "", "expected": meta["expected_setzero"]})',
)
src = src.replace(
    '"status": "MISSING",\n'
    '                      "message": (f"You didn\'t set each factor to zero. "\n'
    '                                  f"The memo wants a line like: {meta[\'expected_setzero\']}."),\n'
    '                      "evidence": ""})',
    '"status": "MISSING",\n'
    '                      "message": (f"You didn\'t set each factor to zero. "\n'
    '                                  f"The memo wants a line like: {meta[\'expected_setzero\']}."),\n'
    '                      "evidence": "", "expected": meta["expected_setzero"]})',
)

# 7. Step 3 CORRECT
src = src.replace(
    'steps.append({"step": 3, "name": "Write both roots", "marks": 1,\n'
    '                      "status": "CORRECT", "message": "", "evidence": r["line"]})',
    'steps.append({"step": 3, "name": "Write both roots", "marks": 1,\n'
    '                      "status": "CORRECT", "message": "", "evidence": r["line"],\n'
    '                      "expected": meta["expected_roots"]})',
)

# 8. Step 3 WRONG
src = src.replace(
    'steps.append({"step": 3, "name": "Write both roots", "marks": 1,\n'
    '                      "status": "WRONG", "message": msg, "evidence": r["line"]})',
    'steps.append({"step": 3, "name": "Write both roots", "marks": 1,\n'
    '                      "status": "WRONG", "message": msg, "evidence": r["line"],\n'
    '                      "expected": meta["expected_roots"]})',
)

# 9. Step 3 MISSING
src = src.replace(
    'steps.append({"step": 3, "name": "Write both roots", "marks": 1,\n'
    '                      "status": "MISSING",\n'
    '                      "message": (f"The final roots line is missing or incomplete. "\n'
    '                                  f"The memo wants both solutions, e.g. {meta[\'expected_roots\']}."),\n'
    '                      "evidence": ""})',
    'steps.append({"step": 3, "name": "Write both roots", "marks": 1,\n'
    '                      "status": "MISSING",\n'
    '                      "message": (f"The final roots line is missing or incomplete. "\n'
    '                                  f"The memo wants both solutions, e.g. {meta[\'expected_roots\']}."),\n'
    '                      "evidence": "", "expected": meta["expected_roots"]})',
)

DRILL.write_text(src, encoding="utf-8")

# Count how many steps now carry "expected"
count = src.count('"expected": meta[')
print(f"✓ added 'expected' to {count} step variants")
print(f"  drill.py written ({len(src):,} chars)")
print()

if count < 9:
    print("⚠ only some patterns matched — showing diagnostics")
    # Find what didn't match by inspecting the function
    i = src.find("def _diagnose_factorable_steps")
    j = src.find("\ndef ", i+10)
    body = src[i:j]
    for line in body.split("\n"):
        if 'steps.append' in line or '"expected"' in line:
            print(f"  {line.strip()[:110]}")

✓ added 'expected' to 10 step variants
  drill.py written (53,574 chars)



In [ ]:
import sys, importlib, inspect
from pathlib import Path
ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
sys.path.insert(0, str(ROOT / "src"))

import tutor
for name in list(vars(tutor).keys()):
    if not name.startswith("__"):
        try: delattr(tutor, name)
        except Exception: pass
for m in list(sys.modules):
    if m == "tutor" or m.startswith("tutor."):
        del sys.modules[m]
importlib.invalidate_caches()

from tutor import analogue_generators, analogue_library, drill

analogue_library.POOL["quadratic_standard_factorable"] = \
    analogue_generators.gen_quadratic_standard_factorable(count=50, seed=101)

# find the meta for x^2 + 5x - 36 = 0
meta = None
for v in analogue_library.POOL["quadratic_standard_factorable"]:
    if v["meta"]["b"] == 5 and v["meta"]["c"] == -36:
        meta = v["meta"]
        break
print("meta:", meta)
print()

# ── 1. Call _diagnose_factorable_steps DIRECTLY on all four inputs ──
for label, lines in [
    ("A", ["(x + 4)(x - 8) = 0", "x = -4 or x = 8"]),
    ("B", ["(x - 4)(x + 9) = 0", "x - 4 = 0  or  x + 9 = 0", "x = 4 or x = -9"]),
    ("C", ["(x - 4)(x + 9) = 0", "x = 4 or x = -9"]),
    ("D", ["(x - 9)(x + 4) = 0", "x = 9 or x = -4"]),
]:
    print(f"═══ direct call TEST {label} ═══")
    try:
        steps = drill._diagnose_factorable_steps(lines, meta)
        print(f"  got {len(steps)} steps")
        for d in steps:
            print(f"  {d['status']:8}  {d['name']:30}  evidence={d.get('evidence','')!r}")
    except Exception as e:
        import traceback
        traceback.print_exc()
    print()

# ── 2. Print the actual source of grade_submission from disk ──
print("═══ grade_submission source, first 80 lines ═══")
src = inspect.getsource(drill.grade_submission)
for i, line in enumerate(src.split("\n")[:80], 1):
    print(f"{i:3}  {line}")

meta: {'r1': 4, 'r2': -9, 'b': 5, 'c': -36, 'p': -4, 'q': 9}

═══ direct call TEST A ═══
  got 3 steps
  WRONG     Factorise                       evidence='(x + 4)(x - 8) = 0'
  MISSING   Set each factor to zero         evidence=''
  WRONG     Write both roots                evidence='x = -4 or x = 8'

═══ direct call TEST B ═══
  got 3 steps
  CORRECT   Factorise                       evidence='(x - 4)(x + 9) = 0'
  CORRECT   Set each factor to zero         evidence='x - 4 = 0  or  x + 9 = 0'
  CORRECT   Write both roots                evidence='x = 4 or x = -9'

═══ direct call TEST C ═══
  got 3 steps
  CORRECT   Factorise                       evidence='(x - 4)(x + 9) = 0'
  MISSING   Set each factor to zero         evidence=''
  CORRECT   Write both roots                evidence='x = 4 or x = -9'

═══ direct call TEST D ═══
  got 3 steps
  WRONG     Factorise                       evidence='(x - 9)(x + 4) = 0'
  MISSING   Set each factor to zero         evidence=''
  WRONG     Wr

In [54]:
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")

# ── 1. Files with "diagnos" in the name anywhere in the tree ──
print("═══ files with 'diagnos' in the name ═══")
for p in ROOT.rglob("*diagnos*"):
    if p.is_file():
        size = p.stat().st_size
        rel = p.relative_to(ROOT)
        print(f"  {rel}  ({size:,} bytes)")

# ── 2. Files with "report" in the name ──
print()
print("═══ files with 'report' in the name ═══")
for p in ROOT.rglob("*report*"):
    if p.is_file():
        size = p.stat().st_size
        rel = p.relative_to(ROOT)
        print(f"  {rel}  ({size:,} bytes)")

# ── 3. Everything inside data/processed/diagnostics (from project state) ──
print()
print("═══ data/processed/diagnostics ═══")
d = ROOT / "data" / "processed" / "diagnostics"
if d.exists():
    for p in sorted(d.rglob("*")):
        if p.is_file():
            rel = p.relative_to(ROOT)
            print(f"  {rel}  ({p.stat().st_size:,} bytes)")
else:
    print("  (folder does not exist)")

# ── 4. Any CSV / JSON in data/processed/tutor with "row" or "diag" in name ──
print()
print("═══ data/processed/tutor — rows/diag/analysis CSVs ═══")
t = ROOT / "data" / "processed" / "tutor"
if t.exists():
    for p in sorted(t.iterdir()):
        if p.is_file() and any(k in p.name.lower()
                                for k in ("row", "diag", "analysis", "report")):
            print(f"  {p.name}  ({p.stat().st_size:,} bytes)")

# ── 5. Recently downloaded files in common Download locations ──
print()
print("═══ Downloads folder — recent CSVs/XLSX mentioning methodmarker/matric/diag ═══")
for candidate in [
    Path.home() / "Downloads",
    Path.home() / "Desktop",
]:
    if not candidate.exists():
        continue
    print(f"  {candidate}:")
    try:
        files = sorted(
            [p for p in candidate.iterdir()
             if p.is_file()
             and p.suffix.lower() in (".csv", ".xlsx", ".xls", ".json")
             and any(k in p.name.lower()
                     for k in ("method", "matric", "diag", "report",
                               "learner", "skill"))],
            key=lambda p: p.stat().st_mtime,
            reverse=True,
        )
        for p in files[:20]:
            import datetime as _dt
            mtime = _dt.datetime.fromtimestamp(p.stat().st_mtime)
            print(f"    {p.name}  ({p.stat().st_size:,} bytes, "
                  f"{mtime.strftime('%Y-%m-%d %H:%M')})")
    except Exception as e:
        print(f"    (could not list: {e})")

# ── 6. Anything that looks like a diagnostic report by content ──
print()
print("═══ any CSV in the project with 'misconception' or 'error_type' columns ═══")
hits = 0
for p in ROOT.rglob("*.csv"):
    try:
        with open(p, "r", encoding="utf-8", errors="ignore") as f:
            header = f.readline().lower()
        if "misconception" in header or "error_type" in header or "diagnosis" in header:
            rel = p.relative_to(ROOT)
            print(f"  {rel}")
            print(f"    header: {header.strip()[:120]}")
            hits += 1
            if hits >= 20:
                break
    except Exception:
        continue
if hits == 0:
    print("  (none found)")

═══ files with 'diagnos' in the name ═══
  docs\Learner Error _and_ Diagnostic _Difficulty _Analysis.ipynb  (218,260 bytes)
  reports\diagnostic_conclusions_2014_2025.md  (10,270 bytes)
  scripts\diagnose_near_miss.py  (1,177 bytes)
  scripts\diagnostic_survey.py  (2,041 bytes)
  scripts\parse_diagnostic_2014_2022.py  (4,672 bytes)
  scripts\parse_diagnostic_v3.py  (10,430 bytes)
  scripts\parse_diagnostic_v4.py  (10,104 bytes)
  scripts\parse_diagnostic_v5.py  (10,712 bytes)
  scripts\parse_diagnostic_v6.py  (11,724 bytes)
  scripts\parse_diagnostic_v7.py  (11,060 bytes)
  scripts\parse_diagnostic_v8.py  (12,858 bytes)
  scripts\parse_diagnostic_v9.py  (12,858 bytes)
  scripts\patch_diagnosis_graceful.py  (1,535 bytes)
  scripts\patch_diagnosis_v2.py  (5,596 bytes)
  tests\test_diagnosis_v1_2.py  (3,200 bytes)
  src\analysis\n06_diagnostic_v2.py  (4,520 bytes)
  src\tutor\diagnosis.py  (14,490 bytes)
  src\tutor\llm_diagnoser.py  (6,219 bytes)
  src\tutor\write _diagnosis.py  (5,206 b

In [55]:
import csv, json
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")

def head_csv(path, n=5):
    print(f"── {path.relative_to(ROOT)} ──")
    try:
        with open(path, encoding="utf-8", errors="ignore") as f:
            reader = csv.reader(f)
            header = next(reader)
            print(f"  columns ({len(header)}): {header}")
            for i, row in enumerate(reader):
                if i >= n: break
                print(f"    {row}")
    except Exception as e:
        print(f"  error: {e}")
    print()

# ── 1. The two master diagnostic files ──
head_csv(ROOT / "data/processed/diagnostics/diagnostic_errors_v9.csv", 3)
head_csv(ROOT / "data/processed/diagnostics/diagnostic_errors_with_skill.csv", 3)

# ── 2. Misconceptions catalog ──
head_csv(ROOT / "data/processed/tutor/misconceptions_v2.csv", 5)

# ── 3. Intervention patterns ──
head_csv(ROOT / "data/processed/intervention/intervention_specification_v1.csv", 5)

# ── 4. Error class breakdown (topic × year) ──
head_csv(ROOT / "data/processed/diagnostics/error_class_topic_year.csv", 3)

# ── 5. Per-skill error profile? ──
for cand in [
    ROOT / "data/processed/diagnostics/persistence_table.csv",
    ROOT / "data/processed/diagnostics/topic_year_matrix.csv",
    ROOT / "data/processed/diagnostics/performance_all_years.csv",
]:
    if cand.exists():
        head_csv(cand, 3)

# ── 6. Does the tutor read any of these? ──
print("═══ which tutor modules mention 'diagnostics' or these file names ═══")
for py in (ROOT / "src" / "tutor").rglob("*.py"):
    try:
        text = py.read_text(encoding="utf-8")
    except Exception:
        continue
    hits = []
    for probe in ["diagnostic_errors", "misconceptions_v2", "intervention_spec",
                  "diagnostics/", "data/processed/diagnostics"]:
        if probe in text:
            hits.append(probe)
    if hits:
        print(f"  {py.relative_to(ROOT)}")
        for h in hits:
            print(f"    mentions: {h}")

# ── 7. What does n06_diagnostic_v2 or diagnosis.py actually load? ──
print()
print("═══ src/analysis/n06_diagnostic_v2.py — top 40 lines ═══")
p = ROOT / "src" / "analysis" / "n06_diagnostic_v2.py"
if p.exists():
    for i, line in enumerate(p.read_text(encoding="utf-8").split("\n")[:40], 1):
        print(f"  {i:3}  {line}")

── data\processed\diagnostics\diagnostic_errors_v9.csv ──
  columns (17): ['year', 'paper', 'question_number', 'question_title', 'topic', 'subquestion_ref', 'error_text', 'error_class', 'cross_topic', 'diagram_assumption', 'notation_subtype', 'source_section', 'source', 'raw_char_count', 'cleaned_char_count', 'mojibake_cleaned', 'suggestion_text']
    ['2014', 'P1', '1', 'ALGEBRA', 'ALG', '', 'Candidates wrote down the quadratic formula incorrectly from the information sheet. Some might have learnt the formula incorrectly and did not use the information sheet and this was also noted in the previous diagnostic report. Some candidates substituted the value of 2 instead of –2 for b. A few candidates were unable to write the equation in standard form. 112 REPORT ON NATIONAL SENIOR CERTIFICATE EXAMINATION', 'procedural', 'False', 'False', '', 'errors', 'DBE diagnostic', '419', '414', 'True', '']
    ['2014', 'P1', '1', 'ALGEBRA', 'ALG', '', 'When solving exponential equations, candidates di

In [56]:
from pathlib import Path
p = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\src\tutor\diagnosis.py")
text = p.read_text(encoding="utf-8")
print(text[:3000])

"""Diagnosis rules — v1.1 base + v1.2 additive rules. Matching uses normalised text."""

from __future__ import annotations

import re
from pathlib import Path
from .answer_matcher import is_correct as _graceful_is_correct
from typing import Callable, Dict, List, Tuple

from .schemas import DiagnosisResult, ErrorType


def _norm(text: str) -> str:
    if text is None:
        return ""
    s = str(text).lower().strip()
    s = s.replace("θ", "theta").replace("Θ", "theta")
    s = s.replace("≥", ">=").replace("≤", "<=")
    s = s.replace("–", "-").replace("—", "-")
    s = s.replace("**", "^")  # v1.2
    s = s.replace("0r", " or ")
    s = s.replace("ror", " or ")
    s = re.sub(r"\bor(\d)", r"or \1", s)
    s = re.sub(r"(\d)or\b", r"\1 or", s)
    s = re.sub(r"\b(sin|cos|tan|cot|sec|csc)(theta|x|\()", r"\1 \2", s)
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"\bor(\d)", r"or \1", s)   # v1.2: or3 -> or 3
    s = re.sub(r"(\d)or\b", r"\1 or", s)
    s = re.sub(r"\s*=\s*", "=", s)
   

In [57]:
from pathlib import Path
p = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\src\tutor\n08_loader.py")
text = p.read_text(encoding="utf-8")
print(text[:3000])

"""Load and validate N08 intervention_specification CSV with safe fallback."""

from __future__ import annotations

import hashlib
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Optional, Tuple, List, Any

import pandas as pd

from .schemas import DiagnosisResult, InterventionSelection, ProvenanceKind

# Same five-case map as before (fallback)
FALLBACK_MAP = {
    "M_SIGN_ERROR_FACTORISATION": ("IP_MISCONCEPTION_CONTRAST", "N08_ALG_QUAD_SIGN_01"),
    "M_TP_COORD_CONFUSION": ("IP_REPRESENTATION_SHIFT", "N08_FUNC_PARAB_02"),
    "M_REDUCTION_SIGN": ("IP_MISCONCEPTION_CONTRAST", "N08_TRIG_RED_01"),
    "M_DIAGRAM_ASSUMPTION": ("IP_STEP_ISOLATION", "N08_AGEO_PROOF_01"),
    "M_MISSING_THEOREM_CITATION": ("IP_SELF_EXPLANATION", "N08_EUCL_PROOF_01"),
}

REQUIRED_COLUMNS = {
    "intervention_id",
    "topic",
    "misconception_id",
    "intervention_pattern",
}

# Optional renames from older N08 exports
COLUMN_ALIASES = {
    "pattern": "intervention_pa

In [58]:
from pathlib import Path
ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")

# Does grade_submission call n08_loader or the intervention library?
src = (ROOT / "src" / "tutor" / "drill.py").read_text(encoding="utf-8")
for probe in ["n08_loader", "load_n08_library", "intervention_pattern",
              "InterventionSelection", "misconceptions_v2", "diagnostic_errors_with_skill",
              "persistence_table", "error_class_topic_year"]:
    print(f"  {probe:<35} {'✓ in drill.py' if probe in src else '✗ not in drill.py'}")

print()

# Does anything else in the tutor read the DBE diagnostic corpus?
print("── which tutor modules read the corpus files? ──")
for probe in ["diagnostic_errors_with_skill", "persistence_table",
              "error_class_topic_year", "diagnostic_errors_v9"]:
    hits = []
    for py in (ROOT / "src" / "tutor").rglob("*.py"):
        try:
            if probe in py.read_text(encoding="utf-8"):
                hits.append(py.name)
        except Exception:
            pass
    print(f"  {probe:<35} {hits if hits else 'NOT read by any tutor module'}")

print()

# Does diagnosis.py call n08_loader?
diag = (ROOT / "src" / "tutor" / "diagnosis.py").read_text(encoding="utf-8")
for probe in ["n08_loader", "load_n08_library", "intervention_pattern"]:
    print(f"  diagnosis.py mentions {probe:<30} {'✓' if probe in diag else '✗'}")

  n08_loader                          ✗ not in drill.py
  load_n08_library                    ✗ not in drill.py
  intervention_pattern                ✗ not in drill.py
  InterventionSelection               ✗ not in drill.py
  misconceptions_v2                   ✗ not in drill.py
  diagnostic_errors_with_skill        ✗ not in drill.py
  persistence_table                   ✗ not in drill.py
  error_class_topic_year              ✗ not in drill.py

── which tutor modules read the corpus files? ──
  diagnostic_errors_with_skill        NOT read by any tutor module
  persistence_table                   NOT read by any tutor module
  error_class_topic_year              NOT read by any tutor module
  diagnostic_errors_v9                NOT read by any tutor module

  diagnosis.py mentions n08_loader                     ✗
  diagnosis.py mentions load_n08_library               ✗
  diagnosis.py mentions intervention_pattern           ✗


In [59]:
from pathlib import Path
import json, csv

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")

def show_file(path, lines=30):
    print(f"\n── {path.relative_to(ROOT)} ──")
    try:
        if path.suffix == ".md":
            text = path.read_text(encoding="utf-8", errors="ignore")
            for line in text.split("\n")[:lines]:
                print(f"  {line}")
            if text.count("\n") > lines:
                print(f"  ... ({text.count(chr(10))} lines total)")
        elif path.suffix == ".json":
            obj = json.loads(path.read_text(encoding="utf-8"))
            print(f"  {json.dumps(obj, indent=2, ensure_ascii=False)[:1500]}")
        elif path.suffix == ".csv":
            with open(path, encoding="utf-8", errors="ignore") as f:
                reader = csv.reader(f)
                for i, row in enumerate(reader):
                    if i >= lines: break
                    print(f"  {row}")
        else:
            print(f"  ({path.stat().st_size:,} bytes)")
    except Exception as e:
        print(f"  error: {e}")

# ══════════════════════════════════════════════════════════════
# 1. The conclusions doc
# ══════════════════════════════════════════════════════════════
p = ROOT / "reports" / "diagnostic_conclusions_2014_2025.md"
if p.exists():
    print("═" * 70)
    print("diagnostic_conclusions_2014_2025.md")
    print("═" * 70)
    print(p.read_text(encoding="utf-8"))

# ══════════════════════════════════════════════════════════════
# 2. The manifest
# ══════════════════════════════════════════════════════════════
print()
print("═" * 70)
print("diagnostic_manifest.json")
print("═" * 70)
show_file(ROOT / "data/processed/diagnostics/diagnostic_manifest.json")

# ══════════════════════════════════════════════════════════════
# 3. Search for "presentation" reports / 210 items
# ══════════════════════════════════════════════════════════════
print()
print("═" * 70)
print("Files/folders mentioning 'presentation'")
print("═" * 70)
for p in ROOT.rglob("*presentation*"):
    if p.is_file():
        print(f"  {p.relative_to(ROOT)}  ({p.stat().st_size:,} bytes)")

# ══════════════════════════════════════════════════════════════
# 4. Everything in reports/
# ══════════════════════════════════════════════════════════════
print()
print("═" * 70)
print("Everything under reports/")
print("═" * 70)
r = ROOT / "reports"
if r.exists():
    for p in sorted(r.rglob("*")):
        if p.is_file():
            print(f"  {p.relative_to(ROOT)}  ({p.stat().st_size:,} bytes)")

# ══════════════════════════════════════════════════════════════
# 5. Everything under docs/pilot/
# ══════════════════════════════════════════════════════════════
print()
print("═" * 70)
print("Everything under docs/pilot/")
print("═" * 70)
d = ROOT / "docs" / "pilot"
if d.exists():
    for p in sorted(d.rglob("*")):
        if p.is_file():
            print(f"  {p.relative_to(ROOT)}  ({p.stat().st_size:,} bytes)")

# ══════════════════════════════════════════════════════════════
# 6. Notebooks and scripts that did the parsing
# ══════════════════════════════════════════════════════════════
print()
print("═" * 70)
print("Analysis notebooks")
print("═" * 70)
for p in (ROOT / "docs").rglob("*.ipynb"):
    if any(k in p.name.lower() for k in ("diagnos", "error", "difficulty", "report")):
        print(f"  {p.relative_to(ROOT)}  ({p.stat().st_size:,} bytes)")

# ══════════════════════════════════════════════════════════════
# 7. Whatever the parse_* scripts produced
# ══════════════════════════════════════════════════════════════
print()
print("═" * 70)
print("Outputs of the parse_diagnostic_* scripts")
print("═" * 70)
# The scripts read PDFs and write CSVs — check what's in data/processed/diagnostics
d = ROOT / "data" / "processed" / "diagnostics"
if d.exists():
    total = 0
    for p in sorted(d.iterdir()):
        if p.is_file():
            total += p.stat().st_size
            print(f"  {p.name:<50} {p.stat().st_size:>10,} bytes")
    print(f"  {'':50} {'-'*10}")
    print(f"  {'total':<50} {total:>10,} bytes")

# ══════════════════════════════════════════════════════════════
# 8. The notebook that produced the conclusions
# ══════════════════════════════════════════════════════════════
print()
print("═" * 70)
print("Learner Error and Diagnostic Difficulty Analysis notebook")
print("═" * 70)
nb_path = ROOT / "docs" / "Learner Error _and_ Diagnostic _Difficulty _Analysis.ipynb"
if not nb_path.exists():
    for p in (ROOT / "docs").rglob("*.ipynb"):
        if "Learner Error" in p.name or "Difficulty" in p.name:
            nb_path = p
            break
if nb_path.exists():
    try:
        nb = json.loads(nb_path.read_text(encoding="utf-8"))
        cells = nb.get("cells", [])
        print(f"  notebook: {nb_path.name}")
        print(f"  cells: {len(cells)}")
        print()
        print("  -- code cells (first line of each) --")
        for i, c in enumerate(cells):
            if c.get("cell_type") == "code":
                src = "".join(c.get("source", []))
                first_line = src.split("\n")[0][:90]
                print(f"    [{i:3}] {first_line}")
            elif c.get("cell_type") == "markdown":
                src = "".join(c.get("source", []))
                if src.strip():
                    first_line = src.split("\n")[0][:80]
                    print(f"    [{i:3}] # {first_line}")
    except Exception as e:
        print(f"  could not parse notebook: {e}")

══════════════════════════════════════════════════════════════════════
diagnostic_conclusions_2014_2025.md
══════════════════════════════════════════════════════════════════════
# NSC Mathematics Diagnostic Conclusions
## 2014–2025 — Twelve Years of DBE Evidence

---

## 1. Method & Corpus

| Metric | Value |
|---|---|
| Years analysed | **12** (2014–2025, no gaps) |
| Papers | 24 (P1 + P2 per year) |
| Error narratives | **1111** |
| Topics covered | 10 (all mapped, 0 unmapped) |
| Sources joined | DBE diagnostic narrative + per-question averages |
| Join rate | **100%** — every error carries a real numeric priority |

**Sources:** DBE NSC Diagnostic Reports, Chapter 10 (Mathematics), 2014–2025.  
Per-question averages extracted from each report. Error narratives parsed and classified by rule-based tagger.

**Limitations:** 2019 was parsed with paragraph-based splitting due to scrambled PDF layout — its 75 errors are consistent with other years. 2020 P1, 2021 P2, 2022 P2 averages were

In [60]:
from pathlib import Path
p = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\scripts\wire_presentation_into_drill.py")
print(p.read_text(encoding="utf-8"))

"""Wire presentation_matcher into drill.py — notes appear after the verdict."""
from pathlib import Path
import ast

P = Path("src/tutor/drill.py")
src = P.read_text(encoding="utf-8-sig")

# Import
if "from .presentation_matcher import check_presentation, format_note" not in src:
    old_imp = "from .template_router import route_template"
    new_imp = ("from .template_router import route_template\n"
               "from .presentation_matcher import check_presentation, format_note")
    if old_imp in src:
        src = src.replace(old_imp, new_imp, 1)
        print("Added presentation_matcher import")
    else:
        print("WARN: import anchor not found")

# 1. After MATCH — add presentation notes
old_match = '''            if result.kind == MatchKind.MATCH:
                correct_count += 1
                print()
                print(f"  ✓ Correct.  ({correct_count} correct this drill)")
                break'''

new_match = '''            if result.kind == MatchKind.MATCH:
     

In [61]:
import csv
from pathlib import Path
p = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\processed\tutor\presentation_rules_v1.csv")
with open(p, encoding="utf-8") as f:
    r = csv.reader(f)
    header = next(r)
    n = sum(1 for _ in r)
print(f"rows: {n}")
print(f"columns: {header}")

rows: 210
columns: ['rule_id', 'category', 'topic', 'trigger', 'advice', 'why', 'dbe_note', 'priority', 'source']


In [62]:
import sys, csv, inspect, importlib
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
sys.path.insert(0, str(ROOT / "src"))

# ══════════════════════════════════════════════════════════════
# 1. What does presentation_matcher.py actually do?
# ══════════════════════════════════════════════════════════════
import tutor
for name in list(vars(tutor).keys()):
    if not name.startswith("__"):
        try: delattr(tutor, name)
        except Exception: pass
for m in list(sys.modules):
    if m == "tutor" or m.startswith("tutor."):
        del sys.modules[m]
importlib.invalidate_caches()

from tutor import presentation_matcher as pm

print("═══ presentation_matcher source ═══")
print(inspect.getsource(pm))

═══ presentation_matcher source ═══
"""
Presentation matcher.

Reads presentation_rules_v1.csv and matches the learner's working
against presentation triggers. Returns gentle coaching notes for
the highest-priority matches only — never blocks the main verdict.

Public:
    check_presentation(working_lines, topic=None) -> list[PresentationNote]
    format_note(note) -> str
"""
from __future__ import annotations

import csv
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Optional


ROOT = Path(__file__).resolve().parents[2]
CSV_PATH = ROOT / "data" / "processed" / "tutor" / "presentation_rules_v1.csv"

PRIORITY_RANK = {"critical": 4, "high": 3, "medium": 2, "low": 1}


@dataclass
class PresentationNote:
    rule_id: str
    category: str
    topic: str
    priority: str
    advice: str
    dbe_note: str
    evidence: str = ""


# ------------------------------------------------------------------
# Load rules
# -------------------------

In [63]:
import sys, csv, importlib
from pathlib import Path
ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
sys.path.insert(0, str(ROOT / "src"))

import tutor
for name in list(vars(tutor).keys()):
    if not name.startswith("__"):
        try: delattr(tutor, name)
        except Exception: pass
for m in list(sys.modules):
    if m == "tutor" or m.startswith("tutor."):
        del sys.modules[m]
importlib.invalidate_caches()

from tutor import presentation_matcher as pm

# ── A. What rule_ids exist in the CSV vs which have predicates ──
rules_csv = pm._rules()
csv_ids = {r["rule_id"] for r in rules_csv}
pred_ids = set(pm._RULE_PREDICATES.keys())

print(f"rules in CSV:              {len(csv_ids)}")
print(f"rules with predicates:     {len(pred_ids)}")
print(f"rules CSV-only (fallback): {len(csv_ids - pred_ids)}")
print(f"predicates with no rule:   {len(pred_ids - csv_ids)}")
print()

# which of the predicate entries are stubs (return False always)
stubs = []
for rid, fn in pm._RULE_PREDICATES.items():
    # try a bunch of inputs
    fired = False
    for text in ["x = 1", "y = -5(x + 2)", "(a+b)^2 = a^2+2ab+b^2", "sin theta = 0.5",
                 "x^2 + 5x - 36 = 0", "answer: 12", "!!!", "~~", "3/4 + 1/2"]:
        try:
            if fn(text, "ALG") or fn(text, "EUCL") or fn(text, "FIN"):
                fired = True
                break
        except Exception:
            pass
    if not fired:
        stubs.append(rid)

print(f"predicates that NEVER fire on plausible input: {len(stubs)}")
for rid in stubs:
    print(f"  {rid}")
print()

# ── B. Live test on realistic learner submissions ──
test_cases = [
    ("answer only, no working",   ["x = 4 or x = 3"],  "algebra.quadratic.solve"),
    ("all on one line",            ["(x+4)(x-3)=0 x=4 or x=-3"], "algebra.quadratic.solve"),
    ("no reason in Euclidean",     ["AB = CD"], "euclidean"),
    ("fraction with slash",        ["1/2 + 1/3 = 5/6"], "algebra"),
    ("negative with bracket",      ["= -3(2)"], "algebra"),
    ("no units in finance",        ["A = 5000", "A = 5000(1.05)^3"], "finance"),
    ("single root for quadratic",  ["x = 4"], "algebra.quadratic.solve"),
]

print("═" * 60)
print("Live test — which rules actually fire?")
print("═" * 60)
for label, lines, topic in test_cases:
    notes = pm.check_presentation(lines, topic=topic, max_notes=5)
    print(f"\n  {label}")
    print(f"    input: {lines}")
    print(f"    notes: {len(notes)}")
    for n in notes:
        print(f"      → {n.rule_id}  ({n.priority}): {n.advice[:70]}")

rules in CSV:              210
rules with predicates:     19
rules CSV-only (fallback): 191
predicates with no rule:   0

predicates that NEVER fire on plausible input: 11
  PR_GEN_004
  PR_ALG_012
  PR_ALG_016
  PR_ALG_020
  PR_AGEO_007
  PR_DIAG_008
  PR_STAT_012
  PR_PROB_014
  PR_DBE_002
  PR_DBE_004
  PR_DBE_010

════════════════════════════════════════════════════════════
Live test — which rules actually fire?
════════════════════════════════════════════════════════════

  answer only, no working
    input: ['x = 4 or x = 3']
    notes: 2
      → PR_GEN_009  (critical): Show every substitution step; don't skip from formula to number.
      → PR_GEN_001  (high): Write your final answer on a new line and box or underline it.

  all on one line
    input: ['(x+4)(x-3)=0 x=4 or x=-3']
    notes: 4
      → PR_GEN_009  (critical): Show every substitution step; don't skip from formula to number.
      → PR_DBE_010  (critical): One mathematical step per line, especially when transformati

In [64]:
import csv
from pathlib import Path
ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
p = ROOT / "data" / "processed" / "tutor" / "presentation_rules_v1.csv"

with open(p, encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

# show 15 rules across different topics
seen_topics = set()
shown = 0
for r in rows:
    if r["topic"] in seen_topics and len(seen_topics) < 6:
        continue
    seen_topics.add(r["topic"])
    shown += 1
    print(f"[{r['rule_id']}] topic={r['topic']}  cat={r['category']}  priority={r['priority']}")
    print(f"  trigger: {r['trigger']}")
    print(f"  advice:  {r['advice'][:100]}")
    print(f"  dbe:     {r['dbe_note'][:100]}")
    print()
    if shown >= 15:
        break

# count how many triggers look like regex
import re
regex_like = 0
word_like = 0
for r in rows:
    t = r.get("trigger", "")
    if any(c in t for c in ["(", ")", "|", "[", "]", "\\", "*", "+", "?"]):
        regex_like += 1
    else:
        word_like += 1

print(f"triggers that look like regex: {regex_like}")
print(f"triggers that look like plain text: {word_like}")

[PR_GEN_001] topic=all  cat=general  priority=high
  trigger: final answer not boxed or on new line
  advice:  Write your final answer on a new line and box or underline it.
  dbe:     Quick one — every year the DBE diagnostic reports repeat that markers lose answers in the working. I

[PR_ALG_001] topic=ALG  cat=algebra  priority=critical
  trigger: quadratic not in standard form before formula
  advice:  Write the equation in standard form before applying the formula.
  dbe:     Every year since 2014 the DBE flags this. 'They substituted 6 instead of –6 because the equation was

[PR_SEQ_001] topic=SEQ  cat=sequences  priority=high
  trigger: AP vs GP not stated
  advice:  State whether it's arithmetic or geometric before writing a formula.
  dbe:     Did you know — the DBE flags 'used wrong formula' every year. One line stating AP or GP prevents the

[PR_FUNC_001] topic=FUNC  cat=functions  priority=high
  trigger: domain and range on same line
  advice:  State domain and range on se

In [65]:
import sys, inspect, json
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
sys.path.insert(0, str(ROOT / "src"))

# ══════════════════════════════════════════════════════════════
# 1. Discovery store — does it exist? What fields?
# ══════════════════════════════════════════════════════════════
print("═" * 70)
print("1. discovery_store.py — the persistence contract")
print("═" * 70)
p = ROOT / "src" / "tutor" / "discovery_store.py"
if p.exists():
    text = p.read_text(encoding="utf-8")
    print(f"  file: {p.relative_to(ROOT)}  ({p.stat().st_size:,} bytes)")
    print()
    # show the create table / schema
    for line in text.split("\n"):
        if any(k in line.upper() for k in
               ["CREATE TABLE", "DISCOVERY_ID", "PATTERN_DESCRIPTION",
                "EXAMPLE_LINES", "TIMES_SEEN", "LEARNERS_AFFECTED",
                "CONFIDENCE", "LLM_EXPLANATION", "PROMOTED_TO_RULE",
                "TEACHER_NOTES", "STATUS", "FIRST_SEEN", "LAST_SEEN"]):
            print(f"  {line.strip()[:110]}")
else:
    print("  ✗ discovery_store.py NOT FOUND")

# ══════════════════════════════════════════════════════════════
# 2. Is the discovery store wired into learner_store?
# ══════════════════════════════════════════════════════════════
print()
print("═" * 70)
print("2. learner_store.py — is discoveries exposed on the store?")
print("═" * 70)
p = ROOT / "src" / "tutor" / "learner_store.py"
text = p.read_text(encoding="utf-8")
for probe in ["self.discoveries", "DiscoveryStore", "def discoveries",
              "self.interventions", "InterventionStore"]:
    hits = [i for i, l in enumerate(text.split("\n"))
            if probe in l]
    print(f"  {probe:<30} {'✓' if hits else '✗'}  {hits[:3] if hits else ''}")

# ══════════════════════════════════════════════════════════════
# 3. Where does drill.py call discoveries.record?
# ══════════════════════════════════════════════════════════════
print()
print("═" * 70)
print("3. drill.py — how it calls discoveries.record")
print("═" * 70)
text = (ROOT / "src" / "tutor" / "drill.py").read_text(encoding="utf-8")
lines = text.split("\n")
for i, line in enumerate(lines):
    if "discoveries.record" in line:
        for j in range(max(0, i-5), min(len(lines), i+8)):
            print(f"  {j+1:4}  {lines[j]}")
        print()

# ══════════════════════════════════════════════════════════════
# 4. LLM layer — 7 modules exist. Which are called?
# ══════════════════════════════════════════════════════════════
print("═" * 70)
print("4. LLM layer — who imports the LLM modules?")
print("═" * 70)
llm_modules = ["llm_caller", "llm_config", "llm_diagnoser",
               "llm_gate", "llm_phaser", "llm_conversation_prompt"]
for mod in llm_modules:
    callers = []
    for py in (ROOT / "src" / "tutor").rglob("*.py"):
        if py.stem == mod:
            continue
        try:
            if mod in py.read_text(encoding="utf-8"):
                callers.append(py.stem)
        except Exception:
            pass
    print(f"  {mod:<28} called by: {callers if callers else '✗ nobody'}")

# ══════════════════════════════════════════════════════════════
# 5. Layer 2 pattern search — is there anything that searches past attempts?
# ══════════════════════════════════════════════════════════════
print()
print("═" * 70)
print("5. Pattern search — anything looking across past attempts?")
print("═" * 70)
for probe in ["pattern_search", "search_patterns", "recurrence",
              "learners_affected", "times_seen", "promote_discovery",
              "promote_rule"]:
    hits = []
    for py in (ROOT / "src" / "tutor").rglob("*.py"):
        try:
            if probe in py.read_text(encoding="utf-8"):
                hits.append(py.stem)
        except Exception:
            pass
    print(f"  {probe:<25} {hits if hits else '✗ not implemented'}")

# ══════════════════════════════════════════════════════════════
# 6. Teacher referral layer
# ══════════════════════════════════════════════════════════════
print()
print("═" * 70)
print("6. Teacher referral — does it exist?")
print("═" * 70)
for probe in ["teacher_referral", "teacher_notes", "referral",
              "teacher_dashboard", "teacher_report"]:
    hits = []
    for py in ROOT.rglob("*.py"):
        try:
            if probe in py.read_text(encoding="utf-8"):
                hits.append(str(py.relative_to(ROOT)))
        except Exception:
            pass
    print(f"  {probe:<25} {hits[:5] if hits else '✗ not implemented'}")

# ══════════════════════════════════════════════════════════════
# 7. Live — is tutor_discoveries in the SQLite database?
# ══════════════════════════════════════════════════════════════
print()
print("═" * 70)
print("7. Live DB — what tables exist in learners.db?")
print("═" * 70)
import sqlite3
db = ROOT / "data" / "learner_store" / "learners.db"
if db.exists():
    conn = sqlite3.connect(str(db))
    cur = conn.cursor()
    cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
    tables = [r[0] for r in cur.fetchall()]
    for t in tables:
        cur.execute(f"SELECT COUNT(*) FROM {t}")
        n = cur.fetchone()[0]
        print(f"  {t:<30} {n:>8} rows")
    conn.close()

══════════════════════════════════════════════════════════════════════
1. discovery_store.py — the persistence contract
══════════════════════════════════════════════════════════════════════
  file: src\tutor\discovery_store.py  (14,918 bytes)

  - recurrence strengthens (times_seen, learners_affected)
  llm_explanation: Optional[str] = None,
  """Record an unresolved case. Returns the discovery_id.
  adds this learner, and recomputes confidence from all signals.
  "SELECT discovery_id, times_seen, learners_affected, evidence_signals "
  "WHERE skill_id = ? AND pattern_description = ?",
  learners = json.loads(existing["learners_affected"])
  signals["learners_affected"] = learners
  new_conf = compute_confidence(signals)
  new_status = derive_status(new_conf, signals)
  "  times_seen = times_seen + 1, "
  "  last_seen_at = ?, "
  "  learners_affected = ?, "
  "  confidence = ?, "
  "  status = ? "
  "WHERE discovery_id = ?",
  new_conf, new_status, existing["discovery_id"]),
  return 

In [66]:
import sqlite3, json
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
db = ROOT / "data" / "learner_store" / "learners.db"
conn = sqlite3.connect(str(db))
conn.row_factory = sqlite3.Row
cur = conn.cursor()

cur.execute("SELECT * FROM tutor_discovery ORDER BY last_seen_at DESC")
rows = cur.fetchall()

print(f"═══ {len(rows)} discoveries on file ═══\n")
for r in rows:
    d = dict(r)
    print(f"── {d['discovery_id']} ──")
    print(f"   skill:          {d.get('skill_id')}")
    print(f"   pattern:        {d.get('pattern_description','')[:100]}")
    print(f"   times_seen:     {d.get('times_seen')}")
    print(f"   learners:       {d.get('learners_affected')}")
    print(f"   confidence:     {d.get('confidence')}")
    print(f"   status:         {d.get('status')}")
    print(f"   first_seen:     {d.get('first_seen_at')}")
    print(f"   last_seen:      {d.get('last_seen_at')}")
    print(f"   llm_explanation:{'<empty>' if not d.get('llm_explanation') else d['llm_explanation'][:60]}")
    print(f"   teacher_notes:  {'<empty>' if not d.get('teacher_notes') else d['teacher_notes'][:60]}")
    print(f"   promoted:       {d.get('promoted_to_rule')}")
    print(f"   example_lines:  {(d.get('example_lines') or '')[:150]}")
    print()
conn.close()

═══ 5 discoveries on file ═══

── disc_42e396333fd4 ──
   skill:          algebra.quadratic.solve
   pattern:        x=4orx=3
   times_seen:     17
   learners:       ["tanaka"]
   confidence:     0.3
   status:         recurring
   first_seen:     2026-10-05T12:59:42.203106+00:00
   last_seen:      2026-10-05T20:26:11.432631+00:00
   llm_explanation:<empty>
   teacher_notes:  <empty>
   promoted:       0
   example_lines:  ["x = 4 or x = 3"]

── disc_fd37712314ec ──
   skill:          algebra.quadratic.solve
   pattern:        x=3or1
   times_seen:     3
   learners:       ["tanaka", "sipho", "test_tanaka"]
   confidence:     0.3
   status:         noticed
   first_seen:     2026-09-28T15:03:02.010908+00:00
   last_seen:      2026-09-28T15:34:00.699489+00:00
   llm_explanation:<empty>
   teacher_notes:  Confirmed — sign error in root extraction
   promoted:       0
   example_lines:  ["x = 3 or 1"]

── disc_e3a6287893cd ──
   skill:          test.skill.a
   pattern:        x=3or1
   t